# Publication-Grade Nested Cross-Validation Pipeline

**Data:** `PFS_ML_train_cof_wo_IG.xlsx` (n=34) and `PFS_ML_test_cof_wo_IG.xlsx` (n=16) - log2 MS protein intensities

**Target:** Treatment classification (PFS: S/L)

## Pipeline Overview

1. **Data Preprocessing**: Randomization + StandardScaler (z-score)
2. **Nested CV with Grid Search**:
   - Outer loop: Model evaluation (unbiased performance estimate)
   - Inner loop: ANOVA feature filtering → Greedy forward selection or Elastic net (no pre-filtering)→ Classifer gid search hyperparameter tuning
   - **Optimization metric: F1 score**
3. **Champion Selection**: Composite score (4 parameters: Scoring weights: outer cv_f1: 30%, test_f1: 40%, cv_frequency: 15%, cv_test_agreement: 15%, parsimony_bonus: 0.01 per feature below max)
4. **Final Model**: Calibrated probability estimates
5. **Statistical Analysis**: Post-hoc CV 15x3-fold (dynamic) CV champion vs. ZeroR, Bootstrap 95% CI on test set predictions for all basic metrics, advanced metrics, calibration, learning curves, clinical utility, stability analyses, ROC, Youden
6. **Publication-Grade Outputs**: Visualizations (PNG/PDF) + Excel data tables

In [ ]:
# =============================================================================
# CONFIGURATION - TREATMENT CLASSIFICATION (FOLFI vs GemNab)
# =============================================================================
# Dataset: n=34 training, n=16 test, p=895 features, balanced classes
# Optimization date: 2025-12-02

# File paths
TRAIN_XLSX_PATH = r"PFS_ML_train_cof_wo_IG.xlsx"
TEST_XLSX_PATH  = r"PFS_ML_test_cof_wo_IG.xlsx"
SHEET_NAME      = 0
# OUTDIR          = r"C:\Users\timei\Desktop\Nested CV-Final pipeline generalization_031225\Nested_CV_PFS_386_pickle_Elastic net k15_Anova_Stability-FALSE"  # Update to relative path

# =============================================================================
# TARGET VARIABLE CONFIGURATION
# =============================================================================
# Define the dependent variable column name and class labels
TARGET_COLUMN   = "PFS_group"          # Column name in Excel file
CLASS_LABELS    = {0: "L", 1: "S"}  # {numeric_code: label_string}
POSITIVE_CLASS  = "S"             # Which class is considered "positive" (class 1)

# Nested CV parameters
SEED            = 42
OUTER_SPLITS    = 3       # 3-fold outer CV (11-12 samples/fold)
OUTER_REPEATS   = 15      # 15 repeats = 45 total outer evaluations (increased for stability)
INNER_SPLITS    = 3       # 3-fold inner CV for feature selection + grid search

# Feature selection parameters - OPTIMIZED FOR SMALL n
TOP_K_FILTER       = 50    # Reduced from 100 (conservative for n=34)
CANDIDATES_PER_STEP = 15   # Reduced from 25 (less overfitting risk)
MIN_SIGNATURE      = 3     # Minimum signature size
MAX_SIGNATURE      = 4     # Reduced from 5 (rule: ~10 samples per feature)
EPS_IMPROVE        = 1e-6  # Early stopping threshold

# =============================================================================
# FEATURE SELECTION METHOD TOGGLES
# =============================================================================

# Option B: Multi-Filter Ensemble (ANOVA + MI + Correlation)
USE_MULTI_FILTER = False  # OFF - ANOVA sufficient for balanced data

# Multi-filter weights (only used if USE_MULTI_FILTER = True)
FILTER_WEIGHTS = {
    'anova': 0.4,
    'mutual_info': 0.4,
    'correlation': 0.2
}

# Option A: Stability Selection (Meinshausen & Buhlmann, 2010)
USE_STABILITY_SELECTION = False  # ON - Critical for small samples

# Stability selection parameters - OPTIMIZED
STABILITY_N_BOOTSTRAP = 100     # Increased from 50 (more robust)
STABILITY_THRESHOLD = 0.7       # Increased from 0.6 (more stringent)
STABILITY_SUBSAMPLE_RATIO = 0.8 # Keep 80% per bootstrap

# Option C: Elastic Net Pre-filtering
USE_ELASTIC_NET = True  # ON - Good for high p/n ratio

ELASTIC_NET_MODE = 'replace_wrapper'  # Direct Elastic Net feature selection

# Elastic Net parameters - OPTIMIZED
ELASTIC_NET_L1_RATIO = 0.5      # Balanced L1/L2
ELASTIC_NET_N_ALPHAS = 50       # Alpha grid size
ELASTIC_NET_CV_FOLDS = 3        # Reduced from 5 (small inner fold size)
ELASTIC_NET_TOP_K = 15          # Reduced from 20 (conservative for n=34)

# =============================================================================
# OTHER PARAMETERS
# =============================================================================

OPTIMIZATION_METRIC = "f1"       # F1 for balanced classification
SCALER_KIND        = "standard"  # z-score for log2 MS data
RANDOMIZE_DATA     = True

# Bootstrap parameters - kept robust
N_BOOTSTRAP        = 5000  # Sufficient for stable 95% CI
CI_LEVEL           = 0.95

# Calibration
CALIBRATE_FINAL    = True  # Platt scaling for final model

VERBOSE = True

print("Configuration loaded.")
print(f"  Target variable: {TARGET_COLUMN}")
print(f"  Class labels: {CLASS_LABELS}")
print(f"  Positive class (1): {POSITIVE_CLASS}")
print(f"  Dataset: n=34 train, n=16 test, p=895 features")
print(f"  Optimization metric: {OPTIMIZATION_METRIC.upper()}")
print(f"  Feature Selection Methods:")
print(f"    - Multi-filter ensemble: {'ON' if USE_MULTI_FILTER else 'OFF (ANOVA only)'}")
print(f"    - Stability selection: {'ON' if USE_STABILITY_SELECTION else 'OFF'}")
if USE_STABILITY_SELECTION:
    print(f"      * Bootstrap iterations: {STABILITY_N_BOOTSTRAP}")
    print(f"      * Selection threshold: {STABILITY_THRESHOLD*100:.0f}%")
print(f"    - Elastic Net: {'ON (' + ELASTIC_NET_MODE + ')' if USE_ELASTIC_NET else 'OFF'}")
if USE_ELASTIC_NET:
    print(f"      * L1 ratio: {ELASTIC_NET_L1_RATIO}")
    print(f"      * Top-k features: {ELASTIC_NET_TOP_K}")
print(f"  Max signature size: {MAX_SIGNATURE} (optimized for n=34)")
print(f"  Outer CV: {OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats = {OUTER_SPLITS * OUTER_REPEATS} evaluations")
print(f"  Bootstrap CI iterations: {N_BOOTSTRAP}")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os, sys, math, json, warnings, copy
from collections import Counter, defaultdict
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats

from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedKFold, cross_val_score,
    GridSearchCV, cross_val_predict
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import f_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, matthews_corrcoef,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, auc, precision_recall_curve,
    average_precision_score, brier_score_loss, log_loss,
    classification_report, make_scorer
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils import check_random_state, resample
from sklearn.base import clone

warnings.filterwarnings("ignore")

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# Set publication-quality plot defaults
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print("Imports loaded. Publication-quality plot settings applied.")

In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

os.makedirs(OUTDIR, exist_ok=True)
LOG_PATH = os.path.join(OUTDIR, "run_log.txt")

def _flush():
    sys.stdout.flush(); sys.stderr.flush()

def vlog(msg):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {msg}"
    if VERBOSE: print(line); _flush()
    try:
        with open(LOG_PATH, "a") as f: f.write(line + "\n")
    except: pass

def pbar(iterable, desc=None, total=None, leave=False):
    if HAS_TQDM: return tqdm(iterable, desc=desc, total=total, leave=leave)
    return iterable

def make_scaler(kind):
    if kind is None: return None
    k = str(kind).strip().lower()
    if k in ("none", "off", "no", "false"): return None
    if k in ("standard", "z", "zscore"): return StandardScaler()
    return MinMaxScaler()

def savefig(outdir, name, fig=None):
    """Save figure in PNG and PDF formats."""
    os.makedirs(outdir, exist_ok=True)
    if fig is None: fig = plt.gcf()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(outdir, f"{name}.{ext}"), 
                    bbox_inches="tight", dpi=300, facecolor='white')
    plt.close(fig)

def write_excel_safely(path, sheets):
    """Write multiple DataFrames to Excel with fallback to CSV."""
    for eng in ("openpyxl", "xlsxwriter", None):
        try:
            with pd.ExcelWriter(path, engine=eng) as writer:
                for name, df in sheets.items():
                    if isinstance(df, dict): df = pd.DataFrame(df)
                    df.to_excel(writer, index=False, sheet_name=str(name)[:31])
            vlog(f"[Saved] {path}")
            return path
        except Exception: continue
    # Fallback
    folder = os.path.splitext(path)[0] + "_csv"
    os.makedirs(folder, exist_ok=True)
    for name, df in sheets.items():
        if isinstance(df, dict): df = pd.DataFrame(df)
        df.to_csv(os.path.join(folder, f"{name}.csv"), index=False)
    return folder

print("Utility functions defined.")

In [ ]:
# =============================================================================
# BOOTSTRAP CONFIDENCE INTERVAL FUNCTIONS (STRATIFIED)
# =============================================================================

def stratified_bootstrap_metric(y_true, y_pred, y_proba, metric_fn, n_bootstrap=2000, ci=0.95, seed=42):
    """
    Compute STRATIFIED bootstrap confidence interval for a metric.
    Maintains class proportions in each bootstrap sample.
    Returns: (point_estimate, ci_lower, ci_upper)
    """
    # Compute RAW point estimate on original data first
    try:
        if y_proba is not None and hasattr(metric_fn, '__name__') and 'proba' in metric_fn.__name__:
            raw_point = metric_fn(y_true, y_proba)
        else:
            raw_point = metric_fn(y_true, y_pred)
    except:
        raw_point = np.nan
    
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    
    # Get indices for each class
    idx_class0 = np.where(y_true == 0)[0]
    idx_class1 = np.where(y_true == 1)[0]
    n0, n1 = len(idx_class0), len(idx_class1)
    
    for _ in pbar(range(n_bootstrap), desc="Bootstrap", leave=False):
        # Stratified sampling: sample from each class proportionally
        boot_idx0 = rng.choice(idx_class0, size=n0, replace=True)
        boot_idx1 = rng.choice(idx_class1, size=n1, replace=True)
        idx = np.concatenate([boot_idx0, boot_idx1])
        
        y_t = y_true[idx]
        y_p = y_pred[idx]
        y_prob = y_proba[idx] if y_proba is not None else None
        
        try:
            if y_prob is not None and 'proba' in metric_fn.__name__:
                score = metric_fn(y_t, y_prob)
            else:
                score = metric_fn(y_t, y_p)
            scores.append(score)
        except:
            continue
    
    if len(scores) < 100:
        return (np.nan, np.nan, np.nan)
    
    alpha = 1 - ci
    lower = np.percentile(scores, alpha/2 * 100)
    upper = np.percentile(scores, (1 - alpha/2) * 100)
    # Use RAW point estimate, not bootstrap mean
    return (raw_point, lower, upper)

def bootstrap_metric(y_true, y_pred, y_proba, metric_fn, n_bootstrap=2000, ci=0.95, seed=42, stratified=True):
    """
    Compute bootstrap confidence interval for a metric.
    Args:
        stratified: If True, use stratified bootstrap (preserves class proportions)
    Returns: (point_estimate, ci_lower, ci_upper)
    """
    if stratified:
        return stratified_bootstrap_metric(y_true, y_pred, y_proba, metric_fn, n_bootstrap, ci, seed)
    
    # Compute RAW point estimate on original data first
    try:
        if y_proba is not None and hasattr(metric_fn, '__name__') and 'proba' in metric_fn.__name__:
            raw_point = metric_fn(y_true, y_proba)
        else:
            raw_point = metric_fn(y_true, y_pred)
    except:
        raw_point = np.nan
    
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    
    for _ in pbar(range(n_bootstrap), desc="Bootstrap", leave=False):
        idx = rng.choice(n, size=n, replace=True)
        y_t = y_true[idx]
        y_p = y_pred[idx]
        y_prob = y_proba[idx] if y_proba is not None else None
        
        try:
            if y_prob is not None and 'proba' in metric_fn.__name__:
                score = metric_fn(y_t, y_prob)
            else:
                score = metric_fn(y_t, y_p)
            scores.append(score)
        except:
            continue
    
    if len(scores) < 100:
        return (np.nan, np.nan, np.nan)
    
    alpha = 1 - ci
    lower = np.percentile(scores, alpha/2 * 100)
    upper = np.percentile(scores, (1 - alpha/2) * 100)
    # Use RAW point estimate, not bootstrap mean
    return (raw_point, lower, upper)

def compute_all_metrics_with_ci(y_true, y_pred, y_proba=None, n_bootstrap=2000, ci=0.95, seed=42, stratified=True):
    """
    Compute all classification metrics with bootstrap confidence intervals.
    Returns a dictionary with metric names as keys and (point, lower, upper) as values.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_proba is not None:
        y_proba = np.asarray(y_proba)
    
    metrics = {}
    
    # Classification metrics
    def acc_fn(yt, yp): return accuracy_score(yt, yp)
    def bacc_fn(yt, yp): return balanced_accuracy_score(yt, yp)
    def f1_fn(yt, yp): return f1_score(yt, yp, zero_division=0)
    def prec_fn(yt, yp): return precision_score(yt, yp, zero_division=0)
    def rec_fn(yt, yp): return recall_score(yt, yp, zero_division=0)
    def mcc_fn(yt, yp): return matthews_corrcoef(yt, yp)
    def spec_fn(yt, yp):
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0
    def npv_fn(yt, yp):
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
        return tn / (tn + fn) if (tn + fn) > 0 else 0
    
    # Point estimates with stratified bootstrap
    metrics['Accuracy'] = bootstrap_metric(y_true, y_pred, y_proba, acc_fn, n_bootstrap, ci, seed, stratified)
    metrics['Balanced_Accuracy'] = bootstrap_metric(y_true, y_pred, y_proba, bacc_fn, n_bootstrap, ci, seed, stratified)
    metrics['F1_Score'] = bootstrap_metric(y_true, y_pred, y_proba, f1_fn, n_bootstrap, ci, seed, stratified)
    metrics['Precision'] = bootstrap_metric(y_true, y_pred, y_proba, prec_fn, n_bootstrap, ci, seed, stratified)
    metrics['Recall_Sensitivity'] = bootstrap_metric(y_true, y_pred, y_proba, rec_fn, n_bootstrap, ci, seed, stratified)
    metrics['Specificity'] = bootstrap_metric(y_true, y_pred, y_proba, spec_fn, n_bootstrap, ci, seed, stratified)
    metrics['NPV'] = bootstrap_metric(y_true, y_pred, y_proba, npv_fn, n_bootstrap, ci, seed, stratified)
    metrics['MCC'] = bootstrap_metric(y_true, y_pred, y_proba, mcc_fn, n_bootstrap, ci, seed, stratified)
    
    # Probability-based metrics
    if y_proba is not None:
        def auc_fn_proba(yt, yprob): return roc_auc_score(yt, yprob)
        def ap_fn_proba(yt, yprob): return average_precision_score(yt, yprob)
        def brier_fn_proba(yt, yprob): return brier_score_loss(yt, yprob)
        
        auc_fn_proba.__name__ = 'proba'
        ap_fn_proba.__name__ = 'proba'
        brier_fn_proba.__name__ = 'proba'
        
        metrics['ROC_AUC'] = bootstrap_metric(y_true, y_pred, y_proba, auc_fn_proba, n_bootstrap, ci, seed, stratified)
        metrics['Average_Precision'] = bootstrap_metric(y_true, y_pred, y_proba, ap_fn_proba, n_bootstrap, ci, seed, stratified)
        metrics['Brier_Score'] = bootstrap_metric(y_true, y_pred, y_proba, brier_fn_proba, n_bootstrap, ci, seed, stratified)
    
    return metrics

def format_metric_with_ci(point, lower, upper, decimals=3):
    """Format metric with CI as string."""
    if np.isnan(point):
        return "N/A"
    return f"{point:.{decimals}f} [{lower:.{decimals}f}-{upper:.{decimals}f}]"

print("Bootstrap CI functions defined (with STRATIFIED sampling option).")

In [ ]:
# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

def _find_col(cols, candidates):
    low = {c.lower().replace(" ", "").replace("_", ""): c for c in cols}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        if key in low: return low[key]
    return None

def _resolve_schema(df, target_col=None):
    cols = list(df.columns)
    # Use configured target column, or auto-detect
    if target_col and target_col in cols:
        y_col = target_col
    else:
        y_col = _find_col(cols, ["Treatment", "PFS_group", "PFS_median", "Label", "Outcome", "Target"])
        if y_col is None: y_col = cols[-1]
    id_col = _find_col(cols, ["patient_ID", "patientID", "ID", "Patient", "Sample"])
    skip = {c for c in [id_col, y_col] if c is not None}
    prot_cols = [c for c in cols if c not in skip and df[c].dtype in [np.float64, np.int64, float, int]]
    return {"y": y_col, "id": id_col, "proteins": prot_cols, "covariates": []}

def _prepare_y(series, class_labels):
    """Convert target variable to binary (0/1) using configured class labels."""
    y = series.copy()
    if y.dtype == object:
        y_str = y.astype(str).str.strip()
        label_to_code = {v.upper(): k for k, v in class_labels.items()}
        y = y_str.str.upper().map(label_to_code)
    return y.astype(int)

# Load data
print(f"Loading: {TRAIN_XLSX_PATH}")
df_train = pd.read_excel(TRAIN_XLSX_PATH, sheet_name=SHEET_NAME)
schema = _resolve_schema(df_train, target_col=TARGET_COLUMN)

print()
print("[Training Data]")
print(f"  Shape: {df_train.shape}")
print(f"  Target column: {schema['y']}")
print(f"  Class labels: {CLASS_LABELS}")
print(f"  Features: {len(schema['proteins'])}")
print(f"  Raw classes: {df_train[schema['y']].value_counts().to_dict()}")

# Randomize
if RANDOMIZE_DATA:
    df_train = df_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"  [Shuffled with seed={SEED}]")

print()
print(f"Loading: {TEST_XLSX_PATH}")
df_test = pd.read_excel(TEST_XLSX_PATH, sheet_name=SHEET_NAME)
print(f"  Shape: {df_test.shape}")
print(f"  Raw classes: {df_test[schema['y']].value_counts().to_dict()}")


In [ ]:
# =============================================================================
# DATA QUALITY ASSESSMENT (Bioinformatic Standards)
# =============================================================================
# Reference: TRIPOD+AI, Ten Quick Tips for Biomarker Discovery
# Assesses: Missing data, class balance, outliers, distributions, zero-variance features

print("\n" + "="*70)
print("[CHART] DATA QUALITY ASSESSMENT")
print("="*70)

# Prepare data for quality checks
y_train_qc = _prepare_y(df_train[schema['y']], CLASS_LABELS)
y_test_qc = _prepare_y(df_test[schema['y']], CLASS_LABELS)
X_train_qc = df_train[schema['proteins']]
X_test_qc = df_test[schema['proteins']]

# Store quality metrics for export
data_quality_results = {}

# -----------------------------------------------------------------------------
# 1. MISSING VALUE ANALYSIS
# -----------------------------------------------------------------------------
print("\n[LIST] 1. MISSING VALUE ANALYSIS")

train_missing = X_train_qc.isnull().sum()
test_missing = X_test_qc.isnull().sum()
train_missing_pct = (train_missing / len(X_train_qc) * 100)
test_missing_pct = (test_missing / len(X_test_qc) * 100)

train_total_missing = train_missing.sum()
test_total_missing = test_missing.sum()

print(f"   Training set: {train_total_missing} missing values ({train_total_missing/(X_train_qc.shape[0]*X_train_qc.shape[1])*100:.2f}%)")
print(f"   Test set: {test_total_missing} missing values ({test_total_missing/(X_test_qc.shape[0]*X_test_qc.shape[1])*100:.2f}%)")

if train_total_missing > 0:
    features_with_missing = train_missing[train_missing > 0]
    print(f"   Features with missing (train): {len(features_with_missing)}")

data_quality_results['missing_train'] = train_total_missing
data_quality_results['missing_test'] = test_total_missing
data_quality_results['missing_pct_train'] = train_total_missing/(X_train_qc.shape[0]*X_train_qc.shape[1])*100
data_quality_results['missing_pct_test'] = test_total_missing/(X_test_qc.shape[0]*X_test_qc.shape[1])*100

# -----------------------------------------------------------------------------
# 2. CLASS BALANCE ANALYSIS
# -----------------------------------------------------------------------------
print("\n[SCALE] 2. CLASS BALANCE ANALYSIS")

train_class_counts = y_train_qc.value_counts().sort_index()
test_class_counts = y_test_qc.value_counts().sort_index()
train_imbalance_ratio = train_class_counts.max() / train_class_counts.min()
test_imbalance_ratio = test_class_counts.max() / test_class_counts.min()

print(f"   Training: Class 0={train_class_counts.get(0,0)}, Class 1={train_class_counts.get(1,0)} (ratio={train_imbalance_ratio:.2f})")
print(f"   Test: Class 0={test_class_counts.get(0,0)}, Class 1={test_class_counts.get(1,0)} (ratio={test_imbalance_ratio:.2f})")

if train_imbalance_ratio > 3:
    print("   [!] WARNING: Significant class imbalance detected (>3:1)")
else:
    print("   [OK] Class balance acceptable")

data_quality_results['train_class_0'] = int(train_class_counts.get(0,0))
data_quality_results['train_class_1'] = int(train_class_counts.get(1,0))
data_quality_results['test_class_0'] = int(test_class_counts.get(0,0))
data_quality_results['test_class_1'] = int(test_class_counts.get(1,0))
data_quality_results['train_imbalance_ratio'] = train_imbalance_ratio
data_quality_results['test_imbalance_ratio'] = test_imbalance_ratio

# -----------------------------------------------------------------------------
# 3. ZERO-VARIANCE FEATURES
# -----------------------------------------------------------------------------
print("\n[CHART] 3. ZERO/LOW-VARIANCE FEATURES")

train_var = X_train_qc.var()
zero_var_features = train_var[train_var == 0].index.tolist()
low_var_threshold = train_var.quantile(0.05)
low_var_features = train_var[train_var < low_var_threshold].index.tolist()

print(f"   Zero-variance features: {len(zero_var_features)}")
print(f"   Low-variance features (<5th percentile): {len(low_var_features)}")

if len(zero_var_features) > 0:
    print(f"   [!] Zero-variance features will be excluded: {zero_var_features[:5]}...")
else:
    print("   [OK] No zero-variance features")

data_quality_results['zero_var_features'] = len(zero_var_features)
data_quality_results['low_var_features'] = len(low_var_features)

# -----------------------------------------------------------------------------
# 4. OUTLIER DETECTION (IQR Method)
# -----------------------------------------------------------------------------
print("\n[SEARCH] 4. OUTLIER DETECTION (IQR Method)")

Q1 = X_train_qc.quantile(0.25)
Q3 = X_train_qc.quantile(0.75)
IQR = Q3 - Q1
outlier_mask = ((X_train_qc < (Q1 - 1.5 * IQR)) | (X_train_qc > (Q3 + 1.5 * IQR)))
outliers_per_sample = outlier_mask.sum(axis=1)
outliers_per_feature = outlier_mask.sum(axis=0)
total_outliers = outlier_mask.sum().sum()
outlier_pct = total_outliers / (X_train_qc.shape[0] * X_train_qc.shape[1]) * 100

print(f"   Total outlier values: {total_outliers} ({outlier_pct:.2f}%)")
print(f"   Samples with >10% outliers: {(outliers_per_sample > X_train_qc.shape[1]*0.1).sum()}")
print(f"   Features with >20% outliers: {(outliers_per_feature > X_train_qc.shape[0]*0.2).sum()}")

data_quality_results['total_outliers'] = int(total_outliers)
data_quality_results['outlier_pct'] = outlier_pct
data_quality_results['samples_high_outliers'] = int((outliers_per_sample > X_train_qc.shape[1]*0.1).sum())

# -----------------------------------------------------------------------------
# 5. DISTRIBUTION NORMALITY
# -----------------------------------------------------------------------------
print("\n[CHART] 5. DISTRIBUTION ANALYSIS")

# Check skewness
skewness = X_train_qc.skew()
highly_skewed = (abs(skewness) > 2).sum()
print(f"   Highly skewed features (|skew|>2): {highly_skewed}/{len(schema['proteins'])}")

# Check if data appears log-transformed
has_negative = (X_train_qc < 0).any().any()
typical_range = (X_train_qc.median().median(), X_train_qc.max().max())
print(f"   Contains negative values: {has_negative}")
print(f"   Typical value range: {typical_range[0]:.2f} to {typical_range[1]:.2f}")

if not has_negative and typical_range[1] < 50:
    print("   [OK] Data appears to be log2-transformed (recommended for MS data)")
else:
    print("   [!] Check if data is properly log-transformed")

data_quality_results['highly_skewed_features'] = int(highly_skewed)
data_quality_results['has_negative_values'] = has_negative
data_quality_results['median_value'] = float(X_train_qc.median().median())
data_quality_results['max_value'] = float(X_train_qc.max().max())

# -----------------------------------------------------------------------------
# 6. SAMPLE SIZE ADEQUACY
# -----------------------------------------------------------------------------
print("\n[RULER] 6. SAMPLE SIZE ASSESSMENT")

n_train = len(y_train_qc)
n_features = len(schema['proteins'])
events_per_variable = min(train_class_counts) / MAX_SIGNATURE

print(f"   Training samples: {n_train}")
print(f"   Total features: {n_features}")
print(f"   Feature:sample ratio: 1:{n_train/n_features:.1f}")
print(f"   Events per variable (EPV): {events_per_variable:.1f}")

if events_per_variable < 10:
    print("   [!] EPV < 10: Risk of overfitting (TRIPOD guideline)")
else:
    print("   [OK] EPV adequate for reliable modeling")

data_quality_results['n_train'] = n_train
data_quality_results['n_test'] = len(y_test_qc)
data_quality_results['n_features'] = n_features
data_quality_results['feature_sample_ratio'] = n_features / n_train
data_quality_results['events_per_variable'] = events_per_variable

# -----------------------------------------------------------------------------
# 7. TRAIN-TEST DISTRIBUTION COMPARISON
# -----------------------------------------------------------------------------
print("\n[SYNC] 7. TRAIN-TEST DISTRIBUTION COMPARISON")

train_means = X_train_qc.mean()
test_means = X_test_qc.mean()
mean_diff = abs(train_means - test_means)
train_std = X_train_qc.std()
standardized_diff = mean_diff / (train_std + 1e-10)

large_diff_features = (standardized_diff > 0.5).sum()
print(f"   Features with |standardized mean diff| > 0.5: {large_diff_features}/{n_features}")

if large_diff_features > n_features * 0.2:
    print("   [!] Potential batch effect or distribution shift detected")
else:
    print("   [OK] Train-test distributions appear similar")

data_quality_results['features_large_shift'] = int(large_diff_features)

# =============================================================================
# DATA QUALITY VISUALIZATION (Publication-Ready)
# =============================================================================

print("\n" + "="*70)
print("[CHART] GENERATING DATA QUALITY VISUALIZATIONS")
print("="*70)

fig_dq = plt.figure(figsize=(20, 16))
gs_dq = GridSpec(3, 3, figure=fig_dq, hspace=0.35, wspace=0.3)

# 1. Class Distribution (Train vs Test)
ax1 = fig_dq.add_subplot(gs_dq[0, 0])
x_pos = np.arange(2)
width = 0.35
bars1 = ax1.bar(x_pos - width/2, [train_class_counts.get(0,0), train_class_counts.get(1,0)],
                width, label='Train', color='steelblue', edgecolor='black')
bars2 = ax1.bar(x_pos + width/2, [test_class_counts.get(0,0), test_class_counts.get(1,0)],
                width, label='Test', color='coral', edgecolor='black')
ax1.set_xlabel('Class', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'Class 0 ({CLASS_LABELS[0]})', f'Class 1 ({CLASS_LABELS[1]})'])
ax1.legend()
for bar in bars1 + bars2:
    height = bar.get_height()
    ax1.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

# 2. Feature Value Distribution (Histogram)
ax2 = fig_dq.add_subplot(gs_dq[0, 1])
all_values = X_train_qc.values.flatten()
all_values = all_values[~np.isnan(all_values)]
ax2.hist(all_values, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(np.median(all_values), color='red', linestyle='--', linewidth=2, label=f'Median: {np.median(all_values):.2f}')
ax2.set_xlabel('Feature Values (log2 intensity)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Overall Feature Distribution', fontsize=14, fontweight='bold')
ax2.legend()

# 3. Variance Distribution
ax3 = fig_dq.add_subplot(gs_dq[0, 2])
if train_total_missing > 0:
    missing_matrix = X_train_qc.isnull().astype(int)
    sns.heatmap(missing_matrix.iloc[:, :min(50, len(schema['proteins']))],
                cmap='YlOrRd', cbar_kws={'label': 'Missing'}, ax=ax3)
    ax3.set_title('Missing Values Pattern', fontsize=14, fontweight='bold')
else:
    ax3.hist(np.log10(train_var + 1e-10), bins=30, color='green', edgecolor='black', alpha=0.7)
    ax3.axvline(np.log10(low_var_threshold + 1e-10), color='red', linestyle='--',
                linewidth=2, label=f'5th percentile')
    ax3.set_xlabel('log10(Variance)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Feature Variance Distribution', fontsize=14, fontweight='bold')
    ax3.legend()

# 4. Outliers per Sample
ax4 = fig_dq.add_subplot(gs_dq[1, 0])
ax4.hist(outliers_per_sample, bins=20, color='orange', edgecolor='black', alpha=0.7)
ax4.axvline(X_train_qc.shape[1]*0.1, color='red', linestyle='--', linewidth=2,
            label=f'10% threshold ({int(X_train_qc.shape[1]*0.1)} features)')
ax4.set_xlabel('Number of Outlier Features per Sample', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.set_title('Outlier Distribution (per Sample)', fontsize=14, fontweight='bold')
ax4.legend()

# 5. Skewness Distribution
ax5 = fig_dq.add_subplot(gs_dq[1, 1])
ax5.hist(skewness, bins=30, color='purple', edgecolor='black', alpha=0.7)
ax5.axvline(-2, color='red', linestyle='--', linewidth=2)
ax5.axvline(2, color='red', linestyle='--', linewidth=2, label='|Skew|=2 threshold')
ax5.set_xlabel('Skewness', fontsize=12)
ax5.set_ylabel('Frequency', fontsize=12)
ax5.set_title('Feature Skewness Distribution', fontsize=14, fontweight='bold')
ax5.legend()

# 6. Train-Test Mean Comparison
ax6 = fig_dq.add_subplot(gs_dq[1, 2])
ax6.scatter(train_means, test_means, alpha=0.5, s=20, c='steelblue')
lims = [min(train_means.min(), test_means.min()), max(train_means.max(), test_means.max())]
ax6.plot(lims, lims, 'r--', linewidth=2, label='y=x (perfect agreement)')
ax6.set_xlabel('Training Set Mean', fontsize=12)
ax6.set_ylabel('Test Set Mean', fontsize=12)
ax6.set_title('Train-Test Feature Means', fontsize=14, fontweight='bold')
ax6.legend()

# 7. Sample Size Summary
ax7 = fig_dq.add_subplot(gs_dq[2, 0])
categories = ['Training\nSamples', 'Test\nSamples', 'Total\nFeatures', 'Min Class\n(Events)']
values = [n_train, len(y_test_qc), n_features, int(min(train_class_counts))]
colors = ['steelblue', 'coral', 'green', 'orange']
bars = ax7.bar(categories, values, color=colors, edgecolor='black')
ax7.set_ylabel('Count', fontsize=12)
ax7.set_title('Dataset Size Summary', fontsize=14, fontweight='bold')
for bar, val in zip(bars, values):
    ax7.annotate(f'{val}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=11, fontweight='bold')

# 8. Correlation of top variance features
ax8 = fig_dq.add_subplot(gs_dq[2, 1])
top_var_features = train_var.nlargest(10).index.tolist()
corr_subset = X_train_qc[top_var_features].corr()
mask = np.triu(np.ones_like(corr_subset, dtype=bool), k=1)
sns.heatmap(corr_subset, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', ax=ax8, cbar_kws={'label': 'Correlation'},
            annot_kws={'size': 8})
ax8.set_title('Top 10 Variance Features\nCorrelation Matrix', fontsize=14, fontweight='bold')

# 9. Quality Summary Scorecard
ax9 = fig_dq.add_subplot(gs_dq[2, 2])
ax9.axis('off')

quality_checks = [
    ('Missing Values', 'PASS' if train_total_missing == 0 else 'WARN'),
    ('Class Balance', 'PASS' if train_imbalance_ratio <= 3 else 'WARN'),
    ('Zero Variance', 'PASS' if len(zero_var_features) == 0 else 'WARN'),
    ('Outliers', 'PASS' if outlier_pct < 5 else 'WARN'),
    ('EPV Adequacy', 'PASS' if events_per_variable >= 10 else 'WARN'),
    ('Train-Test Shift', 'PASS' if large_diff_features < n_features*0.2 else 'WARN'),
    ('Log Transform', 'PASS' if not has_negative and typical_range[1] < 50 else 'WARN')
]

scorecard_text = "DATA QUALITY SCORECARD\n" + "="*30 + "\n\n"
for check, result in quality_checks:
    symbol = "OK" if result == 'PASS' else "!"
    scorecard_text += f"[{symbol}] {check}: {result}\n"

ax9.text(0.1, 0.9, scorecard_text, transform=ax9.transAxes, fontsize=12,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.suptitle('Data Quality Assessment Report', fontsize=18, fontweight='bold', y=0.98)

# Save figure
savefig(OUTDIR, "data_quality_assessment", fig_dq)
plt.show()

# Store quality results for export
data_quality_df = pd.DataFrame([data_quality_results])
print("\n[OK] Data quality assessment complete. Results stored for export.")


In [ ]:
# =============================================================================
# CLASSIFIER CONFIGURATIONS WITH HYPERPARAMETER GRIDS
# =============================================================================

def get_estimator_configs():
    """
    Returns dict of classifier configurations with hyperparameter grids for GridSearchCV.
    Each entry: {name: {"estimator": clf, "param_grid": {params}}}
    """
    return {
        "NB": {
            "estimator": GaussianNB(),
            "param_grid": {"var_smoothing": [1e-9, 1e-8, 1e-7]}
        },
        "kNN": {
            "estimator": KNeighborsClassifier(),
            "param_grid": {"n_neighbors": [1, 3, 5], "weights": ["uniform", "distance"]}
        },
        "DT": {
            "estimator": DecisionTreeClassifier(criterion="entropy", random_state=SEED),
            "param_grid": {"max_depth": [None, 3, 5, 7], "min_samples_leaf": [1, 2, 3]}
        },
        "LR": {
            "estimator": LogisticRegression(solver="liblinear", max_iter=500, random_state=SEED),
            "param_grid": {"C": [0.01, 0.1, 1.0, 10.0], "penalty": ["l1", "l2"]}
        },
        "SVM": {
            "estimator": SVC(kernel="linear", probability=True, random_state=SEED, max_iter=2000),
            "param_grid": {"C": [0.01, 0.1, 1.0, 10.0]}
        },
        "RF": {
            "estimator": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
            "param_grid": {"max_depth": [None, 5, 10], "min_samples_leaf": [1, 2]}
        }
    }

# =============================================================================
# FEATURE FILTERING FUNCTIONS
# =============================================================================

def _anova_filter(X, y, protein_cols, top_k):
    """ANOVA F-test feature filtering (original method)."""
    P = X[protein_cols].values
    F, p = f_classif(P, y.values)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
    order = np.argsort(-F)
    k = min(top_k, len(protein_cols))
    keep = [protein_cols[i] for i in order[:k]]
    fmap = {protein_cols[i]: float(F[i]) for i in range(len(protein_cols))}
    return keep, fmap

def _mutual_info_scores(X, y, protein_cols):
    """Calculate mutual information scores for each feature."""
    from sklearn.feature_selection import mutual_info_classif
    P = X[protein_cols].values
    mi_scores = mutual_info_classif(P, y.values, random_state=SEED, n_neighbors=3)
    mi_scores = np.nan_to_num(mi_scores, nan=0.0, posinf=0.0, neginf=0.0)
    return {protein_cols[i]: float(mi_scores[i]) for i in range(len(protein_cols))}

def _correlation_scores(X, y, protein_cols):
    """Calculate absolute Pearson correlation with target."""
    corr_scores = {}
    y_vals = y.values.astype(float)
    for col in protein_cols:
        try:
            corr = np.abs(np.corrcoef(X[col].values, y_vals)[0, 1])
            corr_scores[col] = float(corr) if not np.isnan(corr) else 0.0
        except:
            corr_scores[col] = 0.0
    return corr_scores

def _multi_filter_ensemble(X, y, protein_cols, top_k, weights=None):
    """
    Multi-filter ensemble: combines ANOVA + Mutual Information + Correlation.
    Uses rank aggregation (average rank) weighted by filter importance.

    Parameters:
    -----------
    X : DataFrame - Feature matrix
    y : Series - Target labels
    protein_cols : list - Column names to consider
    top_k : int - Number of top features to return
    weights : dict - Weights for each filter method (default: equal weights)

    Returns:
    --------
    keep : list - Top-k features by ensemble ranking
    ensemble_scores : dict - Combined scores for each feature
    """
    if weights is None:
        weights = {'anova': 0.4, 'mutual_info': 0.4, 'correlation': 0.2}

    # Normalize weights
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}

    # Get scores from each filter
    _, anova_scores = _anova_filter(X, y, protein_cols, len(protein_cols))
    mi_scores = _mutual_info_scores(X, y, protein_cols)
    corr_scores = _correlation_scores(X, y, protein_cols)

    # Convert to ranks (higher score = lower rank = better)
    def scores_to_ranks(scores_dict):
        sorted_items = sorted(scores_dict.items(), key=lambda x: -x[1])
        return {item[0]: rank + 1 for rank, item in enumerate(sorted_items)}

    anova_ranks = scores_to_ranks(anova_scores)
    mi_ranks = scores_to_ranks(mi_scores)
    corr_ranks = scores_to_ranks(corr_scores)

    # Weighted average rank (lower is better)
    ensemble_ranks = {}
    for col in protein_cols:
        weighted_rank = (
            weights['anova'] * anova_ranks[col] +
            weights['mutual_info'] * mi_ranks[col] +
            weights['correlation'] * corr_ranks[col]
        )
        ensemble_ranks[col] = weighted_rank

    # Sort by ensemble rank (ascending - lower rank is better)
    sorted_features = sorted(ensemble_ranks.items(), key=lambda x: x[1])

    k = min(top_k, len(protein_cols))
    keep = [f[0] for f in sorted_features[:k]]

    # Create ensemble scores (inverse of rank for compatibility with existing code)
    max_rank = len(protein_cols)
    ensemble_scores = {col: max_rank - rank + 1 for col, rank in ensemble_ranks.items()}

    return keep, ensemble_scores

def _feature_filter(X, y, protein_cols, top_k, use_multi_filter=False, filter_weights=None):
    """
    Unified feature filtering function.
    Dispatches to ANOVA-only or multi-filter ensemble based on configuration.
    """
    if use_multi_filter:
        return _multi_filter_ensemble(X, y, protein_cols, top_k, filter_weights)
    else:
        return _anova_filter(X, y, protein_cols, top_k)

# =============================================================================
# STABILITY SELECTION (Meinshausen & Buhlmann, 2010)
# =============================================================================

def _stability_selection(X, y, protein_cols, top_k,
                         n_bootstrap=50, threshold=0.6, subsample_ratio=0.8,
                         use_multi_filter=False, filter_weights=None, seed=42):
    """
    Stability Selection for robust feature filtering.

    Runs feature selection on multiple bootstrap subsamples and returns
    features that are consistently selected across iterations.

    Parameters:
    -----------
    X : DataFrame - Feature matrix
    y : Series - Target labels
    protein_cols : list - Column names to consider
    top_k : int - Number of top features per bootstrap iteration
    n_bootstrap : int - Number of bootstrap iterations
    threshold : float - Minimum selection frequency (0.0-1.0)
    subsample_ratio : float - Fraction of samples per bootstrap
    use_multi_filter : bool - Use multi-filter ensemble vs ANOVA only
    filter_weights : dict - Weights for multi-filter (if used)
    seed : int - Random seed

    Returns:
    --------
    stable_features : list - Features passing stability threshold, sorted by frequency
    selection_freq : dict - Selection frequency for each feature
    filter_scores : dict - Average filter scores across bootstraps
    """
    rng = np.random.RandomState(seed)
    n_samples = len(X)
    subsample_size = int(n_samples * subsample_ratio)

    # Track selection counts and cumulative scores
    selection_counts = {col: 0 for col in protein_cols}
    cumulative_scores = {col: 0.0 for col in protein_cols}

    for b in range(n_bootstrap):
        # Stratified bootstrap subsample
        # Get indices for each class
        class_0_idx = np.where(y.values == 0)[0]
        class_1_idx = np.where(y.values == 1)[0]

        # Subsample from each class proportionally
        n_class_0 = int(len(class_0_idx) * subsample_ratio)
        n_class_1 = int(len(class_1_idx) * subsample_ratio)

        # Ensure at least 1 sample from each class
        n_class_0 = max(1, n_class_0)
        n_class_1 = max(1, n_class_1)

        boot_idx_0 = rng.choice(class_0_idx, size=n_class_0, replace=True)
        boot_idx_1 = rng.choice(class_1_idx, size=n_class_1, replace=True)
        boot_idx = np.concatenate([boot_idx_0, boot_idx_1])

        X_boot = X.iloc[boot_idx].reset_index(drop=True)
        y_boot = y.iloc[boot_idx].reset_index(drop=True)

        # Run feature filtering on bootstrap sample
        selected, scores = _feature_filter(
            X_boot, y_boot, protein_cols, top_k,
            use_multi_filter=use_multi_filter,
            filter_weights=filter_weights
        )

        # Update counts and scores
        for feat in selected:
            selection_counts[feat] += 1
        for feat, score in scores.items():
            cumulative_scores[feat] += score

    # Calculate selection frequencies
    selection_freq = {col: count / n_bootstrap for col, count in selection_counts.items()}

    # Calculate average scores
    avg_scores = {col: score / n_bootstrap for col, score in cumulative_scores.items()}

    # Filter by threshold and sort by frequency (descending), then by score
    stable_features = [
        (col, selection_freq[col], avg_scores[col])
        for col in protein_cols
        if selection_freq[col] >= threshold
    ]

    # Sort by frequency first, then by average score
    stable_features.sort(key=lambda x: (-x[1], -x[2]))

    stable_feature_names = [f[0] for f in stable_features]

    return stable_feature_names, selection_freq, avg_scores

def get_filtered_features(X, y, protein_cols, top_k,
                          use_stability=True, use_multi_filter=False,
                          stability_n_bootstrap=50, stability_threshold=0.6,
                          stability_subsample_ratio=0.8, filter_weights=None, seed=42):
    """
    Main feature filtering function that combines stability selection and multi-filter.

    This is the primary entry point for feature filtering in the nested CV loop.

    Returns:
    --------
    filtered_features : list - Features to use for wrapper selection
    feature_scores : dict - Scores/frequencies for each feature
    stability_info : dict - Additional stability selection info (if used)
    """
    stability_info = {}

    if use_stability:
        # Run stability selection
        stable_features, selection_freq, avg_scores = _stability_selection(
            X, y, protein_cols, top_k,
            n_bootstrap=stability_n_bootstrap,
            threshold=stability_threshold,
            subsample_ratio=stability_subsample_ratio,
            use_multi_filter=use_multi_filter,
            filter_weights=filter_weights,
            seed=seed
        )

        stability_info = {
            'selection_frequencies': selection_freq,
            'average_scores': avg_scores,
            'n_stable_features': len(stable_features),
            'threshold': stability_threshold
        }

        # If too few features pass threshold, take top-k by frequency
        if len(stable_features) < 5:
            # Sort all by frequency and take top-k
            sorted_by_freq = sorted(selection_freq.items(), key=lambda x: -x[1])
            stable_features = [f[0] for f in sorted_by_freq[:top_k]]
            stability_info['fallback_used'] = True

        return stable_features, selection_freq, stability_info

    else:
        # No stability selection - use direct filtering
        filtered, scores = _feature_filter(
            X, y, protein_cols, top_k,
            use_multi_filter=use_multi_filter,
            filter_weights=filter_weights
        )
        return filtered, scores, stability_info

print("Classifier configurations defined.")
print("Feature selection functions defined:")
print("  - _anova_filter (original)")
print("  - _mutual_info_scores (new)")
print("  - _correlation_scores (new)")
print("  - _multi_filter_ensemble (new - Option B)")
print("  - _stability_selection (new - Option A)")
print("  - get_filtered_features (unified entry point)")
print("Algorithms:", list(get_estimator_configs().keys()))

# =============================================================================
# ELASTIC NET FEATURE SELECTION (Option C)
# =============================================================================

def _elastic_net_filter(X, y, protein_cols, top_k=20, l1_ratio=0.5, n_alphas=50, cv_folds=5, seed=42):
    """
    Elastic Net (L1+L2 regularization) based feature selection.

    Uses cross-validated Elastic Net to find optimal regularization,
    then selects features with non-zero coefficients.

    Parameters:
    -----------
    X : DataFrame - Feature matrix
    y : Series - Target labels
    protein_cols : list - Column names to consider
    top_k : int - Maximum number of features to return
    l1_ratio : float - Balance between L1 and L2 (0=Ridge, 1=Lasso, 0.5=balanced)
    n_alphas : int - Number of alpha values to try
    cv_folds : int - Number of CV folds for alpha selection
    seed : int - Random seed

    Returns:
    --------
    selected_features : list - Features with non-zero coefficients (up to top_k)
    feature_importance : dict - Absolute coefficient values for each feature
    enet_info : dict - Model information (alpha, l1_ratio, n_selected)
    """
    from sklearn.linear_model import ElasticNetCV
    from sklearn.preprocessing import StandardScaler

    # Prepare data
    P = X[protein_cols].values

    # Scale features (required for regularization to work properly)
    scaler = StandardScaler()
    P_scaled = scaler.fit_transform(P)

    # Convert labels to numeric
    y_vals = y.values.astype(float)

    # Fit Elastic Net with CV for alpha selection
    enet = ElasticNetCV(
        l1_ratio=l1_ratio,
        n_alphas=n_alphas,
        cv=cv_folds,
        random_state=seed,
        max_iter=5000,
        selection='random'  # Faster convergence
    )

    try:
        enet.fit(P_scaled, y_vals)

        # Get coefficients
        coefs = np.abs(enet.coef_)

        # Create importance dict
        feature_importance = {protein_cols[i]: float(coefs[i]) for i in range(len(protein_cols))}

        # Select non-zero coefficients
        nonzero_mask = coefs > 1e-10
        nonzero_features = [(protein_cols[i], coefs[i]) for i in range(len(protein_cols)) if nonzero_mask[i]]

        # Sort by absolute coefficient value (descending)
        nonzero_features.sort(key=lambda x: -x[1])

        # Take top-k
        selected_features = [f[0] for f in nonzero_features[:top_k]]

        enet_info = {
            'alpha': enet.alpha_,
            'l1_ratio': l1_ratio,
            'n_nonzero': int(np.sum(nonzero_mask)),
            'n_selected': len(selected_features)
        }

        return selected_features, feature_importance, enet_info

    except Exception as e:
        # Fallback to ANOVA if Elastic Net fails
        print(f"  [!] Elastic Net failed ({str(e)}), falling back to ANOVA")
        keep, fmap = _anova_filter(X, y, protein_cols, top_k)
        return keep, fmap, {'error': str(e), 'fallback': 'anova'}

def get_filtered_features_v2(X, y, protein_cols, top_k,
                              use_stability=True, use_multi_filter=False,
                              use_elastic_net=False, elastic_net_mode='pre_filter',
                              stability_n_bootstrap=50, stability_threshold=0.6,
                              stability_subsample_ratio=0.8, filter_weights=None,
                              elastic_net_l1_ratio=0.5, elastic_net_n_alphas=50,
                              elastic_net_cv_folds=5, elastic_net_top_k=20,
                              seed=42):
    """
    Enhanced feature filtering with all options: Stability + Multi-Filter + Elastic Net.

    This is the v2 entry point supporting all feature selection methods.

    Parameters:
    -----------
    use_stability : bool - Apply stability selection
    use_multi_filter : bool - Use multi-filter ensemble (vs ANOVA only)
    use_elastic_net : bool - Use Elastic Net filtering
    elastic_net_mode : str - 'replace_wrapper' or 'pre_filter'

    Returns:
    --------
    filtered_features : list - Features to use for wrapper selection
    feature_scores : dict - Scores/frequencies for each feature
    selection_info : dict - Detailed info about selection process
    """
    selection_info = {
        'stability_used': use_stability,
        'multi_filter_used': use_multi_filter,
        'elastic_net_used': use_elastic_net
    }

    working_features = protein_cols.copy()
    feature_scores = {}

    # Step 1: Elastic Net pre-filtering (if enabled and mode is 'pre_filter')
    if use_elastic_net and elastic_net_mode == 'pre_filter':
        enet_features, enet_scores, enet_info = _elastic_net_filter(
            X, y, working_features,
            top_k=elastic_net_top_k,
            l1_ratio=elastic_net_l1_ratio,
            n_alphas=elastic_net_n_alphas,
            cv_folds=elastic_net_cv_folds,
            seed=seed
        )
        working_features = enet_features if enet_features else working_features
        selection_info['elastic_net_info'] = enet_info
        selection_info['elastic_net_scores'] = enet_scores
        vlog(f"  Elastic Net pre-filter: {len(enet_features)} features selected")

    # Step 2: Elastic Net as replacement for wrapper (skip to final if this mode)
    if use_elastic_net and elastic_net_mode == 'replace_wrapper':
        enet_features, enet_scores, enet_info = _elastic_net_filter(
            X, y, working_features,
            top_k=elastic_net_top_k,
            l1_ratio=elastic_net_l1_ratio,
            n_alphas=elastic_net_n_alphas,
            cv_folds=elastic_net_cv_folds,
            seed=seed
        )
        selection_info['elastic_net_info'] = enet_info
        selection_info['elastic_net_scores'] = enet_scores
        selection_info['wrapper_replaced'] = True
        return enet_features, enet_scores, selection_info

    # Step 3: Apply stability selection or direct filtering
    if use_stability:
        stable_features, selection_freq, avg_scores = _stability_selection(
            X, y, working_features, top_k,
            n_bootstrap=stability_n_bootstrap,
            threshold=stability_threshold,
            subsample_ratio=stability_subsample_ratio,
            use_multi_filter=use_multi_filter,
            filter_weights=filter_weights,
            seed=seed
        )

        selection_info['stability_info'] = {
            'n_stable_features': len(stable_features),
            'threshold': stability_threshold
        }
        selection_info['selection_frequencies'] = selection_freq

        # If too few features pass threshold, take top by frequency
        if len(stable_features) < 5:
            sorted_by_freq = sorted(selection_freq.items(), key=lambda x: -x[1])
            stable_features = [f[0] for f in sorted_by_freq[:top_k]]
            selection_info['stability_info']['fallback_used'] = True

        return stable_features, selection_freq, selection_info

    else:
        # No stability - use direct filtering
        filtered, scores = _feature_filter(
            X, y, working_features, top_k,
            use_multi_filter=use_multi_filter,
            filter_weights=filter_weights
        )
        return filtered, scores, selection_info

print("  - _elastic_net_filter (new - Option C)")
print("  - get_filtered_features_v2 (unified with all options)")


In [ ]:
# =============================================================================
# NESTED CV WITH GRID SEARCH - ENHANCED FEATURE SELECTION (v1.8)
# =============================================================================

def nested_cv_with_gridsearch(
    df, label_col, protein_cols,
    min_sig=2, max_sig=5, top_k=50, candidates_per_step=25,
    inner_splits=3, outer_splits=3, outer_repeats=10,
    optimization_metric="f1", seed=42, eps_improve=1e-6,
    # NEW: Enhanced feature selection parameters
    use_stability=True, use_multi_filter=False, use_elastic_net=False,
    elastic_net_mode='pre_filter',
    stability_n_bootstrap=50, stability_threshold=0.6, stability_subsample_ratio=0.8,
    filter_weights=None,
    elastic_net_l1_ratio=0.5, elastic_net_n_alphas=50, elastic_net_cv_folds=5,
    elastic_net_top_k=20
):
    """
    Publication-grade nested CV with enhanced feature selection (v1.8):

    Feature Selection Options:
    - Option A: Stability Selection (bootstrap-based robust filtering)
    - Option B: Multi-Filter Ensemble (ANOVA + MI + Correlation)
    - Option C: Elastic Net regularization-based filtering

    Wrapper:
    - Greedy forward feature selection
    - Grid search hyperparameter tuning in inner loop
    - F1 optimization (configurable)
    """
    vlog(f"Starting nested CV v1.8 (optimization metric: {optimization_metric.upper()})")
    vlog(f"  Feature selection: stability={use_stability}, multi_filter={use_multi_filter}, elastic_net={use_elastic_net}")

    y = _prepare_y(df[label_col], CLASS_LABELS)
    Xall = df.copy()
    for c in ["patient_ID", "patientID", "Patient", "ID"]:
        if c in Xall.columns: Xall = Xall.drop(columns=[c])

    # Scoring
    if optimization_metric == "f1":
        scorer = make_scorer(f1_score, zero_division=0)
        metric_fn = lambda yt, yp: f1_score(yt, yp, zero_division=0)
    elif optimization_metric == "balanced_accuracy":
        scorer = make_scorer(balanced_accuracy_score)
        metric_fn = balanced_accuracy_score
    else:
        scorer = make_scorer(accuracy_score)
        metric_fn = accuracy_score

    # Storage
    all_fold_results = []
    outer_true, outer_pred, outer_proba = [], [], []
    fold_metrics_over_time = []
    feature_selection_info = []  # NEW: Track selection info per fold

    est_configs = get_estimator_configs()
    rskf = RepeatedStratifiedKFold(n_splits=outer_splits, n_repeats=outer_repeats, random_state=seed)
    total_outer = outer_splits * outer_repeats

    vlog(f"Outer CV: {outer_splits}-fold x {outer_repeats} repeats = {total_outer} evaluations")

    outer_iter = list(rskf.split(Xall, y))

    for out_i, (tr_idx, te_idx) in enumerate(pbar(outer_iter, desc="Outer CV", total=total_outer, leave=True)):
        X_tr = Xall.iloc[tr_idx].reset_index(drop=True)
        y_tr = y.iloc[tr_idx].reset_index(drop=True)
        X_te = Xall.iloc[te_idx].reset_index(drop=True)
        y_te = y.iloc[te_idx].reset_index(drop=True)

        vlog(f"[Outer {out_i+1}/{total_outer}] Train n={len(X_tr)}, Test n={len(X_te)}")

        # =====================================================================
        # ENHANCED FEATURE FILTERING (v1.8)
        # =====================================================================
        filtered_prots, fmap, sel_info = get_filtered_features_v2(
            X_tr, y_tr, protein_cols, top_k,
            use_stability=use_stability,
            use_multi_filter=use_multi_filter,
            use_elastic_net=use_elastic_net,
            elastic_net_mode=elastic_net_mode,
            stability_n_bootstrap=stability_n_bootstrap,
            stability_threshold=stability_threshold,
            stability_subsample_ratio=stability_subsample_ratio,
            filter_weights=filter_weights,
            elastic_net_l1_ratio=elastic_net_l1_ratio,
            elastic_net_n_alphas=elastic_net_n_alphas,
            elastic_net_cv_folds=elastic_net_cv_folds,
            elastic_net_top_k=elastic_net_top_k,
            seed=seed
        )

        sel_info['fold'] = out_i + 1
        feature_selection_info.append(sel_info)

        vlog(f"[Outer {out_i+1}] Feature filter -> {len(filtered_prots)} features")

        # Check if Elastic Net replaced the wrapper
        if sel_info.get('wrapper_replaced', False):
            # Elastic Net already selected final features - skip wrapper
            vlog(f"[Outer {out_i+1}] Elastic Net mode: wrapper replaced, using {len(filtered_prots)} features directly")

            # Still need to select best algorithm via inner CV
            inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)
            best_inner_score = -np.inf
            best_record = None

            # Limit to signature size constraints
            features_to_use = filtered_prots[:max_sig] if len(filtered_prots) > max_sig else filtered_prots
            if len(features_to_use) < min_sig:
                features_to_use = filtered_prots[:min_sig] if len(filtered_prots) >= min_sig else filtered_prots

            for algo_name, config in est_configs.items():
                X_sub = X_tr[features_to_use].values

                scaler = make_scaler(SCALER_KIND)
                if scaler:
                    pipe = Pipeline([("scaler", scaler), ("clf", clone(config["estimator"]))])
                    param_grid = {f"clf__{k}": v for k, v in config["param_grid"].items()}
                else:
                    pipe = Pipeline([("clf", clone(config["estimator"]))])
                    param_grid = {f"clf__{k}": v for k, v in config["param_grid"].items()}

                grid = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring=scorer,
                                   refit=True, n_jobs=-1, error_score=0)
                try:
                    grid.fit(X_sub, y_tr)
                    if grid.best_score_ > best_inner_score:
                        best_inner_score = grid.best_score_
                        best_record = {
                            "algo": algo_name,
                            "features": list(features_to_use),
                            "params": grid.best_params_,
                            "estimator": grid.best_estimator_,
                            "inner_score": grid.best_score_
                        }
                except:
                    continue

        else:
            # Standard wrapper-based feature selection
            inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)

            best_inner_score = -np.inf
            best_record = None

            # Try each algorithm with grid search
            for algo_name, config in pbar(est_configs.items(), desc="  Algorithms", leave=False):
                candidates_sorted = sorted(filtered_prots, key=lambda f: fmap.get(f, 0.0), reverse=True)
                selected = []
                prev_score = -np.inf
                step_records = []

                # Greedy forward selection
                for step in range(1, max_sig + 1):
                    remaining = [f for f in candidates_sorted if f not in selected]
                    try_list = remaining[:min(candidates_per_step, len(remaining))]
                    if not try_list: break

                    best_step_score, best_step_feat = -np.inf, None
                    best_step_params = None
                    best_step_estimator = None

                    for f in try_list:
                        trial = selected + [f]
                        X_sub = X_tr[trial].values

                        scaler = make_scaler(SCALER_KIND)
                        if scaler:
                            pipe = Pipeline([("scaler", scaler), ("clf", clone(config["estimator"]))])
                            param_grid = {f"clf__{k}": v for k, v in config["param_grid"].items()}
                        else:
                            pipe = Pipeline([("clf", clone(config["estimator"]))])
                            param_grid = {f"clf__{k}": v for k, v in config["param_grid"].items()}

                        grid = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring=scorer,
                                           refit=True, n_jobs=-1, error_score=0)
                        try:
                            grid.fit(X_sub, y_tr)
                            mean_score = grid.best_score_
                            if mean_score > best_step_score:
                                best_step_score = mean_score
                                best_step_feat = f
                                best_step_params = grid.best_params_
                                best_step_estimator = grid.best_estimator_
                        except:
                            continue

                    if best_step_feat is None: break

                    selected.append(best_step_feat)
                    step_records.append({
                        "features": list(selected),
                        "score": best_step_score,
                        "params": best_step_params,
                        "estimator": best_step_estimator
                    })

                    if HAS_TQDM:
                        tqdm.write(f"      {algo_name} +{best_step_feat} -> inner {optimization_metric}={best_step_score:.3f}")

                    if (prev_score > -np.inf) and ((best_step_score - prev_score) < eps_improve):
                        break
                    prev_score = best_step_score

                # Select best within signature constraints
                feasible = [r for r in step_records if min_sig <= len(r["features"]) <= max_sig]
                if not feasible: continue

                local_best = max(feasible, key=lambda r: r["score"])
                if local_best["score"] > best_inner_score:
                    best_inner_score = local_best["score"]
                    best_record = {
                        "algo": algo_name,
                        "features": local_best["features"],
                        "params": local_best["params"],
                        "estimator": local_best["estimator"],
                        "inner_score": local_best["score"]
                    }

        if best_record is None:
            vlog(f"[Outer {out_i+1}] WARNING: No valid model found!")
            continue

        # Evaluate on outer test fold
        features = best_record["features"]
        estimator = best_record["estimator"]

        estimator.fit(X_tr[features].values, y_tr)
        yhat = estimator.predict(X_te[features].values)

        try:
            yprob = estimator.predict_proba(X_te[features].values)[:, 1]
        except:
            yprob = np.full(len(y_te), 0.5)

        outer_true.extend(y_te.tolist())
        outer_pred.extend(yhat.tolist())
        outer_proba.extend(yprob.tolist())

        # Compute metrics
        outer_f1 = f1_score(y_te, yhat, zero_division=0)
        outer_acc = accuracy_score(y_te, yhat)
        outer_bacc = balanced_accuracy_score(y_te, yhat)
        outer_mcc = matthews_corrcoef(y_te, yhat)
        try:
            outer_auc = roc_auc_score(y_te, yprob)
        except:
            outer_auc = np.nan

        fold_result = {
            "fold": out_i + 1,
            "algo": best_record["algo"],
            "n_features": len(features),
            "features": features,
            "best_params": best_record["params"],
            "inner_f1": best_record["inner_score"],
            "outer_f1": outer_f1,
            "outer_accuracy": outer_acc,
            "outer_balanced_accuracy": outer_bacc,
            "outer_mcc": outer_mcc,
            "outer_auc": outer_auc,
            "estimator": estimator
        }
        all_fold_results.append(fold_result)

        fold_metrics_over_time.append({
            "fold": out_i + 1,
            "f1": outer_f1,
            "accuracy": outer_acc,
            "balanced_accuracy": outer_bacc,
            "auc": outer_auc,
            "mcc": outer_mcc
        })

        vlog(f"[Outer {out_i+1}] Champion: {best_record['algo']} k={len(features)} | "
             f"inner_{optimization_metric}={best_record['inner_score']:.3f} | outer_F1={outer_f1:.3f}")

    vlog("Nested CV complete.")

    return {
        "fold_results": all_fold_results,
        "oof_true": np.array(outer_true),
        "oof_pred": np.array(outer_pred),
        "oof_proba": np.array(outer_proba),
        "fold_metrics": pd.DataFrame(fold_metrics_over_time),
        "feature_selection_info": feature_selection_info  # NEW: Selection info per fold
    }

print("Nested CV function defined (v1.8 - Enhanced Feature Selection).")
print("  Supports: Stability Selection, Multi-Filter Ensemble, Elastic Net")


In [ ]:
# =============================================================================
# RUN NESTED CV (v1.8 - Enhanced Feature Selection)
# =============================================================================

print("="*70)
print("STARTING NESTED CROSS-VALIDATION WITH GRID SEARCH (v1.8)")
print("="*70)
print(f"\nConfiguration:")
print(f"  Outer CV: {OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats = {OUTER_SPLITS * OUTER_REPEATS} folds")
print(f"  Inner CV: {INNER_SPLITS}-fold for feature selection + grid search")
print(f"  Optimization metric: {OPTIMIZATION_METRIC.upper()}")
print(f"  Signature size: {MIN_SIGNATURE}-{MAX_SIGNATURE}")
print(f"  Top-k filter: {TOP_K_FILTER}")
print(f"\nFeature Selection Methods:")
print(f"  Stability Selection: {'ON' if USE_STABILITY_SELECTION else 'OFF'}")
if USE_STABILITY_SELECTION:
    print(f"    - Bootstrap iterations: {STABILITY_N_BOOTSTRAP}")
    print(f"    - Selection threshold: {STABILITY_THRESHOLD*100:.0f}%")
print(f"  Multi-Filter Ensemble: {'ON' if USE_MULTI_FILTER else 'OFF (ANOVA only)'}")
print(f"  Elastic Net: {'ON (' + ELASTIC_NET_MODE + ')' if USE_ELASTIC_NET else 'OFF'}")
if USE_ELASTIC_NET:
    print(f"    - L1 ratio: {ELASTIC_NET_L1_RATIO}")
    print(f"    - Top-k: {ELASTIC_NET_TOP_K}")
print()
_flush()

cv_results = nested_cv_with_gridsearch(
    df=df_train,
    label_col=schema["y"],
    protein_cols=schema["proteins"],
    min_sig=MIN_SIGNATURE,
    max_sig=MAX_SIGNATURE,
    top_k=TOP_K_FILTER,
    candidates_per_step=CANDIDATES_PER_STEP,
    inner_splits=INNER_SPLITS,
    outer_splits=OUTER_SPLITS,
    outer_repeats=OUTER_REPEATS,
    optimization_metric=OPTIMIZATION_METRIC,
    seed=SEED,
    eps_improve=EPS_IMPROVE,
    # NEW: Enhanced feature selection parameters
    use_stability=USE_STABILITY_SELECTION,
    use_multi_filter=USE_MULTI_FILTER,
    use_elastic_net=USE_ELASTIC_NET,
    elastic_net_mode=ELASTIC_NET_MODE,
    stability_n_bootstrap=STABILITY_N_BOOTSTRAP,
    stability_threshold=STABILITY_THRESHOLD,
    stability_subsample_ratio=STABILITY_SUBSAMPLE_RATIO,
    filter_weights=FILTER_WEIGHTS if USE_MULTI_FILTER else None,
    elastic_net_l1_ratio=ELASTIC_NET_L1_RATIO,
    elastic_net_n_alphas=ELASTIC_NET_N_ALPHAS,
    elastic_net_cv_folds=ELASTIC_NET_CV_FOLDS,
    elastic_net_top_k=ELASTIC_NET_TOP_K
)

print(f"\nCompleted {len(cv_results['fold_results'])} outer folds.")

# NEW: Report feature selection stability summary
if cv_results.get('feature_selection_info'):
    sel_info_list = cv_results['feature_selection_info']
    if sel_info_list and 'selection_frequencies' in sel_info_list[0]:
        # Aggregate selection frequencies across all folds
        all_freqs = {}
        for info in sel_info_list:
            for feat, freq in info.get('selection_frequencies', {}).items():
                if feat not in all_freqs:
                    all_freqs[feat] = []
                all_freqs[feat].append(freq)

        # Calculate mean frequency
        mean_freqs = {feat: np.mean(freqs) for feat, freqs in all_freqs.items()}
        top_stable = sorted(mean_freqs.items(), key=lambda x: -x[1])[:10]

        print(f"\nTop 10 Most Stable Features (avg selection frequency across folds):")
        for feat, freq in top_stable:
            print(f"  {feat}: {freq:.1%}")


In [ ]:
# =============================================================================
# CV RESULTS ANALYSIS AND STABILITY
# =============================================================================

fold_results = cv_results["fold_results"]
fold_metrics_df = cv_results["fold_metrics"]

# Build summary table
cv_summary_rows = []
for r in fold_results:
    cv_summary_rows.append({
        "Fold": r["fold"],
        "Algorithm": r["algo"],
        "N_Features": r["n_features"],
        "Features": ", ".join(r["features"]),
        "Inner_F1": r["inner_f1"],
        "Outer_F1": r["outer_f1"],
        "Outer_Accuracy": r["outer_accuracy"],
        "Outer_BalancedAcc": r["outer_balanced_accuracy"],
        "Outer_MCC": r["outer_mcc"],
        "Outer_AUC": r["outer_auc"]
    })

cv_summary_df = pd.DataFrame(cv_summary_rows)

# Aggregate statistics
print("\n" + "="*70)
print("CROSS-VALIDATION SUMMARY")
print("="*70)

metrics_summary = {
    "Metric": ["F1", "Accuracy", "Balanced Accuracy", "MCC", "AUC"],
    "Mean": [
        cv_summary_df["Outer_F1"].mean(),
        cv_summary_df["Outer_Accuracy"].mean(),
        cv_summary_df["Outer_BalancedAcc"].mean(),
        cv_summary_df["Outer_MCC"].mean(),
        cv_summary_df["Outer_AUC"].mean()
    ],
    "Std": [
        cv_summary_df["Outer_F1"].std(),
        cv_summary_df["Outer_Accuracy"].std(),
        cv_summary_df["Outer_BalancedAcc"].std(),
        cv_summary_df["Outer_MCC"].std(),
        cv_summary_df["Outer_AUC"].std()
    ],
    "Min": [
        cv_summary_df["Outer_F1"].min(),
        cv_summary_df["Outer_Accuracy"].min(),
        cv_summary_df["Outer_BalancedAcc"].min(),
        cv_summary_df["Outer_MCC"].min(),
        cv_summary_df["Outer_AUC"].min()
    ],
    "Max": [
        cv_summary_df["Outer_F1"].max(),
        cv_summary_df["Outer_Accuracy"].max(),
        cv_summary_df["Outer_BalancedAcc"].max(),
        cv_summary_df["Outer_MCC"].max(),
        cv_summary_df["Outer_AUC"].max()
    ]
}
metrics_summary_df = pd.DataFrame(metrics_summary)
print(metrics_summary_df.to_string(index=False))

# Algorithm frequency
algo_counts = cv_summary_df["Algorithm"].value_counts()
print(f"\nAlgorithm Win Counts:")
for algo, count in algo_counts.items():
    print(f"  {algo}: {count} ({count/len(cv_summary_df)*100:.1f}%)")

# Feature stability
all_features = []
for r in fold_results:
    all_features.extend(r["features"])
feat_counts = Counter(all_features)
stability_df = pd.DataFrame([
    {"Feature": f, "Selection_Count": c, "Selection_Rate": c/len(fold_results)}
    for f, c in feat_counts.most_common()
])

print(f"\nTop 10 Most Stable Features:")
print(stability_df.head(10).to_string(index=False))

In [ ]:
# =============================================================================
# CV STABILITY VISUALIZATION (Publication Grade)
# =============================================================================

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.25)

# 1. Metrics over folds (stability trend)
ax1 = fig.add_subplot(gs[0, 0])
folds = fold_metrics_df["fold"].values
ax1.plot(folds, fold_metrics_df["f1"], 'o-', color='#2ecc71', label='F1', linewidth=2, markersize=4)
ax1.plot(folds, fold_metrics_df["balanced_accuracy"], 's-', color='#3498db', label='Balanced Acc', linewidth=2, markersize=4)
ax1.plot(folds, fold_metrics_df["auc"], '^-', color='#e74c3c', label='AUC', linewidth=2, markersize=4)
ax1.axhline(fold_metrics_df["f1"].mean(), color='#2ecc71', linestyle='--', alpha=0.5)
# 95% CI = 1.96 * std / sqrt(n_folds)
ci_factor = 1.96 / np.sqrt(len(folds))
ax1.fill_between(folds, fold_metrics_df["f1"].mean() - ci_factor * fold_metrics_df["f1"].std(),
                 fold_metrics_df["f1"].mean() + ci_factor * fold_metrics_df["f1"].std(), alpha=0.2, color='#2ecc71')
ax1.set_xlabel('Outer Fold')
ax1.set_ylabel('Metric Value (shaded: 95% CI)')
ax1.set_title('A. Performance Stability Across Outer CV Folds')
ax1.legend(loc='lower right')
ax1.set_ylim([0, 1.05])
ax1.grid(alpha=0.3)

# 2. Algorithm distribution
ax2 = fig.add_subplot(gs[0, 1])
colors = plt.cm.Set2(np.linspace(0, 1, len(algo_counts)))
bars = ax2.bar(algo_counts.index, algo_counts.values, color=colors, edgecolor='black', linewidth=1.2)
ax2.set_xlabel('Algorithm')
ax2.set_ylabel('Win Count')
ax2.set_title('B. Champion Algorithm Distribution')
for bar, val in zip(bars, algo_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val),
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Boxplot of metrics by algorithm
ax3 = fig.add_subplot(gs[1, 0])
algo_f1_data = [cv_summary_df[cv_summary_df["Algorithm"]==a]["Outer_F1"].values for a in algo_counts.index]
bp = ax3.boxplot(algo_f1_data, labels=algo_counts.index, patch_artist=True, showmeans=True,
                 meanprops=dict(marker='D', markerfacecolor='red', markersize=6))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_xlabel('Algorithm')
ax3.set_ylabel('Outer Fold F1 Score')
ax3.set_title('C. F1 Score Distribution by Algorithm')
ax3.grid(axis='y', alpha=0.3)

# 4. Feature stability
ax4 = fig.add_subplot(gs[1, 1])
top_feats = stability_df.head(15)
colors_feat = plt.cm.viridis(top_feats["Selection_Rate"].values)
ax4.barh(range(len(top_feats)), top_feats["Selection_Count"].values, color=colors_feat)
ax4.set_yticks(range(len(top_feats)))
ax4.set_yticklabels(top_feats["Feature"].values)
ax4.invert_yaxis()
ax4.set_xlabel('Selection Count')
ax4.set_title('D. Top 15 Most Stable Features')

plt.suptitle('Cross-Validation Performance and Stability Analysis', fontsize=16, fontweight='bold', y=1.02)
savefig(OUTDIR, "cv_stability_analysis")
plt.show()

print(f"\nFigure saved: {OUTDIR}/cv_stability_analysis.png")

In [ ]:
# =============================================================================
# AGGREGATE UNIQUE CHAMPIONS FOR TEST EVALUATION
# =============================================================================

# Group by algorithm + features combination
combo_results = defaultdict(list)
for r in fold_results:
    key = (r["algo"], tuple(sorted(r["features"])))
    combo_results[key].append(r)

# Rank combinations by mean F1
champion_candidates = []
for (algo, feats), records in combo_results.items():
    champion_candidates.append({
        "algo": algo,
        "features": list(feats),
        "n_features": len(feats),
        "cv_frequency": len(records),
        "cv_mean_f1": np.mean([r["outer_f1"] for r in records]),
        "cv_std_f1": np.std([r["outer_f1"] for r in records]),
        "cv_mean_auc": np.mean([r["outer_auc"] for r in records]),
        "best_params": records[0]["best_params"]  # Use params from first occurrence
    })

# Sort by frequency then by mean F1
champion_candidates.sort(key=lambda x: (-x["cv_frequency"], -x["cv_mean_f1"]))

print("\n" + "="*70)
print("UNIQUE CHAMPION CANDIDATES FOR TEST EVALUATION")
print("="*70)
for i, c in enumerate(champion_candidates[:10], 1):
    print(f"\n{i}. {c['algo']} (k={c['n_features']})")
    print(f"   CV Frequency: {c['cv_frequency']}/{len(fold_results)}")
    print(f"   CV F1: {c['cv_mean_f1']:.3f} +/- {c['cv_std_f1']:.3f}")
    print(f"   Features: {c['features']}")

In [ ]:
# =============================================================================
# EXTERNAL TEST SET EVALUATION - ALL CHAMPIONS
# =============================================================================

print("\n" + "="*70)
print("EXTERNAL TEST SET EVALUATION")
print("="*70)

# Prepare data
y_train = _prepare_y(df_train[schema['y']], CLASS_LABELS)
y_test = _prepare_y(df_test[schema['y']], CLASS_LABELS)

X_train = df_train.copy()
X_test = df_test.copy()
for c in ["patient_ID", "patientID", "Patient", "ID"]:
    if c in X_train.columns: X_train = X_train.drop(columns=[c])
    if c in X_test.columns: X_test = X_test.drop(columns=[c])

print(f"Training: n={len(y_train)}")
print(f"Test: n={len(y_test)}")

# Build estimators with best params
def build_estimator_with_params(algo, params):
    configs = get_estimator_configs()
    est = clone(configs[algo]["estimator"])
    # Extract clf__ params
    clean_params = {k.replace("clf__", ""): v for k, v in params.items() if k.startswith("clf__")}
    est.set_params(**clean_params)
    return est

# Evaluate all champion candidates
test_results = []

for i, cand in enumerate(pbar(champion_candidates, desc="Evaluating on test"), 1):
    algo = cand["algo"]
    features = cand["features"]
    params = cand["best_params"]
    
    # Check features exist
    missing = [f for f in features if f not in X_test.columns]
    if missing:
        continue
    
    # Build pipeline
    estimator = build_estimator_with_params(algo, params)
    scaler = make_scaler(SCALER_KIND)
    if scaler:
        pipe = Pipeline([("scaler", scaler), ("clf", estimator)])
    else:
        pipe = Pipeline([("clf", estimator)])
    
    # Train on full training set
    pipe.fit(X_train[features].values, y_train)
    
    # Predict on test
    yhat = pipe.predict(X_test[features].values)
    try:
        yprob = pipe.predict_proba(X_test[features].values)[:, 1]
    except:
        yprob = None
    
    # Compute all metrics with CI
    metrics_ci = compute_all_metrics_with_ci(y_test, yhat, yprob, N_BOOTSTRAP, CI_LEVEL, SEED)
    
    result = {
        "rank": i,
        "algo": algo,
        "n_features": len(features),
        "features": features,
        "cv_frequency": cand["cv_frequency"],
        "cv_mean_f1": cand["cv_mean_f1"],
        "test_metrics_ci": metrics_ci,
        "test_f1": metrics_ci["F1_Score"][0],
        "test_accuracy": metrics_ci["Accuracy"][0],
        "test_bacc": metrics_ci["Balanced_Accuracy"][0],
        "test_auc": metrics_ci.get("ROC_AUC", (np.nan,))[0],
        "pipeline": pipe,
        "y_pred": yhat,
        "y_proba": yprob
    }
    test_results.append(result)

# Sort by test F1 to find final champion
test_results.sort(key=lambda x: -x["test_f1"])

print("\n" + "="*70)
print("TEST SET RESULTS (Ranked by F1)")
print("="*70)

for i, r in enumerate(test_results[:10], 1):
    f1_ci = r["test_metrics_ci"]["F1_Score"]
    marker = " [STAR] FINAL CHAMPION" if i == 1 else ""
    print(f"\n{i}. {r['algo']} (k={r['n_features']}){marker}")
    print(f"   CV: freq={r['cv_frequency']}, mean_F1={r['cv_mean_f1']:.3f}")
    print(f"   TEST F1: {format_metric_with_ci(*f1_ci)}")
    print(f"   Features: {r['features']}")

In [ ]:
# =============================================================================
# DECLARE FINAL CHAMPION - COMPOSITE SCORING SYSTEM
# =============================================================================
# Combines CV performance, Test performance, stability, and CV-Test agreement
# for robust model selection with small sample sizes.

import numpy as np

# =============================================================================
# COMPOSITE SCORING WEIGHTS (optimized for small samples)
# =============================================================================
SCORING_WEIGHTS = {
    'cv_f1': 0.30,           # CV F1 - more stable (45 evaluations)
    'test_f1': 0.40,         # Test F1 - ultimate validation (but noisy with n=16)
    'cv_frequency': 0.15,    # Stability - how often model was selected
    'cv_test_agreement': 0.15 # Agreement - penalize large CV-Test discrepancy
}

# Parsimony bonus (per feature reduction from max)
PARSIMONY_BONUS = 0.01

# Flag threshold for suspicious CV-Test discrepancy
DISCREPANCY_THRESHOLD = 0.15

print("\n" + "="*80)
print("COMPOSITE SCORING FOR CHAMPION SELECTION")
print("="*80)
print("\nScoring weights:")
for key, weight in SCORING_WEIGHTS.items():
    print(f"  {key}: {weight:.0%}")
print(f"  parsimony_bonus: {PARSIMONY_BONUS} per feature below max")

# =============================================================================
# CALCULATE COMPOSITE SCORES
# =============================================================================

def calculate_composite_score(model, max_features, max_frequency):
    """Calculate composite score for a model."""

    # Get metrics (with safe defaults)
    cv_f1 = model.get('cv_mean_f1', 0) or 0
    test_f1 = model.get('test_f1', 0) or 0
    cv_freq = model.get('cv_frequency', 0) or 0
    n_features = model.get('n_features', max_features)

    # Normalize CV frequency (0-1)
    cv_freq_norm = cv_freq / max_frequency if max_frequency > 0 else 0

    # CV-Test Agreement: 1 - |difference|, capped at 0
    cv_test_diff = abs(cv_f1 - test_f1)
    cv_test_agreement = max(0, 1 - cv_test_diff)

    # Parsimony bonus
    parsimony = (max_features - n_features) * PARSIMONY_BONUS

    # Composite score
    score = (
        SCORING_WEIGHTS['cv_f1'] * cv_f1 +
        SCORING_WEIGHTS['test_f1'] * test_f1 +
        SCORING_WEIGHTS['cv_frequency'] * cv_freq_norm +
        SCORING_WEIGHTS['cv_test_agreement'] * cv_test_agreement +
        parsimony
    )

    # Flag suspicious discrepancy
    flagged = cv_test_diff > DISCREPANCY_THRESHOLD

    return {
        'composite_score': score,
        'cv_f1_component': SCORING_WEIGHTS['cv_f1'] * cv_f1,
        'test_f1_component': SCORING_WEIGHTS['test_f1'] * test_f1,
        'cv_freq_component': SCORING_WEIGHTS['cv_frequency'] * cv_freq_norm,
        'agreement_component': SCORING_WEIGHTS['cv_test_agreement'] * cv_test_agreement,
        'parsimony_bonus': parsimony,
        'cv_test_discrepancy': cv_test_diff,
        'flagged_discrepancy': flagged
    }

# Get max values for normalization
max_features = max(r.get('n_features', 1) for r in test_results)
max_frequency = len(fold_results)

# Calculate scores for all models
for model in test_results:
    score_breakdown = calculate_composite_score(model, max_features, max_frequency)
    model['composite_score'] = score_breakdown['composite_score']
    model['score_breakdown'] = score_breakdown

# Sort by composite score (descending)
test_results_sorted = sorted(test_results, key=lambda x: -x['composite_score'])

# =============================================================================
# DISPLAY RANKING TABLE
# =============================================================================
print("\n" + "-"*100)
print("MODEL RANKING BY COMPOSITE SCORE")
print("-"*100)
print(f"{'Rank':<5} {'Algorithm':<8} {'k':<4} {'CV_F1':>8} {'Test_F1':>8} {'Freq':>6} {'Agree':>8} {'Parsim':>8} {'SCORE':>10} {'Flag':>6}")
print("-"*100)

for rank, model in enumerate(test_results_sorted, 1):
    algo = model['algo']
    n_feat = model['n_features']
    cv_f1 = model.get('cv_mean_f1', 0) or 0
    test_f1 = model.get('test_f1', 0) or 0
    freq = model.get('cv_frequency', 0)
    sb = model['score_breakdown']
    flag = "[!]" if sb['flagged_discrepancy'] else ""

    print(f"{rank:<5} {algo:<8} {n_feat:<4} {cv_f1:>8.4f} {test_f1:>8.4f} {freq:>4}/{max_frequency:<4} {sb['agreement_component']:>8.4f} {sb['parsimony_bonus']:>8.4f} {model['composite_score']:>10.4f} {flag:>6}")

print("-"*100)

# =============================================================================
# CHECK FOR TIES ON COMPOSITE SCORE
# =============================================================================
best_score = test_results_sorted[0]['composite_score']
# Consider scores within 0.01 as tied
SCORE_TIE_THRESHOLD = 0.01
tied_models = [r for r in test_results_sorted if abs(r['composite_score'] - best_score) < SCORE_TIE_THRESHOLD]

if len(tied_models) > 1:
    print("\n" + "="*80)
    print(f"[WARNING] {len(tied_models)} MODELS WITHIN {SCORE_TIE_THRESHOLD} OF TOP SCORE")
    print("="*80)

    # Detailed view of tied models
    for idx, model in enumerate(tied_models, 1):
        sb = model['score_breakdown']
        print(f"\n{'-'*80}")
        print(f"MODEL #{idx}: {model['algo']} (k={model['n_features']} features)")
        print(f"{'-'*80}")
        print(f"  Features: {', '.join(model['features'])}")

        params = model.get('best_params', {})
        if params:
            param_str = ', '.join([f"{k}={v}" for k, v in params.items()])
            print(f"  Hyperparameters: {param_str}")

        print(f"\n  Score Breakdown:")
        print(f"    CV F1 component:      {sb['cv_f1_component']:.4f} (CV F1 = {model.get('cv_mean_f1', 0):.4f})")
        print(f"    Test F1 component:    {sb['test_f1_component']:.4f} (Test F1 = {model.get('test_f1', 0):.4f})")
        print(f"    CV Frequency:         {sb['cv_freq_component']:.4f} ({model.get('cv_frequency', 0)}/{max_frequency} folds)")
        print(f"    CV-Test Agreement:    {sb['agreement_component']:.4f} (discrepancy = {sb['cv_test_discrepancy']:.4f})")
        print(f"    Parsimony bonus:      {sb['parsimony_bonus']:.4f}")
        print(f"    {'-'*40}")
        print(f"    TOTAL COMPOSITE:      {model['composite_score']:.4f}")

        if sb['flagged_discrepancy']:
            print(f"\n    [!] WARNING: CV-Test discrepancy ({sb['cv_test_discrepancy']:.2f}) exceeds threshold ({DISCREPANCY_THRESHOLD})")

    print(f"\n{'-'*80}")

    # User selection
    print("\n" + "="*80)
    print("USER SELECTION REQUIRED")
    print("="*80)

    while True:
        try:
            selection = input(f"\nEnter model number to select as FINAL CHAMPION (1-{len(tied_models)}): ").strip()
            selection_idx = int(selection) - 1
            if 0 <= selection_idx < len(tied_models):
                FINAL_CHAMPION = tied_models[selection_idx]
                print(f"\n[OK] Selected Model #{selection}: {FINAL_CHAMPION['algo']} (k={FINAL_CHAMPION['n_features']})")
                break
            else:
                print(f"Invalid selection. Please enter 1-{len(tied_models)}.")
        except ValueError:
            print("Invalid input. Please enter a number.")
        except EOFError:
            print("\n[Non-interactive] Selecting top-ranked model (#1)")
            FINAL_CHAMPION = tied_models[0]
            break
else:
    FINAL_CHAMPION = test_results_sorted[0]

# Add scoring metadata to champion
FINAL_CHAMPION['selection_method'] = 'composite_score'
FINAL_CHAMPION['scoring_weights'] = SCORING_WEIGHTS

# =============================================================================
# FLAG DISCREPANCY WARNING
# =============================================================================
if FINAL_CHAMPION['score_breakdown']['flagged_discrepancy']:
    print("\n" + "!"*80)
    print("[!] WARNING: SELECTED MODEL HAS SUSPICIOUS CV-TEST DISCREPANCY")
    print("!"*80)
    disc = FINAL_CHAMPION['score_breakdown']['cv_test_discrepancy']
    print(f"CV F1: {FINAL_CHAMPION.get('cv_mean_f1', 0):.4f}")
    print(f"Test F1: {FINAL_CHAMPION.get('test_f1', 0):.4f}")
    print(f"Discrepancy: {disc:.4f} (threshold: {DISCREPANCY_THRESHOLD})")
    print("Consider: possible overfitting to test set or unstable model")
    print("!"*80)

# =============================================================================
# DISPLAY FINAL CHAMPION
# =============================================================================
print("\n" + "="*70)
print("[TROPHY] FINAL CHAMPION MODEL [TROPHY]")
print("="*70)

sb = FINAL_CHAMPION['score_breakdown']
print(f"\nCOMPOSITE SCORE: {FINAL_CHAMPION['composite_score']:.4f}")
print(f"  - CV F1 ({SCORING_WEIGHTS['cv_f1']:.0%}):        {sb['cv_f1_component']:.4f}")
print(f"  - Test F1 ({SCORING_WEIGHTS['test_f1']:.0%}):      {sb['test_f1_component']:.4f}")
print(f"  - Stability ({SCORING_WEIGHTS['cv_frequency']:.0%}):   {sb['cv_freq_component']:.4f}")
print(f"  - Agreement ({SCORING_WEIGHTS['cv_test_agreement']:.0%}):   {sb['agreement_component']:.4f}")
print(f"  - Parsimony:         {sb['parsimony_bonus']:.4f}")

print(f"\nAlgorithm: {FINAL_CHAMPION['algo']}")
print(f"Features ({FINAL_CHAMPION['n_features']}): {FINAL_CHAMPION['features']}")

print(f"\nCross-Validation Performance:")
print(f"  CV Frequency: {FINAL_CHAMPION['cv_frequency']}/{len(fold_results)} folds")

cv_mean_f1 = FINAL_CHAMPION.get('cv_mean_f1')
cv_std_f1 = FINAL_CHAMPION.get('cv_std_f1')
if cv_mean_f1 is not None and cv_std_f1 is not None:
    print(f"  CV Mean F1: {cv_mean_f1:.3f} +/- {cv_std_f1:.3f}")
elif cv_mean_f1 is not None:
    print(f"  CV Mean F1: {cv_mean_f1:.3f}")

print(f"\nTest Set Performance:")
test_f1 = FINAL_CHAMPION.get('test_f1')
test_f1_ci = FINAL_CHAMPION.get('test_f1_ci')
if test_f1 is not None:
    if test_f1_ci is not None:
        print(f"  F1 Score: {test_f1:.3f} (95% CI: {test_f1_ci[0]:.3f}-{test_f1_ci[1]:.3f})")
    else:
        print(f"  F1 Score: {test_f1:.3f}")

metrics = ['accuracy', 'balanced_accuracy', 'auc', 'mcc', 'precision', 'recall', 'specificity', 'npv']
for m in metrics:
    val = FINAL_CHAMPION.get(f'test_{m}')
    if val is not None:
        print(f"  {m.replace('_', ' ').title()}: {val:.3f}")

if len(tied_models) > 1:
    print(f"\n  [USER SELECTED from {len(tied_models)} closely-scored models]")

print("="*70)

# =========================================================================
# DISPLAY FULL MODEL CONFIGURATION
# =========================================================================
print("\n" + "="*70)
print("[PKG] FULL MODEL CONFIGURATION (for reproduction)")
print("="*70)

champion_algo = FINAL_CHAMPION['algo']
champion_features = FINAL_CHAMPION['features']
champion_params = FINAL_CHAMPION.get('best_params', {})

print(f"\n1. ALGORITHM: {champion_algo}")

print(f"\n2. SELECTED FEATURES ({len(champion_features)}):")
for i, feat in enumerate(champion_features, 1):
    print(f"   {i}. {feat}")

print(f"\n3. BEST HYPERPARAMETERS (from GridSearchCV):")
if champion_params:
    for param, value in champion_params.items():
        print(f"   {param}: {value}")
else:
    print("   (No hyperparameters recorded)")

print(f"\n4. PREPROCESSING:")
print(f"   Scaler: {SCALER_KIND}")

print(f"\n5. PIPELINE STRUCTURE:")
print(f"   Step 1: {SCALER_KIND.title()}Scaler")
print(f"   Step 2: {champion_algo} Classifier")

print("\n" + "="*70)

In [ ]:
# =============================================================================
# CALIBRATE FINAL CHAMPION MODEL
# =============================================================================

if CALIBRATE_FINAL:
    print("\n" + "="*70)
    print("MODEL CALIBRATION")
    print("="*70)
    
    features = FINAL_CHAMPION["features"]
    base_pipe = FINAL_CHAMPION["pipeline"]
    
    # Calibrate using Platt scaling (sigmoid) with CV
    calibrated_clf = CalibratedClassifierCV(base_pipe, method='sigmoid', cv=3)
    calibrated_clf.fit(X_train[features].values, y_train)
    
    # Get calibrated predictions
    y_pred_cal = calibrated_clf.predict(X_test[features].values)
    y_proba_cal = calibrated_clf.predict_proba(X_test[features].values)[:, 1]
    
    # Compare calibration
    y_proba_uncal = FINAL_CHAMPION["y_proba"]
    
    print(f"\nCalibration comparison (Brier Score, lower is better):")
    if y_proba_uncal is not None:
        brier_uncal = brier_score_loss(y_test, y_proba_uncal)
        print(f"  Uncalibrated: {brier_uncal:.4f}")
    brier_cal = brier_score_loss(y_test, y_proba_cal)
    print(f"  Calibrated:   {brier_cal:.4f}")
    
    # Update champion with calibrated model
    FINAL_CHAMPION["calibrated_pipeline"] = calibrated_clf
    FINAL_CHAMPION["y_pred_calibrated"] = y_pred_cal
    FINAL_CHAMPION["y_proba_calibrated"] = y_proba_cal
    
    # Recalculate metrics with calibrated predictions
    cal_metrics = compute_all_metrics_with_ci(y_test, y_pred_cal, y_proba_cal, N_BOOTSTRAP, CI_LEVEL, SEED)
    FINAL_CHAMPION["calibrated_metrics_ci"] = cal_metrics
    
    print(f"\nCalibrated Test Metrics (95% CI):")
    for metric, values in cal_metrics.items():
        print(f"  {metric}: {format_metric_with_ci(*values)}")

In [ ]:
# =============================================================================
# SAVE FINAL CHAMPION MODEL (PICKLE SERIALIZATION)
# =============================================================================
# Saves the trained model for deployment without retraining

import pickle
from datetime import datetime

# Prepare model package for deployment
model_package = {
    'model': FINAL_CHAMPION.get('calibrated_pipeline', FINAL_CHAMPION['pipeline']),
    'features': FINAL_CHAMPION['features'],
    'algo': FINAL_CHAMPION['algo'],
    'n_features': FINAL_CHAMPION['n_features'],
    'calibrated': CALIBRATE_FINAL,
    'scaler': SCALER_KIND,
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'pipeline_version': '3.8.5',
    'class_labels': CLASS_LABELS,
    'target_column': TARGET_COLUMN,
    'test_metrics': {
        'f1': FINAL_CHAMPION.get('calibrated_metrics_ci', FINAL_CHAMPION.get('test_metrics_ci', {})).get('F1_Score', (None,))[0],
        'accuracy': FINAL_CHAMPION.get('calibrated_metrics_ci', FINAL_CHAMPION.get('test_metrics_ci', {})).get('Accuracy', (None,))[0],
        'roc_auc': FINAL_CHAMPION.get('calibrated_metrics_ci', FINAL_CHAMPION.get('test_metrics_ci', {})).get('ROC_AUC', (None,))[0],
    },
    'usage': """
    # To use this model:
    import pickle
    import pandas as pd
    
    # Load model package
    with open('champion_model.pkl', 'rb') as f:
        pkg = pickle.load(f)
    
    # Prepare new data (must have the same feature columns)
    X_new = df_new[pkg['features']].values
    
    # Predict
    predictions = pkg['model'].predict(X_new)
    probabilities = pkg['model'].predict_proba(X_new)[:, 1]
    """
}

# Save to pickle file
pickle_filename = f"champion_model_{FINAL_CHAMPION['algo']}_{len(FINAL_CHAMPION['features'])}features.pkl"
pickle_path = os.path.join(OUTDIR, pickle_filename)

with open(pickle_path, 'wb') as f:
    pickle.dump(model_package, f, protocol=pickle.HIGHEST_PROTOCOL)

print()
print("=" * 70)
print("MODEL SAVED (PICKLE SERIALIZATION)")
print("=" * 70)
print()
print(f"Saved to: {pickle_path}")
print()
print("Model package contents:")
print(f"  - Algorithm: {model_package['algo']}")
print(f"  - Features: {model_package['features']}")
print(f"  - Calibrated: {model_package['calibrated']}")
print(f"  - Test F1: {model_package['test_metrics']['f1']:.4f}" if model_package['test_metrics']['f1'] else "  - Test F1: N/A")
print(f"  - Test ROC-AUC: {model_package['test_metrics']['roc_auc']:.4f}" if model_package['test_metrics']['roc_auc'] else "  - Test ROC-AUC: N/A")
print()
print("To load and use:")
print(f"  with open('{pickle_filename}', 'rb') as f:")
print("      pkg = pickle.load(f)")
print(f"  predictions = pkg['model'].predict(X_new[pkg['features']])")

In [ ]:
# =============================================================================
# PUBLICATION-GRADE PERFORMANCE VISUALIZATION
# =============================================================================

# Use calibrated if available
if CALIBRATE_FINAL and "y_proba_calibrated" in FINAL_CHAMPION:
    y_pred_final = FINAL_CHAMPION["y_pred_calibrated"]
    y_proba_final = FINAL_CHAMPION["y_proba_calibrated"]
    metrics_final = FINAL_CHAMPION["calibrated_metrics_ci"]
    model_label = f"{FINAL_CHAMPION['algo']} (Calibrated)"
else:
    y_pred_final = FINAL_CHAMPION["y_pred"]
    y_proba_final = FINAL_CHAMPION["y_proba"]
    metrics_final = FINAL_CHAMPION["test_metrics_ci"]
    model_label = FINAL_CHAMPION["algo"]

# Also get train predictions for comparison
if CALIBRATE_FINAL and "calibrated_pipeline" in FINAL_CHAMPION:
    final_pipe = FINAL_CHAMPION["calibrated_pipeline"]
else:
    final_pipe = FINAL_CHAMPION["pipeline"]

features = FINAL_CHAMPION["features"]
y_pred_train = final_pipe.predict(X_train[features].values)
y_proba_train = final_pipe.predict_proba(X_train[features].values)[:, 1]

# Create comprehensive figure
fig = plt.figure(figsize=(20, 16))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Confusion Matrix - Train
ax1 = fig.add_subplot(gs[0, 0])
cm_train = confusion_matrix(y_train, y_pred_train, labels=[0, 1])
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['L (Neg)', 'S (Pos)'], yticklabels=['L (Neg)', 'S (Pos)'],
            annot_kws={'size': 16, 'weight': 'bold'}, cbar=False)
ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('Actual', fontsize=12)
ax1.set_title(f'A. Training Set Confusion Matrix\n(n={len(y_train)})', fontsize=12, fontweight='bold')

# 2. Confusion Matrix - Test
ax2 = fig.add_subplot(gs[0, 1])
cm_test = confusion_matrix(y_test, y_pred_final, labels=[0, 1])
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Oranges', ax=ax2,
            xticklabels=['L (Neg)', 'S (Pos)'], yticklabels=['L (Neg)', 'S (Pos)'],
            annot_kws={'size': 16, 'weight': 'bold'}, cbar=False)
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('Actual', fontsize=12)
ax2.set_title(f'B. Test Set Confusion Matrix\n(n={len(y_test)})', fontsize=12, fontweight='bold')

# 3. Normalized Confusion Matrix - Test
ax3 = fig.add_subplot(gs[0, 2])
cm_test_norm = cm_test.astype('float') / cm_test.sum(axis=1, keepdims=True)
sns.heatmap(cm_test_norm, annot=True, fmt='.2%', cmap='Oranges', ax=ax3,
            xticklabels=['L (Neg)', 'S (Pos)'], yticklabels=['L (Neg)', 'S (Pos)'],
            annot_kws={'size': 14}, cbar=False)
ax3.set_xlabel('Predicted', fontsize=12)
ax3.set_ylabel('Actual', fontsize=12)
ax3.set_title('C. Test Set (Normalized by Row)', fontsize=12, fontweight='bold')

# 4. ROC Curve with CI
ax4 = fig.add_subplot(gs[1, 0])
if y_proba_final is not None:
    # Bootstrap ROC curves for CI
    rng = np.random.RandomState(SEED)
    tprs = []
    aucs = []
    mean_fpr = np.linspace(0, 1, 100)
    
    for _ in pbar(range(500), desc="ROC Bootstrap", leave=False):
        idx = rng.choice(len(y_test), len(y_test), replace=True)
        if len(np.unique(y_test[idx])) < 2: continue
        fpr_b, tpr_b, _ = roc_curve(y_test[idx], y_proba_final[idx])
        tprs.append(np.interp(mean_fpr, fpr_b, tpr_b))
        aucs.append(roc_auc_score(y_test[idx], y_proba_final[idx]))
    
    tprs = np.array(tprs)
    mean_tpr = tprs.mean(axis=0)
    std_tpr = tprs.std(axis=0)
    tpr_upper = np.minimum(mean_tpr + 1.96*std_tpr, 1)
    tpr_lower = np.maximum(mean_tpr - 1.96*std_tpr, 0)
    
    fpr, tpr, _ = roc_curve(y_test, y_proba_final)
    auc_val = roc_auc_score(y_test, y_proba_final)
    auc_ci = metrics_final.get("ROC_AUC", (auc_val, np.nan, np.nan))
    
    ax4.plot(fpr, tpr, color='#e74c3c', lw=2.5, 
             label=f'AUC = {auc_ci[0]:.3f} [{auc_ci[1]:.3f}-{auc_ci[2]:.3f}]')
    ax4.fill_between(mean_fpr, tpr_lower, tpr_upper, color='#e74c3c', alpha=0.2)
    ax4.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.7, label='Random (AUC = 0.5)')
    ax4.set_xlim([-0.02, 1.02])
    ax4.set_ylim([-0.02, 1.02])
    ax4.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
    ax4.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
    ax4.set_title('D. ROC Curve with 95% CI', fontsize=12, fontweight='bold')
    ax4.legend(loc='lower right', fontsize=10)
    ax4.grid(alpha=0.3)

# 5. Precision-Recall Curve
ax5 = fig.add_subplot(gs[1, 1])
if y_proba_final is not None:
    precision, recall, _ = precision_recall_curve(y_test, y_proba_final)
    ap = average_precision_score(y_test, y_proba_final)
    ap_ci = metrics_final.get("Average_Precision", (ap, np.nan, np.nan))
    
    ax5.plot(recall, precision, color='#2ecc71', lw=2.5,
             label=f'AP = {ap_ci[0]:.3f} [{ap_ci[1]:.3f}-{ap_ci[2]:.3f}]')
    ax5.axhline(y_test.mean(), color='gray', linestyle='--', lw=1.5, 
                label=f'Baseline (prevalence = {y_test.mean():.2f})')
    ax5.set_xlim([-0.02, 1.02])
    ax5.set_ylim([-0.02, 1.02])
    ax5.set_xlabel('Recall (Sensitivity)', fontsize=12)
    ax5.set_ylabel('Precision (PPV)', fontsize=12)
    ax5.set_title('E. Precision-Recall Curve', fontsize=12, fontweight='bold')
    ax5.legend(loc='lower left', fontsize=10)
    ax5.grid(alpha=0.3)

# 6. Calibration Curve
ax6 = fig.add_subplot(gs[1, 2])
if y_proba_final is not None:
    prob_true, prob_pred = calibration_curve(y_test, y_proba_final, n_bins=5, strategy='uniform')
    ax6.plot(prob_pred, prob_true, 's-', color='#9b59b6', lw=2, markersize=8, label='Model')
    ax6.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfectly calibrated')
    ax6.set_xlim([-0.02, 1.02])
    ax6.set_ylim([-0.02, 1.02])
    ax6.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax6.set_ylabel('Fraction of Positives', fontsize=12)
    ax6.set_title('F. Calibration Curve', fontsize=12, fontweight='bold')
    ax6.legend(loc='lower right', fontsize=10)
    ax6.grid(alpha=0.3)

# 7. Metrics Summary Table
ax7 = fig.add_subplot(gs[2, 0:2])
ax7.axis('off')

# Create metrics table
table_data = []
for metric, values in metrics_final.items():
    table_data.append([metric.replace('_', ' '), format_metric_with_ci(*values)])

table = ax7.table(cellText=table_data, colLabels=['Metric', 'Value [95% CI]'],
                  loc='center', cellLoc='center',
                  colWidths=[0.4, 0.5])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Style header
for i in range(2):
    table[(0, i)].set_facecolor('#3498db')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax7.set_title('G. Test Set Performance Metrics with 95% Bootstrap CI', 
              fontsize=12, fontweight='bold', pad=20)

# 8. Probability Distribution
ax8 = fig.add_subplot(gs[2, 2])
if y_proba_final is not None:
    for cls, color, label in [(0, '#3498db', 'L (Negative)'), (1, '#e74c3c', 'S (Positive)')]:
        probs = y_proba_final[y_test == cls]
        ax8.hist(probs, bins=10, alpha=0.6, color=color, label=label, edgecolor='black')
    ax8.axvline(0.5, color='black', linestyle='--', lw=2, label='Threshold = 0.5')
    ax8.set_xlabel(f'Predicted Probability ({CLASS_LABELS[1]} class)', fontsize=12)
    ax8.set_ylabel('Count', fontsize=12)
    ax8.set_title('H. Prediction Distribution by Class', fontsize=12, fontweight='bold')
    ax8.legend(fontsize=9)

plt.suptitle(f'Final Champion: {model_label} (k={FINAL_CHAMPION["n_features"]} features)\n'
             f'Features: {", ".join(FINAL_CHAMPION["features"])}',
             fontsize=14, fontweight='bold', y=1.01)

savefig(OUTDIR, "champion_comprehensive_analysis")
plt.show()

print(f"\nFigure saved: {OUTDIR}/champion_comprehensive_analysis.png")

# =============================================================================
# EXPORT CURVE DATA TO EXCEL
# =============================================================================
print("\nExporting curve data to Excel...")

curve_data_xlsx = os.path.join(OUTDIR, "champion_curve_data.xlsx")
curve_sheets = {}

# 1. ROC Curve Data
roc_df = pd.DataFrame({
    'FPR': fpr,
    'TPR': tpr
})
curve_sheets['1_ROC_Curve'] = roc_df

# ROC with bootstrap CI (interpolated)
roc_ci_df = pd.DataFrame({
    'Mean_FPR': mean_fpr,
    'Mean_TPR': mean_tpr,
    'TPR_Lower_95CI': tpr_lower,
    'TPR_Upper_95CI': tpr_upper,
    'Std_TPR': std_tpr
})
curve_sheets['2_ROC_Bootstrap_CI'] = roc_ci_df

# ROC Summary
roc_summary = pd.DataFrame([{
    'AUC': auc_val,
    'AUC_CI_Lower': auc_ci[1] if len(auc_ci) > 1 else np.nan,
    'AUC_CI_Upper': auc_ci[2] if len(auc_ci) > 2 else np.nan,
    'N_Bootstrap': N_BOOTSTRAP
}])
curve_sheets['3_ROC_Summary'] = roc_summary

# 2. Precision-Recall Curve Data
pr_df = pd.DataFrame({
    'Recall': recall,
    'Precision': precision
})
curve_sheets['4_PR_Curve'] = pr_df

# PR Summary
pr_summary = pd.DataFrame([{
    'Average_Precision': ap,
    'AP_CI_Lower': ap_ci[1] if len(ap_ci) > 1 else np.nan,
    'AP_CI_Upper': ap_ci[2] if len(ap_ci) > 2 else np.nan
}])
curve_sheets['5_PR_Summary'] = pr_summary

# 3. Calibration Curve Data
cal_df = pd.DataFrame({
    'Mean_Predicted_Probability': prob_pred,
    'Fraction_of_Positives': prob_true
})
curve_sheets['6_Calibration_Curve'] = cal_df

# 4. Prediction Distribution Data
pred_class0 = y_proba_final[y_test == 0]
pred_class1 = y_proba_final[y_test == 1]
# Create histogram data
bins = np.linspace(0, 1, 21)
hist0, _ = np.histogram(pred_class0, bins=bins)
hist1, _ = np.histogram(pred_class1, bins=bins)
bin_centers = (bins[:-1] + bins[1:]) / 2

pred_dist_df = pd.DataFrame({
    'Bin_Center': bin_centers,
    'Bin_Lower': bins[:-1],
    'Bin_Upper': bins[1:],
    'Count_Class0': hist0,
    'Count_Class1': hist1
})
curve_sheets['7_Prediction_Distribution'] = pred_dist_df

# 5. Raw predictions for reference
raw_pred_df = pd.DataFrame({
    'Sample_Index': range(len(y_test)),
    'True_Label': y_test.values if hasattr(y_test, 'values') else y_test,
    'Predicted_Probability': y_proba_final,
    'Predicted_Label': y_pred_final
})
curve_sheets['8_Raw_Predictions'] = raw_pred_df

# Write to Excel
with pd.ExcelWriter(curve_data_xlsx, engine='openpyxl') as writer:
    for sheet_name, df in curve_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"[OK] Curve data saved to: {curve_data_xlsx}")


In [ ]:
# =============================================================================
# SAVE ALL RESULTS TO EXCEL
# =============================================================================

print("\n" + "="*70)
print("SAVING RESULTS TO EXCEL")
print("="*70)

# 1. CV Results Summary
cv_results_xlsx = os.path.join(OUTDIR, "cv_results.xlsx")
write_excel_safely(cv_results_xlsx, {
    "fold_results": cv_summary_df,
    "metrics_summary": metrics_summary_df,
    "feature_stability": stability_df,
    "fold_metrics_trend": fold_metrics_df
})

# 2. Test Results
test_results_rows = []
for r in test_results:
    # Get score breakdown if available
    sb = r.get("score_breakdown", {})
    row = {
        "Rank": r["rank"],
        "Algorithm": r["algo"],
        "N_Features": r["n_features"],
        "Features": ", ".join(r["features"]),
        "Composite_Score": r.get("composite_score", 0),
        "CV_Frequency": r["cv_frequency"],
        "CV_Mean_F1": r["cv_mean_f1"],
        "CV_Test_Discrepancy": sb.get("cv_test_discrepancy", 0),
        "Flagged_Discrepancy": sb.get("flagged_discrepancy", False)
    }
    for metric, values in r["test_metrics_ci"].items():
        row[f"Test_{metric}"] = values[0]
        row[f"Test_{metric}_CI_Lower"] = values[1]
        row[f"Test_{metric}_CI_Upper"] = values[2]
    test_results_rows.append(row)

test_results_df = pd.DataFrame(test_results_rows)
test_xlsx = os.path.join(OUTDIR, "test_results.xlsx")
write_excel_safely(test_xlsx, {"test_results": test_results_df})

# 3. Final Champion Summary with Full Model Configuration
champion_rows = []
metrics_to_save = FINAL_CHAMPION.get("calibrated_metrics_ci", FINAL_CHAMPION["test_metrics_ci"])
for metric, values in metrics_to_save.items():
    champion_rows.append({
        "Metric": metric,
        "Point_Estimate": values[0],
        "CI_Lower": values[1],
        "CI_Upper": values[2],
        "Formatted": format_metric_with_ci(*values)
    })

champion_df = pd.DataFrame(champion_rows)
champion_info = pd.DataFrame([{
    "Algorithm": FINAL_CHAMPION["algo"],
    "N_Features": FINAL_CHAMPION["n_features"],
    "Features": ", ".join(FINAL_CHAMPION["features"]),
    "CV_Frequency": FINAL_CHAMPION["cv_frequency"],
    "CV_Mean_F1": FINAL_CHAMPION["cv_mean_f1"],
    "Calibrated": CALIBRATE_FINAL
}])

# Create model configuration dataframe
model_config = FINAL_CHAMPION.get("model_config", {})
full_clf_params = model_config.get("full_clf_params", {})
best_params = model_config.get("best_params", {})

# Model configuration rows for Excel
config_rows = []

# Basic info
config_rows.append({"Category": "Model", "Parameter": "algorithm", "Value": str(FINAL_CHAMPION["algo"])})
config_rows.append({"Category": "Model", "Parameter": "classifier_class", "Value": str(model_config.get("classifier_class", ""))})
config_rows.append({"Category": "Model", "Parameter": "scaler", "Value": str(SCALER_KIND)})
config_rows.append({"Category": "Model", "Parameter": "n_features", "Value": str(FINAL_CHAMPION["n_features"])})
config_rows.append({"Category": "Model", "Parameter": "calibrated", "Value": str(CALIBRATE_FINAL)})

# Best params from GridSearchCV
for k, v in sorted(best_params.items()):
    config_rows.append({"Category": "GridSearchCV_Best", "Parameter": k, "Value": str(v)})

# Full classifier parameters
for k, v in sorted(full_clf_params.items()):
    if v is not None and not callable(v):
        config_rows.append({"Category": "Classifier_Full", "Parameter": k, "Value": str(v)})


# v1.8 Feature Selection Configuration
config_rows.append({"Category": "FeatureSelection", "Parameter": "stability_selection", "Value": str(USE_STABILITY_SELECTION)})
if USE_STABILITY_SELECTION:
    config_rows.append({"Category": "FeatureSelection", "Parameter": "stability_n_bootstrap", "Value": str(STABILITY_N_BOOTSTRAP)})
    config_rows.append({"Category": "FeatureSelection", "Parameter": "stability_threshold", "Value": str(STABILITY_THRESHOLD)})
    config_rows.append({"Category": "FeatureSelection", "Parameter": "stability_subsample_ratio", "Value": str(STABILITY_SUBSAMPLE_RATIO)})
config_rows.append({"Category": "FeatureSelection", "Parameter": "multi_filter", "Value": str(USE_MULTI_FILTER)})
if USE_MULTI_FILTER:
    config_rows.append({"Category": "FeatureSelection", "Parameter": "filter_weights", "Value": str(FILTER_WEIGHTS)})
config_rows.append({"Category": "FeatureSelection", "Parameter": "elastic_net", "Value": str(USE_ELASTIC_NET)})
if USE_ELASTIC_NET:
    config_rows.append({"Category": "FeatureSelection", "Parameter": "elastic_net_mode", "Value": str(ELASTIC_NET_MODE)})
    config_rows.append({"Category": "FeatureSelection", "Parameter": "elastic_net_l1_ratio", "Value": str(ELASTIC_NET_L1_RATIO)})
    config_rows.append({"Category": "FeatureSelection", "Parameter": "elastic_net_top_k", "Value": str(ELASTIC_NET_TOP_K)})
config_rows.append({"Category": "FeatureSelection", "Parameter": "top_k_filter", "Value": str(TOP_K_FILTER)})
config_rows.append({"Category": "FeatureSelection", "Parameter": "candidates_per_step", "Value": str(CANDIDATES_PER_STEP)})
config_rows.append({"Category": "FeatureSelection", "Parameter": "min_signature", "Value": str(MIN_SIGNATURE)})
config_rows.append({"Category": "FeatureSelection", "Parameter": "max_signature", "Value": str(MAX_SIGNATURE)})

config_df = pd.DataFrame(config_rows)

# Features dataframe
features_df = pd.DataFrame({
    "Feature_Index": range(1, len(FINAL_CHAMPION["features"]) + 1),
    "Feature_Name": FINAL_CHAMPION["features"]
})

champion_xlsx = os.path.join(OUTDIR, "final_champion.xlsx")
write_excel_safely(champion_xlsx, {
    "champion_info": champion_info,
    "test_metrics_ci": champion_df,
    "model_configuration": config_df,
    "selected_features": features_df
})

# 4. Predictions
predictions_df = pd.DataFrame({
    "True_Label": y_test,
    "Predicted_Label": y_pred_final,
    "Predicted_Probability": y_proba_final if y_proba_final is not None else np.nan
})
pred_xlsx = os.path.join(OUTDIR, "test_predictions.xlsx")
write_excel_safely(pred_xlsx, {"predictions": predictions_df})

print(f"\nFiles saved:")
print(f"  - {cv_results_xlsx}")
print(f"  - {test_xlsx}")
print(f"  - {champion_xlsx} (with model_configuration sheet)")
print(f"  - {pred_xlsx}")

In [ ]:
# =============================================================================
# FINAL SUMMARY JSON
# =============================================================================

# Get metrics for export
metrics_to_save = FINAL_CHAMPION.get("calibrated_metrics_ci", FINAL_CHAMPION["test_metrics_ci"])

final_summary = {
    "pipeline_info": {
        "train_file": TRAIN_XLSX_PATH,
        "test_file": TEST_XLSX_PATH,
        "target_column": TARGET_COLUMN,
        "class_labels": CLASS_LABELS,
        "positive_class": POSITIVE_CLASS,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "n_features_total": len(schema["proteins"]),
        "optimization_metric": OPTIMIZATION_METRIC,
        "outer_cv": f"{OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats",
        "inner_cv": f"{INNER_SPLITS}-fold",
        "scaler": SCALER_KIND,
        "bootstrap_n": N_BOOTSTRAP,
        "confidence_level": CI_LEVEL,
        "calibrated": CALIBRATE_FINAL,
        "seed": SEED,
        # v1.8 Feature Selection Configuration
        "feature_selection": {
            "stability_selection": USE_STABILITY_SELECTION,
            "stability_n_bootstrap": STABILITY_N_BOOTSTRAP if USE_STABILITY_SELECTION else None,
            "stability_threshold": STABILITY_THRESHOLD if USE_STABILITY_SELECTION else None,
            "stability_subsample_ratio": STABILITY_SUBSAMPLE_RATIO if USE_STABILITY_SELECTION else None,
            "multi_filter": USE_MULTI_FILTER,
            "filter_weights": FILTER_WEIGHTS if USE_MULTI_FILTER else None,
            "elastic_net": USE_ELASTIC_NET,
            "elastic_net_mode": ELASTIC_NET_MODE if USE_ELASTIC_NET else None,
            "elastic_net_l1_ratio": ELASTIC_NET_L1_RATIO if USE_ELASTIC_NET else None,
            "elastic_net_top_k": ELASTIC_NET_TOP_K if USE_ELASTIC_NET else None,
            "top_k_filter": TOP_K_FILTER,
            "candidates_per_step": CANDIDATES_PER_STEP,
            "min_signature": MIN_SIGNATURE,
            "max_signature": MAX_SIGNATURE
        }
    },
    "cv_summary": {
        "n_folds_completed": len(fold_results),
        "mean_f1": float(cv_summary_df["Outer_F1"].mean()),
        "std_f1": float(cv_summary_df["Outer_F1"].std()),
        "algorithm_distribution": dict(algo_counts)
    },
    "final_champion": {
        "algorithm": FINAL_CHAMPION["algo"],
        "features": FINAL_CHAMPION["features"],
        "n_features": FINAL_CHAMPION["n_features"],
        "cv_frequency": FINAL_CHAMPION["cv_frequency"],
        "cv_mean_f1": float(FINAL_CHAMPION["cv_mean_f1"]),
        "composite_score": float(FINAL_CHAMPION.get("composite_score", 0)),
        "selection_method": FINAL_CHAMPION.get("selection_method", "test_f1_only"),
        "score_breakdown": {
            "cv_f1_component": float(FINAL_CHAMPION.get("score_breakdown", {}).get("cv_f1_component", 0)),
            "test_f1_component": float(FINAL_CHAMPION.get("score_breakdown", {}).get("test_f1_component", 0)),
            "cv_freq_component": float(FINAL_CHAMPION.get("score_breakdown", {}).get("cv_freq_component", 0)),
            "agreement_component": float(FINAL_CHAMPION.get("score_breakdown", {}).get("agreement_component", 0)),
            "parsimony_bonus": float(FINAL_CHAMPION.get("score_breakdown", {}).get("parsimony_bonus", 0)),
            "cv_test_discrepancy": float(FINAL_CHAMPION.get("score_breakdown", {}).get("cv_test_discrepancy", 0)),
            "flagged_discrepancy": bool(FINAL_CHAMPION.get("score_breakdown", {}).get("flagged_discrepancy", False))
        },
        "scoring_weights": FINAL_CHAMPION.get("scoring_weights", {}),
        "test_metrics": {
            k: {"point": float(v[0]), "ci_lower": float(v[1]), "ci_upper": float(v[2])}
            for k, v in metrics_to_save.items()
        }
    }
}

# Helper function to convert numpy types to Python types
def convert_numpy(obj):
    if isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(item) for item in obj]
    else:
        return obj

with open(os.path.join(OUTDIR, "final_summary.json"), "w") as f:
    json.dump(convert_numpy(final_summary), f, indent=2)

print("\n" + "="*70)
print("[DONE] PIPELINE COMPLETE [DONE]")
print("="*70)
print(f"\nAll results saved to: {OUTDIR}/")
print("\nGenerated files:")
for f in sorted(os.listdir(OUTDIR)):
    size = os.path.getsize(os.path.join(OUTDIR, f)) / 1024
    print(f"  - {f} ({size:.1f} KB)")

In [ ]:
# =============================================================================
# DUMMY CLASSIFIER BASELINE
# =============================================================================
# Reference: scikit-learn best practices - compare against naive baselines

from sklearn.dummy import DummyClassifier

print("\n" + "="*70)
print("DUMMY CLASSIFIER BASELINE COMPARISON")
print("="*70)

dummy_results = {}

# Test different dummy strategies
strategies = {
    'stratified': 'Random (stratified by class)',
    'most_frequent': 'Always predict majority class',
    'uniform': 'Random (uniform)'
}

cv = RepeatedStratifiedKFold(n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=SEED)

for strategy, description in strategies.items():
    dummy = DummyClassifier(strategy=strategy, random_state=SEED)
    
    # CV scores on training data
    cv_scores = cross_val_score(dummy, X_train[schema['proteins']].values, y_train, 
                                cv=cv, scoring='f1')
    
    # Test set performance
    dummy.fit(X_train[schema['proteins']].values, y_train)
    y_pred_dummy = dummy.predict(X_test[schema['proteins']].values)
    test_f1 = f1_score(y_test, y_pred_dummy, zero_division=0)
    test_acc = accuracy_score(y_test, y_pred_dummy)
    
    dummy_results[strategy] = {
        'description': description,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'test_f1': test_f1,
        'test_accuracy': test_acc
    }
    
    print(f"\n{description} ({strategy}):")
    print(f"  CV F1: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
    print(f"  Test F1: {test_f1:.3f}")
    print(f"  Test Accuracy: {test_acc:.3f}")

# Compare to final champion
champion_test_f1 = FINAL_CHAMPION['test_metrics_ci']['F1_Score'][0]
best_dummy_f1 = max(d['test_f1'] for d in dummy_results.values())

print(f"\n" + "-"*50)
print(f"CHAMPION MODEL vs BASELINES:")
print(f"  Champion Test F1: {champion_test_f1:.3f}")
print(f"  Best Dummy Test F1: {best_dummy_f1:.3f}")
print(f"  Improvement over baseline: {(champion_test_f1 - best_dummy_f1)*100:.1f}%")

# Store for export
dummy_baseline_df = pd.DataFrame([
    {
        'Strategy': strategy,
        'Description': d['description'],
        'CV_F1_Mean': d['cv_f1_mean'],
        'CV_F1_Std': d['cv_f1_std'],
        'Test_F1': d['test_f1'],
        'Test_Accuracy': d['test_accuracy']
    }
    for strategy, d in dummy_results.items()
])

In [ ]:
# =============================================================================
# PERMUTATION TEST FOR STATISTICAL SIGNIFICANCE
# =============================================================================
# Reference: Ojala & Garriga, JMLR 2010 - "Permutation Tests for Studying Classifier Performance"
# Tests whether the model performance is significantly better than chance

from sklearn.model_selection import permutation_test_score

print("\n" + "="*70)
print("PERMUTATION TEST FOR STATISTICAL SIGNIFICANCE")
print("="*70)
print("\nTesting H0: There is no relationship between features and labels")
print("(i.e., the model's performance is no better than chance)")

# Get final champion model configuration
features = FINAL_CHAMPION["features"]
algo = FINAL_CHAMPION["algo"]

# Rebuild the pipeline
estimator = build_estimator_with_params(algo, FINAL_CHAMPION.get("best_params", {}))
scaler = make_scaler(SCALER_KIND)
if scaler:
    perm_pipe = Pipeline([("scaler", scaler), ("clf", estimator)])
else:
    perm_pipe = Pipeline([("clf", estimator)])

# Run permutation test (reduced permutations for speed, increase for publication)
N_PERMUTATIONS = 1000  # Increase to 1000+ for publication
print(f"\nRunning {N_PERMUTATIONS} permutations (this may take a few minutes)...")

cv_perm = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=SEED)

score, perm_scores, p_value = permutation_test_score(
    perm_pipe, 
    X_train[features].values, 
    y_train,
    cv=cv_perm,
    n_permutations=N_PERMUTATIONS,
    scoring='f1',
    random_state=SEED,
    n_jobs=-1
)

print(f"\n" + "-"*50)
print(f"PERMUTATION TEST RESULTS:")
print(f"  True CV F1 Score: {score:.3f}")
print(f"  Mean Permuted Score: {np.mean(perm_scores):.3f} +/- {np.std(perm_scores):.3f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significance (alpha=0.05): {'YES - Model is significantly better than chance' if p_value < 0.05 else 'NO - Cannot reject null hypothesis'}")
print(f"  Significance (alpha=0.01): {'YES' if p_value < 0.01 else 'NO'}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram of permuted scores
ax.hist(perm_scores, bins=30, density=True, alpha=0.7, color='#3498db', 
        edgecolor='black', label='Null Distribution (Permuted)')

# Mark the true score
ax.axvline(score, color='#e74c3c', linestyle='--', linewidth=3, 
           label=f'True Score = {score:.3f}')

# Add p-value annotation
ax.axvline(np.percentile(perm_scores, 95), color='gray', linestyle=':', 
           linewidth=2, label='95th percentile')

ax.set_xlabel('F1 Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Permutation Test: {algo} Model\np-value = {p_value:.4f} (n={N_PERMUTATIONS} permutations)', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10, framealpha=0.95)
ax.grid(alpha=0.3)

# Add text box with interpretation
textstr = f'H0: No relationship\np = {p_value:.4f}\n'
if p_value < 0.01:
    textstr += 'Highly significant (p < 0.01)'
elif p_value < 0.05:
    textstr += 'Significant (p < 0.05)'
else:
    textstr += 'Not significant (p >= 0.05)'
    
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

plt.tight_layout()
savefig(OUTDIR, "permutation_test")
plt.show()

# Store results
permutation_results = {
    'true_score': score,
    'mean_permuted': np.mean(perm_scores),
    'std_permuted': np.std(perm_scores),
    'p_value': p_value,
    'n_permutations': N_PERMUTATIONS,
    'significant_005': p_value < 0.05,
    'significant_001': p_value < 0.01
}

print(f"\nFigure saved: {OUTDIR}/permutation_test.png")

In [ ]:
# =============================================================================
# THRESHOLD OPTIMIZATION (YOUDEN'S J INDEX)
# =============================================================================
# Reference: Youden WJ. Index for rating diagnostic tests. Cancer. 1950
# Finds optimal threshold that maximizes Sensitivity + Specificity - 1

print("\n" + "="*70)
print("THRESHOLD OPTIMIZATION (YOUDEN'S J INDEX)")
print("="*70)

if y_proba_final is not None:
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba_final)
    
    # Calculate Youden's J statistic for each threshold
    youden_j = tpr - fpr  # Equivalent to Sensitivity + Specificity - 1
    
    # Find optimal threshold
    optimal_idx = np.argmax(youden_j)
    optimal_threshold = thresholds[optimal_idx]
    optimal_sensitivity = tpr[optimal_idx]
    optimal_specificity = 1 - fpr[optimal_idx]
    optimal_youden = youden_j[optimal_idx]
    
    print(f"\nOptimal Threshold Analysis:")
    print(f"  Default threshold: 0.500")
    print(f"  Optimal threshold (Youden's J): {optimal_threshold:.3f}")
    print(f"  Youden's J value: {optimal_youden:.3f}")
    print(f"  At optimal threshold:")
    print(f"    Sensitivity: {optimal_sensitivity:.3f}")
    print(f"    Specificity: {optimal_specificity:.3f}")
    
    # Apply optimal threshold
    y_pred_optimal = (y_proba_final >= optimal_threshold).astype(int)
    
    # Calculate metrics at optimal threshold
    print(f"\n" + "-"*50)
    print(f"PERFORMANCE COMPARISON:")
    print(f"\n{'Metric':<25} {'Default (0.5)':<15} {'Optimal ({:.3f})':<15}".format(optimal_threshold))
    print("-"*55)
    
    metrics_comparison = {}
    for metric_name, metric_fn in [
        ('Accuracy', accuracy_score),
        ('Balanced Accuracy', balanced_accuracy_score),
        ('F1 Score', lambda y, p: f1_score(y, p, zero_division=0)),
        ('Precision', lambda y, p: precision_score(y, p, zero_division=0)),
        ('Recall', lambda y, p: recall_score(y, p, zero_division=0)),
        ('MCC', matthews_corrcoef)
    ]:
        default_val = metric_fn(y_test, y_pred_final)
        optimal_val = metric_fn(y_test, y_pred_optimal)
        print(f"{metric_name:<25} {default_val:<15.3f} {optimal_val:<15.3f}")
        metrics_comparison[metric_name] = {'default': default_val, 'optimal': optimal_val}
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: ROC with optimal point
    ax1 = axes[0]
    ax1.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {roc_auc_score(y_test, y_proba_final):.3f})')
    ax1.plot(fpr[optimal_idx], tpr[optimal_idx], 'ro', markersize=12, 
             label=f'Optimal (J={optimal_youden:.3f})')
    ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax1.set_xlabel('False Positive Rate (1 - Specificity)')
    ax1.set_ylabel('True Positive Rate (Sensitivity)')
    ax1.set_title('ROC Curve with Optimal Threshold')
    ax1.legend(loc='lower right')
    ax1.grid(alpha=0.3)
    
    # Plot 2: Youden's J across thresholds
    ax2 = axes[1]
    valid_idx = (thresholds >= 0) & (thresholds <= 1)
    ax2.plot(thresholds[valid_idx], youden_j[valid_idx], 'g-', lw=2)
    ax2.axvline(optimal_threshold, color='r', linestyle='--', lw=2, 
                label=f'Optimal = {optimal_threshold:.3f}')
    ax2.axvline(0.5, color='gray', linestyle=':', lw=2, label='Default = 0.500')
    ax2.set_xlabel('Threshold')
    ax2.set_ylabel("Youden's J (Sensitivity + Specificity - 1)")
    ax2.set_title("Youden's J Index Across Thresholds")
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_xlim([0, 1])
    
    plt.tight_layout()
    savefig(OUTDIR, "threshold_optimization")
    plt.show()
    
    # Store results
    threshold_results = {
        'optimal_threshold': optimal_threshold,
        'youden_j': optimal_youden,
        'optimal_sensitivity': optimal_sensitivity,
        'optimal_specificity': optimal_specificity,
        'metrics_comparison': metrics_comparison
    }
    
    print(f"\nFigure saved: {OUTDIR}/threshold_optimization.png")
else:
    print("Skipped: No probability predictions available")
    threshold_results = None

In [ ]:
# =============================================================================
# DECISION CURVE ANALYSIS (DCA)
# =============================================================================
# Reference: Vickers AJ, Elkin EB. Decision curve analysis. Med Decis Making. 2006
# Reference: TRIPOD+AI recommends DCA for clinical utility assessment
# Compares model's net benefit against "treat all" and "treat none" strategies

print("\n" + "="*70)
print("DECISION CURVE ANALYSIS (Clinical Utility)")
print("="*70)

def calculate_net_benefit(y_true, y_prob, threshold):
    """
    Calculate net benefit at a given threshold probability.
    Net Benefit = (TP/n) - (FP/n) * (pt / (1-pt))
    where pt is the threshold probability
    """
    if threshold <= 0 or threshold >= 1:
        return np.nan
    
    y_pred = (y_prob >= threshold).astype(int)
    n = len(y_true)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    # Net benefit formula
    net_benefit = (tp / n) - (fp / n) * (threshold / (1 - threshold))
    return net_benefit

def calculate_treat_all_benefit(y_true, threshold):
    """
    Net benefit if we treat everyone (always predict positive).
    """
    if threshold <= 0 or threshold >= 1:
        return np.nan
    prevalence = np.mean(y_true)
    return prevalence - (1 - prevalence) * (threshold / (1 - threshold))

if y_proba_final is not None:
    # Calculate net benefit across threshold range
    thresholds_dca = np.linspace(0.01, 0.99, 99)
    
    net_benefits_model = []
    net_benefits_all = []
    net_benefits_none = []
    
    for pt in thresholds_dca:
        nb_model = calculate_net_benefit(y_test, y_proba_final, pt)
        nb_all = calculate_treat_all_benefit(y_test, pt)
        nb_none = 0  # Treat none always has net benefit of 0
        
        net_benefits_model.append(nb_model)
        net_benefits_all.append(nb_all)
        net_benefits_none.append(nb_none)
    
    # Create DCA DataFrame
    dca_df = pd.DataFrame({
        'threshold': thresholds_dca,
        'model': net_benefits_model,
        'treat_all': net_benefits_all,
        'treat_none': net_benefits_none
    })
    
    # Calculate area under the decision curve (useful summary)
    valid_mask = ~np.isnan(net_benefits_model)
    if np.sum(valid_mask) > 10:
        audc = np.trapz(
            np.maximum(np.array(net_benefits_model)[valid_mask], 0),
            thresholds_dca[valid_mask]
        )
    else:
        audc = np.nan
    
    # Publication-grade DCA plot
    fig, ax = plt.subplots(figsize=(10, 7))
    
    ax.plot(thresholds_dca, net_benefits_model, '-', color='#e74c3c', lw=2.5, 
            label=f'{FINAL_CHAMPION["algo"]} Model')
    ax.plot(thresholds_dca, net_benefits_all, '--', color='#3498db', lw=2, 
            label='Treat All')
    ax.plot(thresholds_dca, net_benefits_none, '-', color='#2c3e50', lw=2, 
            label='Treat None')
    
    ax.set_xlim([0, 1])
    ax.set_ylim([-0.1, max(0.5, max(net_benefits_model) * 1.1)])
    ax.set_xlabel('Threshold Probability', fontsize=12)
    ax.set_ylabel('Net Benefit', fontsize=12)
    ax.set_title('Decision Curve Analysis\nClinical Utility Assessment', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
    ax.grid(alpha=0.3)
    
    # Add interpretation zone
    ax.fill_between(thresholds_dca, 0, net_benefits_model, 
                    where=np.array(net_benefits_model) > np.maximum(net_benefits_all, 0),
                    alpha=0.2, color='#2ecc71', label='Model advantage')
    
    # Add text annotation
    textstr = f'INTERPRETATION:\n'
    textstr += f'Model provides net benefit over\n'
    textstr += f'"treat all" and "treat none"\n'
    textstr += f'strategies in threshold ranges\n'
    textstr += f'where the red line is highest.'
    props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
    ax.text(0.65, 0.95, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=props)
    
    plt.tight_layout()
    savefig(OUTDIR, "decision_curve_analysis")
    plt.show()
    
    # Summary statistics
    print("\nDCA Summary:")
    print(f"  Threshold range with positive model net benefit:")
    positive_nb = dca_df[dca_df['model'] > 0]['threshold']
    if len(positive_nb) > 0:
        print(f"    {positive_nb.min():.2f} - {positive_nb.max():.2f}")
    else:
        print(f"    None")
    
    # Find where model beats both alternatives
    model_advantage = dca_df[(dca_df['model'] > dca_df['treat_all']) & 
                             (dca_df['model'] > dca_df['treat_none']) &
                             (dca_df['model'] > 0)]
    if len(model_advantage) > 0:
        print(f"  Threshold range where model is optimal strategy:")
        print(f"    {model_advantage['threshold'].min():.2f} - {model_advantage['threshold'].max():.2f}")
    
    print(f"\nFigure saved: {OUTDIR}/decision_curve_analysis.png")
    
    # Store results
    dca_results = {
        'dca_data': dca_df,
        'optimal_range': (model_advantage['threshold'].min(), model_advantage['threshold'].max()) if len(model_advantage) > 0 else None
    }
else:
    print("Skipped: No probability predictions available")
    dca_results = None

In [ ]:
# =============================================================================
# SHAP ANALYSIS - SINGLE CELL VERSION
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed. Run: pip install shap")

if HAS_SHAP:
    print("\n" + "="*70)
    print("SHAP FEATURE IMPORTANCE ANALYSIS (Publication Quality)")
    print("="*70)
    
    # -------------------------------------------------------------------------
    # SETTINGS
    # -------------------------------------------------------------------------
    FIGURE_WIDTH_MM = 180  # 180=full page, 85=single column
    EXPORT_FORMATS = ['png', 'pdf']
    
    # Publication style
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
        'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
        'axes.linewidth': 1.0, 'axes.grid': False,
        'axes.spines.top': False, 'axes.spines.right': False,
        'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
        'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'ps.fonttype': 42,
    })
    
    SHAP_CMAP = LinearSegmentedColormap.from_list(
        'shap_div', ['#0571b0', '#92c5de', '#f7f7f7', '#f4a582', '#ca0020'])
    BAR_COLOR = '#1f77b4'
    
    # -------------------------------------------------------------------------
    # PREPARE DATA
    # -------------------------------------------------------------------------
    features = FINAL_CHAMPION["features"]
    final_model = FINAL_CHAMPION["pipeline"]
    algo = FINAL_CHAMPION["algo"]
    
    print(f"  Model: {algo}")
    print(f"  Features: {len(features)}")
    print(f"  Test samples: {len(X_test)}")
    
    X_train_feat = X_train[features].values
    X_test_feat = X_test[features].values
    
    # Scale data
    if 'scaler' in final_model.named_steps:
        scaler = final_model.named_steps['scaler']
        X_train_scaled = scaler.transform(X_train_feat)
        X_test_scaled = scaler.transform(X_test_feat)
        print(f"  Scaler: {type(scaler).__name__}")
    elif 'make_scaler' in dir() and 'SCALER_KIND' in dir():
        scaler = make_scaler(SCALER_KIND)
        if scaler:
            X_train_scaled = scaler.fit_transform(X_train_feat)
            X_test_scaled = scaler.transform(X_test_feat)
        else:
            X_train_scaled, X_test_scaled = X_train_feat, X_test_feat
    else:
        X_train_scaled, X_test_scaled = X_train_feat, X_test_feat
    
    clf = final_model.named_steps['clf']
    
    # -------------------------------------------------------------------------
    # COMPUTE SHAP VALUES
    # -------------------------------------------------------------------------
    print("\nComputing SHAP values...")
    
    try:
        if algo in ['RF', 'DT', 'XGB', 'XGBOOST', 'GBM', 'LGBM']:
            explainer = shap.TreeExplainer(clf)
            shap_values = explainer.shap_values(X_test_scaled)
            if isinstance(shap_values, list):
                shap_values = shap_values[1]
        else:
            # SVM, LR, etc. - use KernelExplainer
            background = shap.sample(X_train_scaled, min(50, len(X_train_scaled)))
            
            def predict_pos(X):
                return clf.predict_proba(X)[:, 1]
            
            explainer = shap.KernelExplainer(predict_pos, background)
            shap_values = explainer.shap_values(X_test_scaled)
            shap_values = np.array(shap_values)
            if shap_values.ndim == 3:
                shap_values = shap_values[1]
            elif shap_values.ndim == 1:
                shap_values = shap_values.reshape(1, -1)
        
        print(f"  SHAP values shape: {shap_values.shape}")
        
        # Calculate importance
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        
        shap_importance_df = pd.DataFrame({
            'Feature': features,
            'Mean_Abs_SHAP': mean_abs_shap,
            'Std_SHAP': np.std(shap_values, axis=0)
        }).sort_values('Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
        shap_importance_df['Rank'] = range(1, len(shap_importance_df) + 1)
        
        print("\n" + "-"*50)
        print("SHAP Feature Importance Ranking:")
        print("-"*50)
        print(shap_importance_df[['Rank', 'Feature', 'Mean_Abs_SHAP']].to_string(index=False))
        
        # ---------------------------------------------------------------------
        # FIGURE: COMBINED PANEL
        # ---------------------------------------------------------------------
        print("\nGenerating publication figure...")
        
        fig_width = FIGURE_WIDTH_MM / 25.4
        n_features = len(features)
        fig_height = max(3, n_features * 0.4 + 1.2)
        
        sorted_idx = np.argsort(mean_abs_shap)
        sorted_features = [features[i] for i in sorted_idx]
        sorted_importance = mean_abs_shap[sorted_idx]
        importance_order = np.argsort(mean_abs_shap)[::-1]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(fig_width, fig_height + 0.3),
                                        gridspec_kw={'width_ratios': [1, 1.35], 'wspace': 0.35})
        
        # Panel A: Bar plot
        y_pos = np.arange(n_features)
        bars = ax1.barh(y_pos, sorted_importance, height=0.6, color=BAR_COLOR, alpha=0.85)
        ax1.set_yticks(y_pos)
        ax1.set_yticklabels(sorted_features)
        ax1.set_xlabel('Mean |SHAP value|')
        ax1.set_title('A', fontweight='bold', loc='left', fontsize=14)
        max_val = sorted_importance.max()
        ax1.set_xlim(0, max_val * 1.1)
        ax1.spines['left'].set_visible(False)
        ax1.tick_params(axis='y', length=0)
        
        # Panel B: Beeswarm
        np.random.seed(42)
        for i, feat_idx in enumerate(importance_order):
            feat_shap = shap_values[:, feat_idx]
            feat_vals = X_test_scaled[:, feat_idx]
            fmin, fmax = feat_vals.min(), feat_vals.max()
            feat_norm = (feat_vals - fmin) / (fmax - fmin + 1e-10)
            y_jit = np.random.uniform(-0.25, 0.25, len(feat_shap))
            scatter = ax2.scatter(feat_shap, 
                                  np.full_like(feat_shap, n_features - 1 - i) + y_jit,
                                  c=feat_norm, cmap=SHAP_CMAP, s=22, alpha=0.75,
                                  edgecolors='none', vmin=0, vmax=1)
        
        ax2.set_yticks(range(n_features))
        ax2.set_yticklabels([features[i] for i in importance_order[::-1]])
        ax2.set_xlabel('SHAP value')
        ax2.set_title('B', fontweight='bold', loc='left', fontsize=14)
        ax2.axvline(x=0, color='#666666', linestyle='-', linewidth=0.8, alpha=0.6)
        ax2.spines['left'].set_visible(False)
        ax2.tick_params(axis='y', length=0)
        
        cbar = plt.colorbar(scatter, ax=ax2, shrink=0.5, aspect=25, pad=0.02)
        cbar.set_label('Feature value', fontsize=9)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(['Low', 'High'])
        cbar.outline.set_visible(False)
        
        fig.suptitle(f'SHAP Feature Importance - {algo} Model', 
                     fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        
        # Save
        os.makedirs(OUTDIR, exist_ok=True)
        for fmt in EXPORT_FORMATS:
            fig.savefig(f"{OUTDIR}/shap_combined_panel.{fmt}", format=fmt,
                        dpi=300 if fmt == 'png' else None,
                        bbox_inches='tight', facecolor='white')
        print(f"  [OK] Saved: {OUTDIR}/shap_combined_panel.[{', '.join(EXPORT_FORMATS)}]")
        
        plt.show()
        
        # Save CSV
        shap_importance_df.to_csv(f"{OUTDIR}/shap_importance_values.csv", index=False)
        print(f"  [OK] Saved: {OUTDIR}/shap_importance_values.csv")
        
        print("\n" + "="*70)
        print("SHAP ANALYSIS COMPLETE")
        print("="*70)
        
        # Store results
        shap_results = {
            'shap_values': shap_values,
            'importance_df': shap_importance_df,
            'features': features
        }
        
    except Exception as e:
        print(f"\n[X] SHAP analysis failed: {e}")
        import traceback
        traceback.print_exc()
        shap_results = None
else:
    print("\nTo enable SHAP analysis, install: pip install shap")
    shap_results = None

In [ ]:
# =============================================================================
# FEATURE CORRELATION ANALYSIS
# =============================================================================
# Analyzes correlation between selected features to identify redundancy

print("\n" + "="*70)
print("FEATURE CORRELATION ANALYSIS")
print("="*70)

features = FINAL_CHAMPION["features"]

# Calculate correlation matrix for selected features
X_feat = X_train[features]
corr_matrix = X_feat.corr(method='pearson')

# Find highly correlated pairs
high_corr_threshold = 0.8
high_corr_pairs = []

for i in range(len(features)):
    for j in range(i+1, len(features)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) >= high_corr_threshold:
            high_corr_pairs.append({
                'Feature_1': features[i],
                'Feature_2': features[j],
                'Correlation': corr_val
            })

print(f"\nCorrelation Matrix (Pearson):")
print(corr_matrix.round(3).to_string())

if high_corr_pairs:
    print(f"\n[!]  Highly Correlated Pairs (|r| >= {high_corr_threshold}):")
    for pair in high_corr_pairs:
        print(f"  {pair['Feature_1']} <-> {pair['Feature_2']}: r = {pair['Correlation']:.3f}")
    print("\n  Note: High correlation may indicate redundant features")
else:
    print(f"\nOK No highly correlated pairs found (|r| >= {high_corr_threshold})")
    print("  Feature set shows good diversity")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Correlation heatmap
ax1 = axes[0]
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(250, 10, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, annot=True, fmt='.2f', ax=ax1,
            annot_kws={'size': 10})
ax1.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

# Plot 2: Pairwise scatter plot (if few features)
ax2 = axes[1]
if len(features) <= 5:
    # Create a simple scatter matrix representation
    from itertools import combinations
    pairs = list(combinations(range(len(features)), 2))
    if len(pairs) > 0:
        n_pairs = len(pairs)
        colors = ['#3498db' if y == 0 else '#e74c3c' for y in y_train]
        
        # Just show first pair as example
        i, j = pairs[0]
        ax2.scatter(X_feat.iloc[:, i], X_feat.iloc[:, j], c=colors, alpha=0.6, s=50)
        ax2.set_xlabel(features[i])
        ax2.set_ylabel(features[j])
        ax2.set_title(f'Example: {features[i]} vs {features[j]}\nr = {corr_matrix.iloc[i,j]:.3f}', 
                      fontsize=12, fontweight='bold')
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='#3498db', label='L (Negative)'),
                         Patch(facecolor='#e74c3c', label='S (Positive)')]
        ax2.legend(handles=legend_elements, loc='upper right')
else:
    # Show correlation distribution
    corr_values = corr_matrix.values[np.triu_indices(len(features), k=1)]
    ax2.hist(corr_values, bins=20, color='#3498db', edgecolor='black', alpha=0.7)
    ax2.axvline(high_corr_threshold, color='red', linestyle='--', lw=2, label=f'Threshold = {high_corr_threshold}')
    ax2.axvline(-high_corr_threshold, color='red', linestyle='--', lw=2)
    ax2.set_xlabel('Correlation Coefficient')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Distribution of Pairwise Correlations', fontsize=12, fontweight='bold')
    ax2.legend()

plt.tight_layout()
savefig(OUTDIR, "feature_correlation")
plt.show()

# Store results
correlation_results = {
    'correlation_matrix': corr_matrix,
    'high_corr_pairs': high_corr_pairs
}

print(f"\nFigure saved: {OUTDIR}/feature_correlation.png")

In [ ]:
# =============================================================================
# LEARNING CURVE ANALYSIS
# =============================================================================
# Diagnoses bias-variance tradeoff and assesses sample size adequacy

from sklearn.model_selection import learning_curve

print("\n" + "="*70)
print("LEARNING CURVE ANALYSIS")
print("="*70)
print("\nDiagnosing bias-variance tradeoff and sample size adequacy...")

features = FINAL_CHAMPION["features"]
algo = FINAL_CHAMPION["algo"]

# Rebuild the pipeline
estimator = build_estimator_with_params(algo, FINAL_CHAMPION.get("best_params", {}))
scaler = make_scaler(SCALER_KIND)
if scaler:
    lc_pipe = Pipeline([("scaler", scaler), ("clf", estimator)])
else:
    lc_pipe = Pipeline([("clf", estimator)])

# Calculate learning curves
train_sizes_rel = np.linspace(0.2, 1.0, 8)  # 20% to 100% of training data
cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

train_sizes_abs, train_scores, val_scores = learning_curve(
    lc_pipe,
    X_train[features].values,
    y_train,
    train_sizes=train_sizes_rel,
    cv=cv_lc,
    scoring='f1',
    n_jobs=-1,
    random_state=SEED
)

# Calculate statistics
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

# Interpretation
final_gap = train_mean[-1] - val_mean[-1]
val_trend = val_mean[-1] - val_mean[0]

print(f"\nLearning Curve Summary:")
print(f"  Training samples: {train_sizes_abs[0]} to {train_sizes_abs[-1]}")
print(f"  Final training F1: {train_mean[-1]:.3f} +/- {train_std[-1]:.3f}")
print(f"  Final validation F1: {val_mean[-1]:.3f} +/- {val_std[-1]:.3f}")
print(f"  Train-Val gap: {final_gap:.3f}")
print(f"  Validation trend: {'+' if val_trend > 0 else ''}{val_trend:.3f}")

# Diagnosis
print(f"\nDiagnosis:")
if final_gap > 0.2:
    print("  [!]  HIGH VARIANCE (Overfitting): Large gap between train and validation")
    print("      -> Consider: More training data, regularization, simpler model")
elif val_mean[-1] < 0.6:
    print("  [!]  HIGH BIAS (Underfitting): Low validation performance overall")
    print("      -> Consider: More features, more complex model, less regularization")
elif val_trend > 0.05:
    print("  [CHART] POSITIVE TREND: Validation still improving with more data")
    print("      -> More training samples would likely improve performance")
else:
    print("  OK  GOOD FIT: Reasonable balance between bias and variance")
    print("      -> Model is appropriately complex for the data size")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Learning curves
ax1 = axes[0]
ax1.fill_between(train_sizes_abs, train_mean - 1.96*train_std/np.sqrt(5), train_mean + 1.96*train_std/np.sqrt(5), alpha=0.2, color='#3498db')  # 95% CI
ax1.fill_between(train_sizes_abs, val_mean - 1.96*val_std/np.sqrt(5), val_mean + 1.96*val_std/np.sqrt(5), alpha=0.2, color='#e74c3c')  # 95% CI
ax1.plot(train_sizes_abs, train_mean, 'o-', color='#3498db', lw=2, label='Training score')
ax1.plot(train_sizes_abs, val_mean, 'o-', color='#e74c3c', lw=2, label='Validation score')
ax1.set_xlabel('Number of Training Samples', fontsize=12)
ax1.set_ylabel('F1 Score (shaded: 95% CI)', fontsize=12)
ax1.set_title(f'Learning Curves - {algo} Model', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10, title='Shaded: 95% CI')
ax1.grid(alpha=0.3)
ax1.set_ylim([0, 1.05])

# Plot 2: Gap analysis
ax2 = axes[1]
gap = train_mean - val_mean
ax2.bar(range(len(train_sizes_abs)), gap, color='#9b59b6', alpha=0.7, edgecolor='black')
ax2.axhline(0.2, color='red', linestyle='--', lw=2, label='High variance threshold')
ax2.set_xticks(range(len(train_sizes_abs)))
ax2.set_xticklabels([str(int(s)) for s in train_sizes_abs], rotation=45)
ax2.set_xlabel('Number of Training Samples', fontsize=12)
ax2.set_ylabel('Train-Validation Gap', fontsize=12)
ax2.set_title('Bias-Variance Gap', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
savefig(OUTDIR, "learning_curves")
plt.show()

# Store results
learning_curve_results = {
    'train_sizes': train_sizes_abs.tolist(),
    'train_mean': train_mean.tolist(),
    'train_std': train_std.tolist(),
    'val_mean': val_mean.tolist(),
    'val_std': val_std.tolist(),
    'final_gap': final_gap,
    'val_trend': val_trend
}

print(f"\nFigure saved: {OUTDIR}/learning_curves.png")

In [ ]:
# Helper function to convert numpy types to Python types
def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_numpy(item) for item in obj)
    else:
        return obj

# =============================================================================
# EXPORT ALL ENHANCED ANALYSIS RESULTS
# =============================================================================
print("\n" + "="*70)
print("EXPORTING ENHANCED ANALYSIS RESULTS")
print("="*70)

# 1. Export dummy baseline results
enhanced_xlsx = os.path.join(OUTDIR, "enhanced_analysis.xlsx")
sheets_to_save = {
    "dummy_baseline": dummy_baseline_df
}

# 2. Add permutation test results
if 'permutation_results' in dir():
    perm_df = pd.DataFrame([permutation_results])
    sheets_to_save["permutation_test"] = perm_df

# 3. Add threshold optimization results
if threshold_results is not None:
    threshold_df = pd.DataFrame([{
        'Optimal_Threshold': threshold_results['optimal_threshold'],
        'Youden_J': threshold_results['youden_j'],
        'Sensitivity_at_Optimal': threshold_results['optimal_sensitivity'],
        'Specificity_at_Optimal': threshold_results['optimal_specificity']
    }])
    sheets_to_save["threshold_optimization"] = threshold_df

# 4. Add DCA results
if dca_results is not None:
    sheets_to_save["decision_curve"] = dca_results['dca_data']

# 5. Add SHAP results
if 'shap_results' in dir() and shap_results is not None:
    sheets_to_save["shap_importance"] = shap_results['importance_df']

# 6. Add correlation results
if 'correlation_results' in dir():
    sheets_to_save["feature_correlation"] = correlation_results['correlation_matrix']
    if correlation_results['high_corr_pairs']:
        sheets_to_save["high_corr_pairs"] = pd.DataFrame(correlation_results['high_corr_pairs'])

# 7. Add learning curve results
if 'learning_curve_results' in dir():
    lc_df = pd.DataFrame({
        'Train_Size': learning_curve_results['train_sizes'],
        'Train_Mean_F1': learning_curve_results['train_mean'],
        'Train_Std_F1': learning_curve_results['train_std'],
        'Val_Mean_F1': learning_curve_results['val_mean'],
        'Val_Std_F1': learning_curve_results['val_std']
    })
    sheets_to_save["learning_curves"] = lc_df

# 8. Add stability selection info (v1.8)
if 'cv_results' in dir() and cv_results.get('feature_selection_info'):
    sel_info_list = cv_results['feature_selection_info']
    if sel_info_list and sel_info_list[0].get('selection_frequencies'):
        # Aggregate selection frequencies across all folds
        all_freqs = {}
        for info in sel_info_list:
            for feat, freq in info.get('selection_frequencies', {}).items():
                if feat not in all_freqs:
                    all_freqs[feat] = []
                all_freqs[feat].append(freq)
        
        # Calculate mean and std frequency
        stability_rows = []
        for feat, freqs in sorted(all_freqs.items(), key=lambda x: -np.mean(x[1])):
            stability_rows.append({
                'Feature': feat,
                'Mean_Selection_Freq': np.mean(freqs),
                'Std_Selection_Freq': np.std(freqs),
                'Min_Freq': np.min(freqs),
                'Max_Freq': np.max(freqs),
                'N_Folds': len(freqs)
            })
        
        if stability_rows:
            sheets_to_save["stability_selection"] = pd.DataFrame(stability_rows)



# Save all to Excel
write_excel_safely(enhanced_xlsx, sheets_to_save)
print(f"\n[Saved] {enhanced_xlsx}")

# Update final summary JSON
enhanced_summary = {
    "statistical_significance": {
        "permutation_test_p_value": permutation_results.get('p_value', None) if 'permutation_results' in dir() else None,
        "significant_at_005": permutation_results.get('significant_005', None) if 'permutation_results' in dir() else None
    },
    "baseline_comparison": {
        "best_dummy_f1": max(d['test_f1'] for d in dummy_results.values()) if dummy_results else None,
        "champion_vs_dummy_improvement": (champion_test_f1 - best_dummy_f1) if 'best_dummy_f1' in dir() else None
    },
    "threshold_optimization": {
        "optimal_threshold": threshold_results['optimal_threshold'] if threshold_results else None,
        "youden_j": threshold_results['youden_j'] if threshold_results else None
    },
    "learning_curve_diagnosis": {
        "final_train_val_gap": learning_curve_results.get('final_gap', None) if 'learning_curve_results' in dir() else None,
        "val_trend": learning_curve_results.get('val_trend', None) if 'learning_curve_results' in dir() else None
    }
}

# Update main summary
# Update final_summary if it exists (created in earlier JSON export cell)
if 'final_summary' in dir():
    final_summary["enhanced_analysis"] = enhanced_summary

# Convert all numpy types to native Python types before saving
final_summary_clean = convert_numpy(final_summary)

with open(os.path.join(OUTDIR, "final_summary.json"), "w") as f:
    json.dump(final_summary_clean, f, indent=2)
    
print(f"[Updated] {OUTDIR}/final_summary.json")

In [ ]:
# =============================================================================
# BASELINE COMPARISON USING NESTED CV RESULTS
# =============================================================================

total_folds = OUTER_SPLITS * OUTER_REPEATS

print("\n" + "="*70)
print("BASELINE COMPARISON: Champion vs ZeroR")
print(f"Using {total_folds} Nested CV Outer Folds ({OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats)")
print("="*70)

# Get champion details
algo = FINAL_CHAMPION['algo']
champion_features = FINAL_CHAMPION['features']

print(f"\nChampion: {algo}")
print(f"Features: {champion_features}")

# =============================================================================
# RE-RUN NESTED CV FOR CHAMPION MODEL TO GET PROPER FOLD METRICS
# =============================================================================

from sklearn.dummy import DummyClassifier
from scipy import stats as scipy_stats

# Define the champion model
if algo == 'kNN':
    from sklearn.neighbors import KNeighborsClassifier
    base_model = KNeighborsClassifier()
    param_grid = {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}
elif algo == 'SVM':
    from sklearn.svm import SVC
    base_model = SVC(probability=True, random_state=SEED)
    param_grid = {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']}
elif algo == 'RF':
    from sklearn.ensemble import RandomForestClassifier
    base_model = RandomForestClassifier(random_state=SEED)
    param_grid = {'n_estimators': [50, 100], 'max_depth': [3, 5, None]}
elif algo == 'LR':
    from sklearn.linear_model import LogisticRegression
    base_model = LogisticRegression(random_state=SEED, max_iter=1000)
    param_grid = {'C': [0.1, 1, 10]}
elif algo == 'NB':
    from sklearn.naive_bayes import GaussianNB
    base_model = GaussianNB()
    param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7]}
else:
    raise ValueError(f"Unknown algorithm: {algo}")

# Prepare data with champion features
X_champion = X_train[champion_features]

# Storage for fold metrics
champion_fold_metrics = {
    'Accuracy': [],
    'Balanced_Accuracy': [],
    'Precision': [],
    'Recall': [],
    'Specificity': [],
    'F1': [],
    'MCC': [],
    'ROC_AUC': [],
    'PRC_AUC': []
}

zeror_fold_metrics = {
    'Accuracy': [],
    'Balanced_Accuracy': [],
    'Precision': [],
    'Recall': [],
    'Specificity': [],
    'F1': [],
    'MCC': [],
    'ROC_AUC': [],
    'PRC_AUC': []
}

# Run nested CV
outer_cv = RepeatedStratifiedKFold(n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=SEED)

print(f"\nRunning {total_folds}-fold CV for Champion and ZeroR...")

for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X_champion, y_train)):
    X_fold_train = X_champion.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_champion.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Scale features
    scaler = StandardScaler()
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)
    
    # --- Champion Model ---
    # Inner CV for hyperparameter tuning
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    grid_search = GridSearchCV(base_model, param_grid, cv=inner_cv, scoring='f1', n_jobs=-1)
    grid_search.fit(X_fold_train_scaled, y_fold_train)
    
    best_model = grid_search.best_estimator_
    y_ch_pred = best_model.predict(X_fold_val_scaled)
    try:
        y_ch_proba = best_model.predict_proba(X_fold_val_scaled)[:, 1]
    except:
        y_ch_proba = np.full(len(y_fold_val), 0.5)
    
    # Champion metrics
    champion_fold_metrics['Accuracy'].append(accuracy_score(y_fold_val, y_ch_pred))
    champion_fold_metrics['Balanced_Accuracy'].append(balanced_accuracy_score(y_fold_val, y_ch_pred))
    champion_fold_metrics['Precision'].append(precision_score(y_fold_val, y_ch_pred, zero_division=0))
    champion_fold_metrics['Recall'].append(recall_score(y_fold_val, y_ch_pred, zero_division=0))
    # Specificity = TN / (TN + FP)
    tn, fp, fn, tp = confusion_matrix(y_fold_val, y_ch_pred, labels=[0,1]).ravel()
    champion_fold_metrics['Specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    champion_fold_metrics['F1'].append(f1_score(y_fold_val, y_ch_pred, zero_division=0))
    champion_fold_metrics['MCC'].append(matthews_corrcoef(y_fold_val, y_ch_pred))
    try:
        champion_fold_metrics['ROC_AUC'].append(roc_auc_score(y_fold_val, y_ch_proba))
    except:
        champion_fold_metrics['ROC_AUC'].append(0.5)
    try:
        champion_fold_metrics['PRC_AUC'].append(average_precision_score(y_fold_val, y_ch_proba))
    except:
        champion_fold_metrics['PRC_AUC'].append(0.5)
    
    # --- ZeroR Model ---
    zeror = DummyClassifier(strategy='stratified', random_state=SEED)
    zeror.fit(X_fold_train_scaled, y_fold_train)
    y_zr_pred = zeror.predict(X_fold_val_scaled)
    try:
        y_zr_proba = zeror.predict_proba(X_fold_val_scaled)[:, 1]
    except:
        y_zr_proba = np.full(len(y_fold_val), 0.5)
    
    # ZeroR metrics
    zeror_fold_metrics['Accuracy'].append(accuracy_score(y_fold_val, y_zr_pred))
    zeror_fold_metrics['Balanced_Accuracy'].append(balanced_accuracy_score(y_fold_val, y_zr_pred))
    zeror_fold_metrics['Precision'].append(precision_score(y_fold_val, y_zr_pred, zero_division=0))
    zeror_fold_metrics['Recall'].append(recall_score(y_fold_val, y_zr_pred, zero_division=0))
    # Specificity = TN / (TN + FP)
    tn_zr, fp_zr, fn_zr, tp_zr = confusion_matrix(y_fold_val, y_zr_pred, labels=[0,1]).ravel()
    zeror_fold_metrics['Specificity'].append(tn_zr / (tn_zr + fp_zr) if (tn_zr + fp_zr) > 0 else 0)
    zeror_fold_metrics['F1'].append(f1_score(y_fold_val, y_zr_pred, zero_division=0))
    zeror_fold_metrics['MCC'].append(matthews_corrcoef(y_fold_val, y_zr_pred))
    try:
        zeror_fold_metrics['ROC_AUC'].append(roc_auc_score(y_fold_val, y_zr_proba))
    except:
        zeror_fold_metrics['ROC_AUC'].append(0.5)
    try:
        zeror_fold_metrics['PRC_AUC'].append(average_precision_score(y_fold_val, y_zr_proba))
    except:
        zeror_fold_metrics['PRC_AUC'].append(0.5)
    
    if (fold_idx + 1) % 10 == 0:
        print(f"  Completed {fold_idx + 1}/{total_folds} folds...")

# Convert to arrays
for key in champion_fold_metrics:
    champion_fold_metrics[key] = np.array(champion_fold_metrics[key])
    zeror_fold_metrics[key] = np.array(zeror_fold_metrics[key])

print("Done!")

# =============================================================================
# COMPUTE STATISTICS
# =============================================================================

stats_results = {}
for metric in champion_fold_metrics.keys():
    ch_vals = champion_fold_metrics[metric]
    zr_vals = zeror_fold_metrics[metric]
    
    ch_mean = np.mean(ch_vals)
    ch_std = np.std(ch_vals, ddof=1)
    ch_ci = (np.percentile(ch_vals, 2.5), np.percentile(ch_vals, 97.5))
    
    zr_mean = np.mean(zr_vals)
    zr_std = np.std(zr_vals, ddof=1)
    zr_ci = (np.percentile(zr_vals, 2.5), np.percentile(zr_vals, 97.5))
    
    # Paired t-test
    t_stat, p_val = scipy_stats.ttest_rel(ch_vals, zr_vals)
    
    # Cohen's d for paired samples
    diff = ch_vals - zr_vals
    d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0
    
    stats_results[metric] = {
        'champion_vals': ch_vals,
        'champion_mean': ch_mean,
        'champion_std': ch_std,
        'champion_ci': ch_ci,
        'zeror_vals': zr_vals,
        'zeror_mean': zr_mean,
        'zeror_std': zr_std,
        'zeror_ci': zr_ci,
        'p_value': p_val,
        'cohens_d': d
    }

print(f"\nChampion: {algo}")
print(f"ZeroR: Stratified Dummy Classifier")
print(f"\nMetrics ({total_folds} Nested CV Folds - Paired Comparison):")
print("-" * 70)
for metric, res in stats_results.items():
    sig = '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns'
    print(f"\n{metric}:")
    print(f"  Champion: {res['champion_mean']:.3f} +/- {res['champion_std']:.3f} [95% CI: {res['champion_ci'][0]:.3f}-{res['champion_ci'][1]:.3f}]")
    print(f"  ZeroR:    {res['zeror_mean']:.3f} +/- {res['zeror_std']:.3f} [95% CI: {res['zeror_ci'][0]:.3f}-{res['zeror_ci'][1]:.3f}]")
    print(f"  p-value:  {res['p_value']:.4f} {sig}")
    print(f"  Cohen's d: {res['cohens_d']:.3f}")

# =============================================================================
# FIGURE 1: BAR CHARTS - CHAMPION VS ZEROR  - ALL METRICS
# =============================================================================

# CONSISTENT COLORS: Champion=blue, ZeroR=orange
COLOR_CHAMPION = '#2980b9'
COLOR_ZEROR = '#e67e22'

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Helper function to format value with CI
def fmt_val_ci(val, ci_low, ci_high):
    return f'{val:.2f}\n[{ci_low:.2f}-{ci_high:.2f}]'

# Panel 1: Classification metrics
panel1_metrics = ['Accuracy', 'Balanced_Accuracy', 'F1']
panel1_labels = ['Accuracy', 'Balanced Acc', 'F1']

ax1 = axes[0]
x1 = np.arange(len(panel1_metrics))
width = 0.35

ch_vals_p1 = [stats_results[m]['champion_mean'] for m in panel1_metrics]
ch_ci_low_p1 = [stats_results[m]['champion_ci'][0] for m in panel1_metrics]
ch_ci_high_p1 = [stats_results[m]['champion_ci'][1] for m in panel1_metrics]
ch_err_p1 = [(ch_ci_high_p1[i] - ch_ci_low_p1[i])/2 for i in range(len(panel1_metrics))]

zr_vals_p1 = [stats_results[m]['zeror_mean'] for m in panel1_metrics]
zr_ci_low_p1 = [stats_results[m]['zeror_ci'][0] for m in panel1_metrics]
zr_ci_high_p1 = [stats_results[m]['zeror_ci'][1] for m in panel1_metrics]
zr_err_p1 = [(zr_ci_high_p1[i] - zr_ci_low_p1[i])/2 for i in range(len(panel1_metrics))]

bars1a = ax1.bar(x1 - width/2, ch_vals_p1, width, yerr=ch_err_p1, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars1b = ax1.bar(x1 + width/2, zr_vals_p1, width, yerr=zr_err_p1, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel1_metrics):
    p_val = stats_results[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals_p1[i] + ch_err_p1[i], zr_vals_p1[i] + zr_err_p1[i])
    ax1.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    # Champion with CI
    ax1.text(bars1a[i].get_x() + bars1a[i].get_width()/2, ch_vals_p1[i] + ch_err_p1[i] + 0.02,
             fmt_val_ci(ch_vals_p1[i], ch_ci_low_p1[i], ch_ci_high_p1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    # ZeroR with CI
    ax1.text(bars1b[i].get_x() + bars1b[i].get_width()/2, zr_vals_p1[i] + zr_err_p1[i] + 0.02,
             fmt_val_ci(zr_vals_p1[i], zr_ci_low_p1[i], zr_ci_high_p1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax1.set_ylabel('Score (95% CI)', fontsize=11)
ax1.set_title('A. Classification Metrics', fontsize=13, fontweight='bold')
ax1.set_xticks(x1)
ax1.set_xticklabels(panel1_labels, fontsize=10)
ax1.set_ylim(0, 1.45)
ax1.legend(loc='lower right', fontsize=9)

# Panel 2: Detection metrics
panel2_metrics = ['Precision', 'Recall', 'Specificity']
panel2_labels = ['Precision', 'Recall', 'Specificity']

ax2 = axes[1]
x2 = np.arange(len(panel2_metrics))

ch_vals_p2 = [stats_results[m]['champion_mean'] for m in panel2_metrics]
ch_ci_low_p2 = [stats_results[m]['champion_ci'][0] for m in panel2_metrics]
ch_ci_high_p2 = [stats_results[m]['champion_ci'][1] for m in panel2_metrics]
ch_err_p2 = [(ch_ci_high_p2[i] - ch_ci_low_p2[i])/2 for i in range(len(panel2_metrics))]

zr_vals_p2 = [stats_results[m]['zeror_mean'] for m in panel2_metrics]
zr_ci_low_p2 = [stats_results[m]['zeror_ci'][0] for m in panel2_metrics]
zr_ci_high_p2 = [stats_results[m]['zeror_ci'][1] for m in panel2_metrics]
zr_err_p2 = [(zr_ci_high_p2[i] - zr_ci_low_p2[i])/2 for i in range(len(panel2_metrics))]

bars2a = ax2.bar(x2 - width/2, ch_vals_p2, width, yerr=ch_err_p2, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars2b = ax2.bar(x2 + width/2, zr_vals_p2, width, yerr=zr_err_p2, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel2_metrics):
    p_val = stats_results[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals_p2[i] + ch_err_p2[i], zr_vals_p2[i] + zr_err_p2[i])
    ax2.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax2.text(bars2a[i].get_x() + bars2a[i].get_width()/2, ch_vals_p2[i] + ch_err_p2[i] + 0.02,
             fmt_val_ci(ch_vals_p2[i], ch_ci_low_p2[i], ch_ci_high_p2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax2.text(bars2b[i].get_x() + bars2b[i].get_width()/2, zr_vals_p2[i] + zr_err_p2[i] + 0.02,
             fmt_val_ci(zr_vals_p2[i], zr_ci_low_p2[i], zr_ci_high_p2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax2.set_ylabel('Score (95% CI)', fontsize=11)
ax2.set_title('B. Detection Metrics', fontsize=13, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(panel2_labels, fontsize=10)
ax2.set_ylim(0, 1.45)
ax2.legend(loc='lower right', fontsize=9)

# Panel 3: Advanced metrics
panel3_metrics = ['ROC_AUC', 'PRC_AUC', 'MCC']
panel3_labels = ['ROC-AUC', 'PRC-AUC', 'MCC']

ax3 = axes[2]
x3 = np.arange(len(panel3_metrics))

ch_vals_p3 = [stats_results[m]['champion_mean'] for m in panel3_metrics]
ch_ci_low_p3 = [stats_results[m]['champion_ci'][0] for m in panel3_metrics]
ch_ci_high_p3 = [stats_results[m]['champion_ci'][1] for m in panel3_metrics]
ch_err_p3 = [(ch_ci_high_p3[i] - ch_ci_low_p3[i])/2 for i in range(len(panel3_metrics))]

zr_vals_p3 = [stats_results[m]['zeror_mean'] for m in panel3_metrics]
zr_ci_low_p3 = [stats_results[m]['zeror_ci'][0] for m in panel3_metrics]
zr_ci_high_p3 = [stats_results[m]['zeror_ci'][1] for m in panel3_metrics]
zr_err_p3 = [(zr_ci_high_p3[i] - zr_ci_low_p3[i])/2 for i in range(len(panel3_metrics))]

bars3a = ax3.bar(x3 - width/2, ch_vals_p3, width, yerr=ch_err_p3, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars3b = ax3.bar(x3 + width/2, zr_vals_p3, width, yerr=zr_err_p3, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel3_metrics):
    p_val = stats_results[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals_p3[i] + ch_err_p3[i], zr_vals_p3[i] + zr_err_p3[i])
    ax3.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax3.text(bars3a[i].get_x() + bars3a[i].get_width()/2, ch_vals_p3[i] + ch_err_p3[i] + 0.02,
             fmt_val_ci(ch_vals_p3[i], ch_ci_low_p3[i], ch_ci_high_p3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax3.text(bars3b[i].get_x() + bars3b[i].get_width()/2, zr_vals_p3[i] + zr_err_p3[i] + 0.02,
             fmt_val_ci(zr_vals_p3[i], zr_ci_low_p3[i], zr_ci_high_p3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax3.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax3.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
ax3.set_ylabel('Score (95% CI)', fontsize=11)
ax3.set_title('C. Advanced Metrics', fontsize=13, fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(panel3_labels, fontsize=10)
ax3.set_ylim(-0.4, 1.45)
ax3.legend(loc='lower right', fontsize=9)

plt.suptitle(f'{algo} vs ZeroR - {total_folds} Nested CV Folds ({OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats)\n*** p<0.001, ** p<0.01, * p<0.05',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
savefig(OUTDIR, f"baseline_comparison_bars_{total_folds}cv")
plt.show()

print(f"\nSaved: {OUTDIR}/baseline_comparison_bars_{total_folds}cv.png")

# =============================================================================
# FIGURE 2: PERFORMANCE TREND LINES OVER NESTED CV FOLDS
# =============================================================================

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

metrics_to_plot = ['Accuracy', 'Balanced_Accuracy', 'F1', 'Precision', 'Recall', 'Specificity', 'ROC_AUC', 'PRC_AUC', 'MCC']
x_vals = np.arange(1, total_folds + 1)

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    ch_trend = champion_fold_metrics[metric]
    zr_trend = zeror_fold_metrics[metric]
    
    ax.plot(x_vals, ch_trend, '#2980b9', linestyle='-', marker='o', linewidth=2, markersize=4,
            label=f'Champion (mean={np.mean(ch_trend):.3f})')
    ax.plot(x_vals, zr_trend, '#e67e22', linestyle='--', marker='s', linewidth=2, markersize=4,
            label=f'ZeroR (mean={np.mean(zr_trend):.3f})')
    
    ax.set_title(f'{metric.replace("_", " ")} Trend', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'Nested CV Fold (out of {total_folds})', fontsize=10)
    ax.set_ylabel(metric.replace('_', ' '), fontsize=10)
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)
    
    ch_mean = np.mean(ch_trend)
    zr_mean = np.mean(zr_trend)
    improvement = ((ch_mean - zr_mean) / abs(zr_mean)) * 100 if zr_mean != 0 else 0
    ax.text(0.02, 0.98, f'Delta={improvement:+.1f}%', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightgreen' if improvement > 0 else 'lightcoral', alpha=0.7))

# All 9 subplots used

plt.suptitle(f'Performance Trends - {algo} vs ZeroR\n{total_folds} Nested CV Outer Folds ({OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats)',
             fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
savefig(OUTDIR, f"performance_trends_{total_folds}cv")
plt.show()

print(f"\nSaved: {OUTDIR}/performance_trends_{total_folds}cv.png")


# =============================================================================
# FIGURE 3: COHEN'S D EFFECT SIZE
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

all_metrics = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC', 'MCC']
cohens_d_vals = [stats_results[m]['cohens_d'] for m in all_metrics]
metric_labels = ['Acc', 'Bal Acc', 'Prec', 'Recall', 'Spec', 'F1', 'ROC', 'PRC', 'MCC']

# Color bars by effect size magnitude
colors = []
for d in cohens_d_vals:
    abs_d = abs(d)
    if abs_d >= 0.8:
        colors.append('#2ecc71')  # Large effect - green
    elif abs_d >= 0.5:
        colors.append('#f39c12')  # Medium effect - orange
    elif abs_d >= 0.2:
        colors.append('#3498db')  # Small effect - blue
    else:
        colors.append('#95a5a6')  # Negligible - gray

bars = ax.bar(range(len(all_metrics)), cohens_d_vals, color=colors, edgecolor='black', linewidth=1.5, alpha=0.85)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, cohens_d_vals)):
    y_pos = val + 0.05 if val >= 0 else val - 0.15
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.2f}',
            ha='center', va='bottom' if val >= 0 else 'top', fontsize=11, fontweight='bold')

# Reference lines for effect size thresholds
ax.axhline(y=0.8, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Large (d=0.8)')
ax.axhline(y=0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Medium (d=0.5)')
ax.axhline(y=0.2, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label='Small (d=0.2)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

ax.set_ylabel("Cohen's d Effect Size", fontsize=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_title(f"Effect Size: {algo} vs ZeroR - Cohen's d (paired, n={total_folds} folds)", fontsize=14, fontweight='bold')
ax.set_xticks(range(len(all_metrics)))
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(-0.5, max(cohens_d_vals) + 0.5)
ax.legend(loc='upper right', fontsize=9, title='Effect Size Thresholds')
ax.grid(axis='y', alpha=0.3)

# Add interpretation text box
interpretation = []
for m, d in zip(all_metrics, cohens_d_vals):
    size = 'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small' if abs(d) >= 0.2 else 'negligible'
    interpretation.append(f"{m}: {size}")

textstr = "Effect sizes: " + ", ".join(interpretation)
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=8,
        verticalalignment='top', bbox=props, wrap=True)

plt.tight_layout()
savefig(OUTDIR, "cohens_d_effect_size")
plt.show()

print(f"Saved: {OUTDIR}/cohens_d_effect_size.png")

# =============================================================================
# FIGURE 4: TEST SET - CHAMPION V# =============================================================================
# TEST SET: Champion vs ZeroR Comparison
# =============================================================================

print("\n" + "="*70)
print("TEST SET: Champion vs ZeroR Comparison")
print("="*70)

# CONSISTENT COLORS
COLOR_CHAMPION = '#2980b9'
COLOR_ZEROR = '#e67e22'

# Train ZeroR on training data
zeror_test = DummyClassifier(strategy='stratified', random_state=SEED)
zeror_test.fit(X_train[champion_features], y_train)

# Get ZeroR predictions on test set
y_zr_test_pred = zeror_test.predict(X_test[champion_features])

# For probability-based metrics, use class proportions as probability estimate
# DummyClassifier 'stratified' uses training class proportions
class_prior = np.mean(y_train)  # P(class=1)
y_zr_test_proba = np.full(len(y_test), class_prior)

# Calculate metrics for ZeroR on test set
zeror_test_metrics = {}
zeror_test_metrics['Accuracy'] = accuracy_score(y_test, y_zr_test_pred)
zeror_test_metrics['Balanced_Accuracy'] = balanced_accuracy_score(y_test, y_zr_test_pred)
zeror_test_metrics['Precision'] = precision_score(y_test, y_zr_test_pred, zero_division=0)
zeror_test_metrics['Recall'] = recall_score(y_test, y_zr_test_pred, zero_division=0)
zeror_test_metrics['F1'] = f1_score(y_test, y_zr_test_pred, zero_division=0)
zeror_test_metrics['MCC'] = matthews_corrcoef(y_test, y_zr_test_pred)

# Specificity
tn_zr, fp_zr, fn_zr, tp_zr = confusion_matrix(y_test, y_zr_test_pred, labels=[0,1]).ravel()
zeror_test_metrics['Specificity'] = tn_zr / (tn_zr + fp_zr) if (tn_zr + fp_zr) > 0 else 0

# ROC-AUC and PRC-AUC for ZeroR (using class prior as probability)
try:
    zeror_test_metrics['ROC_AUC'] = roc_auc_score(y_test, y_zr_test_proba)
except:
    zeror_test_metrics['ROC_AUC'] = 0.5
try:
    zeror_test_metrics['PRC_AUC'] = average_precision_score(y_test, y_zr_test_proba)
except:
    zeror_test_metrics['PRC_AUC'] = np.mean(y_test)  # Baseline for PRC is class proportion

# Champion test metrics - use actual predictions, not stored values
# This ensures we have the actual test set performance
champion_test_metrics = {}
champion_test_metrics['Accuracy'] = accuracy_score(y_test, y_pred_final)
champion_test_metrics['Balanced_Accuracy'] = balanced_accuracy_score(y_test, y_pred_final)
champion_test_metrics['Precision'] = precision_score(y_test, y_pred_final, zero_division=0)
champion_test_metrics['Recall'] = recall_score(y_test, y_pred_final, zero_division=0)
champion_test_metrics['F1'] = f1_score(y_test, y_pred_final, zero_division=0)
champion_test_metrics['MCC'] = matthews_corrcoef(y_test, y_pred_final)

# Specificity for champion
tn_ch, fp_ch, fn_ch, tp_ch = confusion_matrix(y_test, y_pred_final, labels=[0,1]).ravel()
champion_test_metrics['Specificity'] = tn_ch / (tn_ch + fp_ch) if (tn_ch + fp_ch) > 0 else 0

# ROC-AUC and PRC-AUC for Champion
try:
    y_proba_ch = y_proba_final if 'y_proba_final' in dir() else FINAL_CHAMPION.get('y_proba_calibrated', None)
    if y_proba_ch is not None:
        champion_test_metrics['ROC_AUC'] = roc_auc_score(y_test, y_proba_ch)
        champion_test_metrics['PRC_AUC'] = average_precision_score(y_test, y_proba_ch)
    else:
        champion_test_metrics['ROC_AUC'] = 0.5
        champion_test_metrics['PRC_AUC'] = 0.5
except Exception as e:
    print(f"Warning: Could not calculate AUC metrics: {e}")
    champion_test_metrics['ROC_AUC'] = 0.5
    champion_test_metrics['PRC_AUC'] = 0.5

print("\nTest Set Metrics Calculated:")
print("-" * 50)
all_test_metrics = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC', 'MCC']
for m in all_test_metrics:
    print(f"  {m}: Champion={champion_test_metrics[m]:.3f}, ZeroR={zeror_test_metrics[m]:.3f}")

# Bootstrap for 95% CI with proper handling
print(f"\nBootstrapping test set metrics ({N_BOOTSTRAP} iterations)...")

test_set_stats = {}
n_boot = N_BOOTSTRAP
n = len(y_test)

# Get probability arrays
y_proba_ch_arr = np.array(y_proba_ch) if 'y_proba_ch' in dir() and y_proba_ch is not None else np.full(n, 0.5)
y_zr_proba_arr = np.array(y_zr_test_proba)
y_ch_pred_arr = np.array(y_pred_final)
y_zr_pred_arr = np.array(y_zr_test_pred)
y_true_arr = np.array(y_test)

for metric in all_test_metrics:
    ch_boots = []
    zr_boots = []

    for b in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        y_true_b = y_true_arr[idx]
        y_ch_b = y_ch_pred_arr[idx]
        y_zr_b = y_zr_pred_arr[idx]
        y_ch_proba_b = y_proba_ch_arr[idx]
        y_zr_proba_b = y_zr_proba_arr[idx]

        try:
            if metric == 'Accuracy':
                ch_boots.append(accuracy_score(y_true_b, y_ch_b))
                zr_boots.append(accuracy_score(y_true_b, y_zr_b))
            elif metric == 'Balanced_Accuracy':
                ch_boots.append(balanced_accuracy_score(y_true_b, y_ch_b))
                zr_boots.append(balanced_accuracy_score(y_true_b, y_zr_b))
            elif metric == 'Precision':
                ch_boots.append(precision_score(y_true_b, y_ch_b, zero_division=0))
                zr_boots.append(precision_score(y_true_b, y_zr_b, zero_division=0))
            elif metric == 'Recall':
                ch_boots.append(recall_score(y_true_b, y_ch_b, zero_division=0))
                zr_boots.append(recall_score(y_true_b, y_zr_b, zero_division=0))
            elif metric == 'Specificity':
                # Champion specificity
                tn_c, fp_c, _, _ = confusion_matrix(y_true_b, y_ch_b, labels=[0,1]).ravel()
                ch_boots.append(tn_c / (tn_c + fp_c) if (tn_c + fp_c) > 0 else 0)
                # ZeroR specificity
                tn_z, fp_z, _, _ = confusion_matrix(y_true_b, y_zr_b, labels=[0,1]).ravel()
                zr_boots.append(tn_z / (tn_z + fp_z) if (tn_z + fp_z) > 0 else 0)
            elif metric == 'F1':
                ch_boots.append(f1_score(y_true_b, y_ch_b, zero_division=0))
                zr_boots.append(f1_score(y_true_b, y_zr_b, zero_division=0))
            elif metric == 'MCC':
                ch_boots.append(matthews_corrcoef(y_true_b, y_ch_b))
                zr_boots.append(matthews_corrcoef(y_true_b, y_zr_b))
            elif metric == 'ROC_AUC':
                # Need both classes in bootstrap sample
                if len(np.unique(y_true_b)) == 2:
                    ch_boots.append(roc_auc_score(y_true_b, y_ch_proba_b))
                    zr_boots.append(roc_auc_score(y_true_b, y_zr_proba_b))
            elif metric == 'PRC_AUC':
                if len(np.unique(y_true_b)) == 2:
                    ch_boots.append(average_precision_score(y_true_b, y_ch_proba_b))
                    zr_boots.append(average_precision_score(y_true_b, y_zr_proba_b))
        except:
            pass

    # Calculate stats
    ch_val = champion_test_metrics[metric]
    zr_val = zeror_test_metrics[metric]

    if len(ch_boots) > 100:
        ch_ci = (np.percentile(ch_boots, 2.5), np.percentile(ch_boots, 97.5))
        ch_std = np.std(ch_boots)
    else:
        ch_ci = (ch_val - 0.1, ch_val + 0.1)
        ch_std = 0.1

    if len(zr_boots) > 100:
        zr_ci = (np.percentile(zr_boots, 2.5), np.percentile(zr_boots, 97.5))
        zr_std = np.std(zr_boots)
    else:
        zr_ci = (zr_val - 0.1, zr_val + 0.1)
        zr_std = 0.1

    # Statistical test (permutation-based p-value approximation)
    # Compare bootstrap distributions
    if len(ch_boots) > 100 and len(zr_boots) > 100:
        # Calculate how often ZeroR >= Champion in bootstrap
        ch_arr = np.array(ch_boots[:min(len(ch_boots), len(zr_boots))])
        zr_arr = np.array(zr_boots[:min(len(ch_boots), len(zr_boots))])
        p_value = np.mean(zr_arr >= ch_arr)
        p_value = max(p_value, 1/n_boot)  # Minimum p-value
    else:
        p_value = 0.5

    test_set_stats[metric] = {
        'champion_val': ch_val,
        'champion_ci': ch_ci,
        'champion_std': ch_std,
        'zeror_val': zr_val,
        'zeror_ci': zr_ci,
        'zeror_std': zr_std,
        'p_value': p_value,
    }

print("\nTest Set Results (Champion vs ZeroR):")
print("-" * 70)
for metric in all_test_metrics:
    s = test_set_stats[metric]
    sig = '***' if s['p_value'] < 0.001 else '**' if s['p_value'] < 0.01 else '*' if s['p_value'] < 0.05 else 'ns'
    print(f"  {metric}:")
    print(f"    Champion: {s['champion_val']:.3f} [95% CI: {s['champion_ci'][0]:.3f}-{s['champion_ci'][1]:.3f}]")
    print(f"    ZeroR:    {s['zeror_val']:.3f} [95% CI: {s['zeror_ci'][0]:.3f}-{s['zeror_ci'][1]:.3f}]")
    print(f"    p-value:  {s['p_value']:.4f} {sig}")

# =============================================================================
# TEST SET BAR CHART WITH SIGNIFICANCE
# =============================================================================

def fmt_val_ci(val, ci_low, ci_high):
    return f'{val:.2f}\n[{ci_low:.2f}-{ci_high:.2f}]'

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel 1: Classification
panel1 = ['Accuracy', 'Balanced_Accuracy', 'F1']
panel1_labels = ['Accuracy', 'Balanced Acc', 'F1']

ax1 = axes[0]
x1 = np.arange(len(panel1))
width = 0.35

ch_vals1 = [test_set_stats[m]['champion_val'] for m in panel1]
ch_ci_low1 = [test_set_stats[m]['champion_ci'][0] for m in panel1]
ch_ci_high1 = [test_set_stats[m]['champion_ci'][1] for m in panel1]
ch_err1 = [(ch_ci_high1[i] - ch_ci_low1[i])/2 for i in range(len(panel1))]

zr_vals1 = [test_set_stats[m]['zeror_val'] for m in panel1]
zr_ci_low1 = [test_set_stats[m]['zeror_ci'][0] for m in panel1]
zr_ci_high1 = [test_set_stats[m]['zeror_ci'][1] for m in panel1]
zr_err1 = [(zr_ci_high1[i] - zr_ci_low1[i])/2 for i in range(len(panel1))]

bars1a = ax1.bar(x1 - width/2, ch_vals1, width, yerr=ch_err1, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars1b = ax1.bar(x1 + width/2, zr_vals1, width, yerr=zr_err1, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel1):
    p_val = test_set_stats[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals1[i] + ch_err1[i], zr_vals1[i] + zr_err1[i])
    ax1.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    # Labels with CI
    ax1.text(bars1a[i].get_x() + bars1a[i].get_width()/2, ch_vals1[i] + ch_err1[i] + 0.02,
             fmt_val_ci(ch_vals1[i], ch_ci_low1[i], ch_ci_high1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax1.text(bars1b[i].get_x() + bars1b[i].get_width()/2, zr_vals1[i] + zr_err1[i] + 0.02,
             fmt_val_ci(zr_vals1[i], zr_ci_low1[i], zr_ci_high1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax1.set_ylabel('Score (95% CI)', fontsize=11)
ax1.set_title('A. Classification Metrics', fontsize=13, fontweight='bold')
ax1.set_xticks(x1)
ax1.set_xticklabels(panel1_labels, fontsize=10)
ax1.set_ylim(0, 1.45)
ax1.legend(loc='lower right', fontsize=9)

# Panel 2: Detection
panel2 = ['Precision', 'Recall', 'Specificity']
panel2_labels = ['Precision', 'Recall', 'Specificity']

ax2 = axes[1]
x2 = np.arange(len(panel2))

ch_vals2 = [test_set_stats[m]['champion_val'] for m in panel2]
ch_ci_low2 = [test_set_stats[m]['champion_ci'][0] for m in panel2]
ch_ci_high2 = [test_set_stats[m]['champion_ci'][1] for m in panel2]
ch_err2 = [(ch_ci_high2[i] - ch_ci_low2[i])/2 for i in range(len(panel2))]

zr_vals2 = [test_set_stats[m]['zeror_val'] for m in panel2]
zr_ci_low2 = [test_set_stats[m]['zeror_ci'][0] for m in panel2]
zr_ci_high2 = [test_set_stats[m]['zeror_ci'][1] for m in panel2]
zr_err2 = [(zr_ci_high2[i] - zr_ci_low2[i])/2 for i in range(len(panel2))]

bars2a = ax2.bar(x2 - width/2, ch_vals2, width, yerr=ch_err2, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars2b = ax2.bar(x2 + width/2, zr_vals2, width, yerr=zr_err2, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel2):
    p_val = test_set_stats[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals2[i] + ch_err2[i], zr_vals2[i] + zr_err2[i])
    ax2.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax2.text(bars2a[i].get_x() + bars2a[i].get_width()/2, ch_vals2[i] + ch_err2[i] + 0.02,
             fmt_val_ci(ch_vals2[i], ch_ci_low2[i], ch_ci_high2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax2.text(bars2b[i].get_x() + bars2b[i].get_width()/2, zr_vals2[i] + zr_err2[i] + 0.02,
             fmt_val_ci(zr_vals2[i], zr_ci_low2[i], zr_ci_high2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax2.set_ylabel('Score (95% CI)', fontsize=11)
ax2.set_title('B. Detection Metrics', fontsize=13, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(panel2_labels, fontsize=10)
ax2.set_ylim(0, 1.45)
ax2.legend(loc='lower right', fontsize=9)

# Panel 3: Advanced
panel3 = ['ROC_AUC', 'PRC_AUC', 'MCC']
panel3_labels = ['ROC-AUC', 'PRC-AUC', 'MCC']

ax3 = axes[2]
x3 = np.arange(len(panel3))

ch_vals3 = [test_set_stats[m]['champion_val'] for m in panel3]
ch_ci_low3 = [test_set_stats[m]['champion_ci'][0] for m in panel3]
ch_ci_high3 = [test_set_stats[m]['champion_ci'][1] for m in panel3]
ch_err3 = [(ch_ci_high3[i] - ch_ci_low3[i])/2 for i in range(len(panel3))]

zr_vals3 = [test_set_stats[m]['zeror_val'] for m in panel3]
zr_ci_low3 = [test_set_stats[m]['zeror_ci'][0] for m in panel3]
zr_ci_high3 = [test_set_stats[m]['zeror_ci'][1] for m in panel3]
zr_err3 = [(zr_ci_high3[i] - zr_ci_low3[i])/2 for i in range(len(panel3))]

bars3a = ax3.bar(x3 - width/2, ch_vals3, width, yerr=ch_err3, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars3b = ax3.bar(x3 + width/2, zr_vals3, width, yerr=zr_err3, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel3):
    p_val = test_set_stats[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals3[i] + ch_err3[i], zr_vals3[i] + zr_err3[i])
    ax3.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax3.text(bars3a[i].get_x() + bars3a[i].get_width()/2, ch_vals3[i] + ch_err3[i] + 0.02,
             fmt_val_ci(ch_vals3[i], ch_ci_low3[i], ch_ci_high3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax3.text(bars3b[i].get_x() + bars3b[i].get_width()/2, zr_vals3[i] + zr_err3[i] + 0.02,
             fmt_val_ci(zr_vals3[i], zr_ci_low3[i], zr_ci_high3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax3.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax3.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
ax3.set_ylabel('Score (95% CI)', fontsize=11)
ax3.set_title('C. Advanced Metrics', fontsize=13, fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(panel3_labels, fontsize=10)
ax3.set_ylim(-0.4, 1.45)
ax3.legend(loc='lower right', fontsize=9)

plt.suptitle(f'{algo} vs ZeroR - TEST SET (n={len(y_test)})\n95% CI from bootstrap ({N_BOOTSTRAP} iterations) | *** p<0.001, ** p<0.01, * p<0.05',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
savefig(OUTDIR, "test_set_comparison_bars")
plt.show()

print(f"\nSaved: {OUTDIR}/test_set_comparison_bars.png")

# =============================================================================
# TEST SET COHEN'S D
# =============================================================================

print("\nCalculating Cohen's d for test set...")

test_cohens_d = {}
for metric in all_test_metrics:
    ch_val = test_set_stats[metric]['champion_val']
    zr_val = test_set_stats[metric]['zeror_val']
    ch_std = test_set_stats[metric]['champion_std']
    zr_std = test_set_stats[metric]['zeror_std']

    # Pooled standard deviation
    pooled_std = np.sqrt((ch_std**2 + zr_std**2) / 2)

    # Cohen's d
    if pooled_std > 0.001:  # Avoid division by very small numbers
        test_cohens_d[metric] = (ch_val - zr_val) / pooled_std
    else:
        # Use difference directly if std is too small (essentially deterministic)
        test_cohens_d[metric] = (ch_val - zr_val) * 10 if ch_val != zr_val else 0

    print(f"  {metric}: d = {test_cohens_d[metric]:.2f} (ch={ch_val:.3f}, zr={zr_val:.3f}, pooled_std={pooled_std:.3f})")

fig, ax = plt.subplots(figsize=(14, 6))

metric_labels = ['Acc', 'Bal Acc', 'Prec', 'Recall', 'Spec', 'F1', 'ROC', 'PRC', 'MCC']
d_vals = [test_cohens_d[m] for m in all_test_metrics]

# Color by effect size
colors = []
for d in d_vals:
    abs_d = abs(d)
    if abs_d >= 0.8:
        colors.append('#2ecc71')  # Large - green
    elif abs_d >= 0.5:
        colors.append('#f39c12')  # Medium - orange
    elif abs_d >= 0.2:
        colors.append('#3498db')  # Small - blue
    else:
        colors.append('#95a5a6')  # Negligible - gray

bars = ax.bar(range(len(all_test_metrics)), d_vals, color=colors, edgecolor='black', linewidth=1.5, alpha=0.85)

for i, (bar, val) in enumerate(zip(bars, d_vals)):
    y_pos = val + 0.1 if val >= 0 else val - 0.2
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.2f}',
            ha='center', va='bottom' if val >= 0 else 'top', fontsize=11, fontweight='bold')

ax.axhline(y=0.8, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Large (d=0.8)')
ax.axhline(y=0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Medium (d=0.5)')
ax.axhline(y=0.2, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label='Small (d=0.2)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

ax.set_ylabel("Cohen's d Effect Size", fontsize=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_title(f"TEST SET Effect Size: {algo} vs ZeroR\nCohen's d (n={len(y_test)})", fontsize=14, fontweight='bold')
ax.set_xticks(range(len(all_test_metrics)))
ax.set_xticklabels(metric_labels, fontsize=11)
ax.legend(loc='upper right', fontsize=9, title='Effect Size Thresholds')
ax.grid(axis='y', alpha=0.3)

# Set y-limits based on data
y_min = min(d_vals) - 0.5
y_max = max(d_vals) + 0.5
ax.set_ylim(y_min, y_max)

plt.tight_layout()
savefig(OUTDIR, "test_set_cohens_d")
plt.show()

print(f"Saved: {OUTDIR}/test_set_cohens_d.png")

# Store results
test_zeror_comparison = {
    'champion_metrics': champion_test_metrics,
    'zeror_metrics': zeror_test_metrics,
    'stats': test_set_stats,
    'cohens_d': test_cohens_d,
}

# =============================================================================
# EXCEL EXPORT - CV BASELINE COMPARISON 
# =============================================================================
print("\n" + "="*70)
print("EXPORTING BASELINE COMPARISON RESULTS TO EXCEL")
print("="*70)

# Create CV Baseline Comparison Excel file
cv_baseline_xlsx = os.path.join(OUTDIR, "baseline_cv_comparison.xlsx")

cv_sheets = {}

# Sheet 1: Summary Statistics
metric_order = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC', 'MCC']
summary_rows = []
for metric in metric_order:
    if metric in stats_results:
        res = stats_results[metric]
        improvement = ((res['champion_mean'] - res['zeror_mean']) / abs(res['zeror_mean']) * 100) if res['zeror_mean'] != 0 else 0
        sig = '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns'
        effect = 'large' if abs(res['cohens_d']) >= 0.8 else 'medium' if abs(res['cohens_d']) >= 0.5 else 'small' if abs(res['cohens_d']) >= 0.2 else 'negligible'
        summary_rows.append({
            'Metric': metric,
            'Champion_Mean': round(res['champion_mean'], 4),
            'Champion_Std': round(res['champion_std'], 4),
            'Champion_CI_Lower': round(res['champion_ci'][0], 4),
            'Champion_CI_Upper': round(res['champion_ci'][1], 4),
            'ZeroR_Mean': round(res['zeror_mean'], 4),
            'ZeroR_Std': round(res['zeror_std'], 4),
            'ZeroR_CI_Lower': round(res['zeror_ci'][0], 4),
            'ZeroR_CI_Upper': round(res['zeror_ci'][1], 4),
            'Difference': round(res['champion_mean'] - res['zeror_mean'], 4),
            'Improvement_Pct': round(improvement, 2),
            'P_Value': res['p_value'],
            'Significance': sig,
            'Cohens_d': round(res['cohens_d'], 4),
            'Effect_Size': effect
        })
cv_sheets['1_Summary_Stats'] = pd.DataFrame(summary_rows)

# Sheet 2: Fold-Level Data (Raw)
fold_rows = []
n_folds = len(next(iter(champion_fold_metrics.values())))
for fold_idx in range(n_folds):
    row = {'Fold': fold_idx + 1}
    for metric in metric_order:
        if metric in champion_fold_metrics:
            row[f'Champion_{metric}'] = round(champion_fold_metrics[metric][fold_idx], 4)
            row[f'ZeroR_{metric}'] = round(zeror_fold_metrics[metric][fold_idx], 4)
            row[f'Diff_{metric}'] = round(champion_fold_metrics[metric][fold_idx] - zeror_fold_metrics[metric][fold_idx], 4)
    fold_rows.append(row)
cv_sheets['2_Fold_Level_Data'] = pd.DataFrame(fold_rows)

# Sheet 3: Cohen's d Summary
cohens_rows = []
for metric in metric_order:
    if metric in stats_results:
        d = stats_results[metric]['cohens_d']
        effect = 'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small' if abs(d) >= 0.2 else 'negligible'
        cohens_rows.append({
            'Metric': metric,
            'Cohens_d': round(d, 4),
            'Abs_Cohens_d': round(abs(d), 4),
            'Effect_Size': effect,
            'Interpretation': f'Champion is {effect}ly better than ZeroR' if d > 0 else f'ZeroR is {effect}ly better' if d < 0 else 'No difference'
        })
cv_sheets['3_Cohens_d'] = pd.DataFrame(cohens_rows)

# Sheet 4: Analysis Info
info_rows = [
    {'Parameter': 'Champion_Algorithm', 'Value': algo},
    {'Parameter': 'Champion_Features', 'Value': ', '.join(champion_features)},
    {'Parameter': 'N_Features', 'Value': len(champion_features)},
    {'Parameter': 'Baseline_Method', 'Value': 'ZeroR (Stratified Dummy Classifier)'},
    {'Parameter': 'N_Outer_Folds', 'Value': total_folds},
    {'Parameter': 'CV_Structure', 'Value': f'{OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats'},
    {'Parameter': 'Statistical_Test', 'Value': 'Paired t-test'},
    {'Parameter': 'Random_Seed', 'Value': SEED}
]
cv_sheets['4_Analysis_Info'] = pd.DataFrame(info_rows)

# Write CV baseline Excel
with pd.ExcelWriter(cv_baseline_xlsx, engine='openpyxl') as writer:
    for sheet_name, df in cv_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"[OK] CV Baseline Comparison saved to: {cv_baseline_xlsx}")

# =============================================================================
# EXCEL EXPORT - TEST SET COMPARISON
# =============================================================================

test_comparison_xlsx = os.path.join(OUTDIR, "test_set_zeror_comparison.xlsx")

test_sheets = {}

# Sheet 1: Summary Statistics
test_summary_rows = []
for metric in metric_order:
    if metric in test_set_stats:
        s = test_set_stats[metric]
        d = test_cohens_d.get(metric, 0)
        improvement = ((s['champion_val'] - s['zeror_val']) / abs(s['zeror_val']) * 100) if s['zeror_val'] != 0 else 0
        sig = '***' if s['p_value'] < 0.001 else '**' if s['p_value'] < 0.01 else '*' if s['p_value'] < 0.05 else 'ns'
        effect = 'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small' if abs(d) >= 0.2 else 'negligible'
        test_summary_rows.append({
            'Metric': metric,
            'Champion_Value': round(s['champion_val'], 4),
            'Champion_CI_Lower': round(s['champion_ci'][0], 4),
            'Champion_CI_Upper': round(s['champion_ci'][1], 4),
            'Champion_Std_Bootstrap': round(s['champion_std'], 4),
            'ZeroR_Value': round(s['zeror_val'], 4),
            'ZeroR_CI_Lower': round(s['zeror_ci'][0], 4),
            'ZeroR_CI_Upper': round(s['zeror_ci'][1], 4),
            'ZeroR_Std_Bootstrap': round(s['zeror_std'], 4),
            'Difference': round(s['champion_val'] - s['zeror_val'], 4),
            'Improvement_Pct': round(improvement, 2),
            'P_Value': s['p_value'],
            'Significance': sig,
            'Cohens_d': round(d, 4),
            'Effect_Size': effect
        })
test_sheets['1_Summary_Stats'] = pd.DataFrame(test_summary_rows)

# Sheet 2: Cohen's d Summary
test_cohens_rows = []
for metric in metric_order:
    if metric in test_cohens_d:
        d = test_cohens_d[metric]
        effect = 'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small' if abs(d) >= 0.2 else 'negligible'
        test_cohens_rows.append({
            'Metric': metric,
            'Cohens_d': round(d, 4),
            'Abs_Cohens_d': round(abs(d), 4),
            'Effect_Size': effect,
            'Interpretation': f'Champion is {effect}ly better than ZeroR' if d > 0 else f'ZeroR is {effect}ly better' if d < 0 else 'No difference'
        })
test_sheets['2_Cohens_d'] = pd.DataFrame(test_cohens_rows)

# Sheet 3: Raw Values
raw_rows = [
    {'Model': 'Champion', **{metric: round(champion_test_metrics[metric], 4) for metric in metric_order}},
    {'Model': 'ZeroR', **{metric: round(zeror_test_metrics[metric], 4) for metric in metric_order}}
]
test_sheets['3_Raw_Values'] = pd.DataFrame(raw_rows)

# Sheet 4: Analysis Info
test_info_rows = [
    {'Parameter': 'Champion_Algorithm', 'Value': algo},
    {'Parameter': 'Champion_Features', 'Value': ', '.join(champion_features)},
    {'Parameter': 'N_Features', 'Value': len(champion_features)},
    {'Parameter': 'Baseline_Method', 'Value': 'ZeroR (Stratified Dummy Classifier)'},
    {'Parameter': 'Test_Set_Size', 'Value': len(y_test)},
    {'Parameter': 'Bootstrap_Iterations', 'Value': N_BOOTSTRAP},
    {'Parameter': 'CI_Level', 'Value': '95%'},
    {'Parameter': 'Random_Seed', 'Value': SEED}
]
test_sheets['4_Analysis_Info'] = pd.DataFrame(test_info_rows)

# Write Test Set Excel
with pd.ExcelWriter(test_comparison_xlsx, engine='openpyxl') as writer:
    for sheet_name, df in test_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"[OK] Test Set Comparison saved to: {test_comparison_xlsx}")

print("\n" + "="*70)
print("BASELINE COMPARISON EXCEL FILES GENERATED")
print("="*70)


# =============================================================================
# ADDITIONAL FIGURES WITHOUT MCC (8 METRICS ONLY)
# =============================================================================
print("\n" + "="*70)
print("GENERATING ADDITIONAL FIGURES WITHOUT MCC")
print("="*70)

# =============================================================================
# FIGURE 1b: BAR CHARTS - WITHOUT MCC 
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Panel 1: Classification metrics (same as before)
panel1_metrics = ['Accuracy', 'Balanced_Accuracy', 'F1']
panel1_labels = ['Accuracy', 'Balanced Acc', 'F1']

ax1 = axes[0]
x1 = np.arange(len(panel1_metrics))
width = 0.35

ch_vals_p1 = [stats_results[m]['champion_mean'] for m in panel1_metrics]
ch_ci_low_p1 = [stats_results[m]['champion_ci'][0] for m in panel1_metrics]
ch_ci_high_p1 = [stats_results[m]['champion_ci'][1] for m in panel1_metrics]
ch_err_p1 = [(ch_ci_high_p1[i] - ch_ci_low_p1[i])/2 for i in range(len(panel1_metrics))]

zr_vals_p1 = [stats_results[m]['zeror_mean'] for m in panel1_metrics]
zr_ci_low_p1 = [stats_results[m]['zeror_ci'][0] for m in panel1_metrics]
zr_ci_high_p1 = [stats_results[m]['zeror_ci'][1] for m in panel1_metrics]
zr_err_p1 = [(zr_ci_high_p1[i] - zr_ci_low_p1[i])/2 for i in range(len(panel1_metrics))]

bars1a = ax1.bar(x1 - width/2, ch_vals_p1, width, yerr=ch_err_p1, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars1b = ax1.bar(x1 + width/2, zr_vals_p1, width, yerr=zr_err_p1, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel1_metrics):
    p_val = stats_results[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals_p1[i] + ch_err_p1[i], zr_vals_p1[i] + zr_err_p1[i])
    ax1.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax1.text(bars1a[i].get_x() + bars1a[i].get_width()/2, ch_vals_p1[i] + ch_err_p1[i] + 0.02,
             fmt_val_ci(ch_vals_p1[i], ch_ci_low_p1[i], ch_ci_high_p1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax1.text(bars1b[i].get_x() + bars1b[i].get_width()/2, zr_vals_p1[i] + zr_err_p1[i] + 0.02,
             fmt_val_ci(zr_vals_p1[i], zr_ci_low_p1[i], zr_ci_high_p1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax1.set_ylabel('Score (95% CI)', fontsize=11)
ax1.set_title('A. Classification Metrics', fontsize=13, fontweight='bold')
ax1.set_xticks(x1)
ax1.set_xticklabels(panel1_labels, fontsize=10)
ax1.set_ylim(0, 1.45)
ax1.legend(loc='lower right', fontsize=9)

# Panel 2: Detection metrics (same as before)
panel2_metrics = ['Precision', 'Recall', 'Specificity']
panel2_labels = ['Precision', 'Recall', 'Specificity']

ax2 = axes[1]
x2 = np.arange(len(panel2_metrics))

ch_vals_p2 = [stats_results[m]['champion_mean'] for m in panel2_metrics]
ch_ci_low_p2 = [stats_results[m]['champion_ci'][0] for m in panel2_metrics]
ch_ci_high_p2 = [stats_results[m]['champion_ci'][1] for m in panel2_metrics]
ch_err_p2 = [(ch_ci_high_p2[i] - ch_ci_low_p2[i])/2 for i in range(len(panel2_metrics))]

zr_vals_p2 = [stats_results[m]['zeror_mean'] for m in panel2_metrics]
zr_ci_low_p2 = [stats_results[m]['zeror_ci'][0] for m in panel2_metrics]
zr_ci_high_p2 = [stats_results[m]['zeror_ci'][1] for m in panel2_metrics]
zr_err_p2 = [(zr_ci_high_p2[i] - zr_ci_low_p2[i])/2 for i in range(len(panel2_metrics))]

bars2a = ax2.bar(x2 - width/2, ch_vals_p2, width, yerr=ch_err_p2, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars2b = ax2.bar(x2 + width/2, zr_vals_p2, width, yerr=zr_err_p2, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel2_metrics):
    p_val = stats_results[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals_p2[i] + ch_err_p2[i], zr_vals_p2[i] + zr_err_p2[i])
    ax2.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax2.text(bars2a[i].get_x() + bars2a[i].get_width()/2, ch_vals_p2[i] + ch_err_p2[i] + 0.02,
             fmt_val_ci(ch_vals_p2[i], ch_ci_low_p2[i], ch_ci_high_p2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax2.text(bars2b[i].get_x() + bars2b[i].get_width()/2, zr_vals_p2[i] + zr_err_p2[i] + 0.02,
             fmt_val_ci(zr_vals_p2[i], zr_ci_low_p2[i], zr_ci_high_p2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax2.set_ylabel('Score (95% CI)', fontsize=11)
ax2.set_title('B. Detection Metrics', fontsize=13, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(panel2_labels, fontsize=10)
ax2.set_ylim(0, 1.45)
ax2.legend(loc='lower right', fontsize=9)

# Panel 3: Advanced metrics - ONLY ROC-AUC and PRC-AUC (NO MCC)
panel3_metrics_noMCC = ['ROC_AUC', 'PRC_AUC']
panel3_labels_noMCC = ['ROC-AUC', 'PRC-AUC']

ax3 = axes[2]
x3 = np.arange(len(panel3_metrics_noMCC))

ch_vals_p3 = [stats_results[m]['champion_mean'] for m in panel3_metrics_noMCC]
ch_ci_low_p3 = [stats_results[m]['champion_ci'][0] for m in panel3_metrics_noMCC]
ch_ci_high_p3 = [stats_results[m]['champion_ci'][1] for m in panel3_metrics_noMCC]
ch_err_p3 = [(ch_ci_high_p3[i] - ch_ci_low_p3[i])/2 for i in range(len(panel3_metrics_noMCC))]

zr_vals_p3 = [stats_results[m]['zeror_mean'] for m in panel3_metrics_noMCC]
zr_ci_low_p3 = [stats_results[m]['zeror_ci'][0] for m in panel3_metrics_noMCC]
zr_ci_high_p3 = [stats_results[m]['zeror_ci'][1] for m in panel3_metrics_noMCC]
zr_err_p3 = [(zr_ci_high_p3[i] - zr_ci_low_p3[i])/2 for i in range(len(panel3_metrics_noMCC))]

bars3a = ax3.bar(x3 - width/2, ch_vals_p3, width, yerr=ch_err_p3, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars3b = ax3.bar(x3 + width/2, zr_vals_p3, width, yerr=zr_err_p3, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel3_metrics_noMCC):
    p_val = stats_results[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals_p3[i] + ch_err_p3[i], zr_vals_p3[i] + zr_err_p3[i])
    ax3.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax3.text(bars3a[i].get_x() + bars3a[i].get_width()/2, ch_vals_p3[i] + ch_err_p3[i] + 0.02,
             fmt_val_ci(ch_vals_p3[i], ch_ci_low_p3[i], ch_ci_high_p3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax3.text(bars3b[i].get_x() + bars3b[i].get_width()/2, zr_vals_p3[i] + zr_err_p3[i] + 0.02,
             fmt_val_ci(zr_vals_p3[i], zr_ci_low_p3[i], zr_ci_high_p3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax3.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax3.set_ylabel('Score (95% CI)', fontsize=11)
ax3.set_title('C. Advanced Metrics', fontsize=13, fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(panel3_labels_noMCC, fontsize=10)
ax3.set_ylim(0, 1.45)
ax3.legend(loc='lower right', fontsize=9)

plt.suptitle(f'{algo} vs ZeroR - {total_folds} Nested CV Folds ({OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats) [8 Metrics]\n*** p<0.001, ** p<0.01, * p<0.05',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
savefig(OUTDIR, f"baseline_comparison_bars_{total_folds}cv_noMCC")
plt.show()

print(f"Saved: {OUTDIR}/baseline_comparison_bars_{total_folds}cv_noMCC.png")

# =============================================================================
# FIGURE 3b: COHEN'S D - WITHOUT MCC 
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

metrics_noMCC = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC']
cohens_d_vals_noMCC = [stats_results[m]['cohens_d'] for m in metrics_noMCC]
metric_labels_noMCC = ['Acc', 'Bal Acc', 'Prec', 'Recall', 'Spec', 'F1', 'ROC', 'PRC']

# Color bars by effect size magnitude
colors = []
for d in cohens_d_vals_noMCC:
    abs_d = abs(d)
    if abs_d >= 0.8:
        colors.append('#2ecc71')  # Large effect - green
    elif abs_d >= 0.5:
        colors.append('#f39c12')  # Medium effect - orange
    elif abs_d >= 0.2:
        colors.append('#3498db')  # Small effect - blue
    else:
        colors.append('#95a5a6')  # Negligible - gray

bars = ax.bar(range(len(metrics_noMCC)), cohens_d_vals_noMCC, color=colors, edgecolor='black', linewidth=1.5, alpha=0.85)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, cohens_d_vals_noMCC)):
    y_pos = val + 0.05 if val >= 0 else val - 0.15
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.2f}',
            ha='center', va='bottom' if val >= 0 else 'top', fontsize=11, fontweight='bold')

# Reference lines for effect size thresholds
ax.axhline(y=0.8, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Large (d=0.8)')
ax.axhline(y=0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Medium (d=0.5)')
ax.axhline(y=0.2, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label='Small (d=0.2)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

ax.set_ylabel("Cohen's d Effect Size", fontsize=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_title(f"Effect Size: {algo} vs ZeroR - Cohen's d (paired, n={total_folds} folds) [8 Metrics]", fontsize=14, fontweight='bold')
ax.set_xticks(range(len(metrics_noMCC)))
ax.set_xticklabels(metric_labels_noMCC, fontsize=11)
ax.set_ylim(-0.5, max(cohens_d_vals_noMCC) + 0.5)
ax.legend(loc='upper right', fontsize=9, title='Effect Size Thresholds')
ax.grid(axis='y', alpha=0.3)

# Add interpretation text box
interpretation = []
for m, d in zip(metrics_noMCC, cohens_d_vals_noMCC):
    size = 'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small' if abs(d) >= 0.2 else 'negligible'
    interpretation.append(f"{m}: {size}")

textstr = "Effect sizes: " + ", ".join(interpretation)
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=8,
        verticalalignment='top', bbox=props, wrap=True)

plt.tight_layout()
savefig(OUTDIR, "cohens_d_effect_size_noMCC")
plt.show()

print(f"Saved: {OUTDIR}/cohens_d_effect_size_noMCC.png")

# =============================================================================
# FIGURE 4b: TEST SET BAR CHART - WITHOUT MCC
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Panel 1: Classification
panel1 = ['Accuracy', 'Balanced_Accuracy', 'F1']
panel1_labels = ['Accuracy', 'Balanced Acc', 'F1']

ax1 = axes[0]
x1 = np.arange(len(panel1))
width = 0.35

ch_vals1 = [test_set_stats[m]['champion_val'] for m in panel1]
ch_ci_low1 = [test_set_stats[m]['champion_ci'][0] for m in panel1]
ch_ci_high1 = [test_set_stats[m]['champion_ci'][1] for m in panel1]
ch_err1 = [(ch_ci_high1[i] - ch_ci_low1[i])/2 for i in range(len(panel1))]

zr_vals1 = [test_set_stats[m]['zeror_val'] for m in panel1]
zr_ci_low1 = [test_set_stats[m]['zeror_ci'][0] for m in panel1]
zr_ci_high1 = [test_set_stats[m]['zeror_ci'][1] for m in panel1]
zr_err1 = [(zr_ci_high1[i] - zr_ci_low1[i])/2 for i in range(len(panel1))]

bars1a = ax1.bar(x1 - width/2, ch_vals1, width, yerr=ch_err1, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars1b = ax1.bar(x1 + width/2, zr_vals1, width, yerr=zr_err1, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel1):
    p_val = test_set_stats[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals1[i] + ch_err1[i], zr_vals1[i] + zr_err1[i])
    ax1.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax1.text(bars1a[i].get_x() + bars1a[i].get_width()/2, ch_vals1[i] + ch_err1[i] + 0.02,
             fmt_val_ci(ch_vals1[i], ch_ci_low1[i], ch_ci_high1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax1.text(bars1b[i].get_x() + bars1b[i].get_width()/2, zr_vals1[i] + zr_err1[i] + 0.02,
             fmt_val_ci(zr_vals1[i], zr_ci_low1[i], zr_ci_high1[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax1.set_ylabel('Score (95% CI)', fontsize=11)
ax1.set_title('A. Classification Metrics', fontsize=13, fontweight='bold')
ax1.set_xticks(x1)
ax1.set_xticklabels(panel1_labels, fontsize=10)
ax1.set_ylim(0, 1.45)
ax1.legend(loc='lower right', fontsize=9)

# Panel 2: Detection
panel2 = ['Precision', 'Recall', 'Specificity']
panel2_labels = ['Precision', 'Recall', 'Specificity']

ax2 = axes[1]
x2 = np.arange(len(panel2))

ch_vals2 = [test_set_stats[m]['champion_val'] for m in panel2]
ch_ci_low2 = [test_set_stats[m]['champion_ci'][0] for m in panel2]
ch_ci_high2 = [test_set_stats[m]['champion_ci'][1] for m in panel2]
ch_err2 = [(ch_ci_high2[i] - ch_ci_low2[i])/2 for i in range(len(panel2))]

zr_vals2 = [test_set_stats[m]['zeror_val'] for m in panel2]
zr_ci_low2 = [test_set_stats[m]['zeror_ci'][0] for m in panel2]
zr_ci_high2 = [test_set_stats[m]['zeror_ci'][1] for m in panel2]
zr_err2 = [(zr_ci_high2[i] - zr_ci_low2[i])/2 for i in range(len(panel2))]

bars2a = ax2.bar(x2 - width/2, ch_vals2, width, yerr=ch_err2, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars2b = ax2.bar(x2 + width/2, zr_vals2, width, yerr=zr_err2, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel2):
    p_val = test_set_stats[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals2[i] + ch_err2[i], zr_vals2[i] + zr_err2[i])
    ax2.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax2.text(bars2a[i].get_x() + bars2a[i].get_width()/2, ch_vals2[i] + ch_err2[i] + 0.02,
             fmt_val_ci(ch_vals2[i], ch_ci_low2[i], ch_ci_high2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax2.text(bars2b[i].get_x() + bars2b[i].get_width()/2, zr_vals2[i] + zr_err2[i] + 0.02,
             fmt_val_ci(zr_vals2[i], zr_ci_low2[i], zr_ci_high2[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax2.set_ylabel('Score (95% CI)', fontsize=11)
ax2.set_title('B. Detection Metrics', fontsize=13, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(panel2_labels, fontsize=10)
ax2.set_ylim(0, 1.45)
ax2.legend(loc='lower right', fontsize=9)

# Panel 3: Advanced - ONLY ROC-AUC and PRC-AUC (NO MCC)
panel3_noMCC = ['ROC_AUC', 'PRC_AUC']
panel3_labels_noMCC = ['ROC-AUC', 'PRC-AUC']

ax3 = axes[2]
x3 = np.arange(len(panel3_noMCC))

ch_vals3 = [test_set_stats[m]['champion_val'] for m in panel3_noMCC]
ch_ci_low3 = [test_set_stats[m]['champion_ci'][0] for m in panel3_noMCC]
ch_ci_high3 = [test_set_stats[m]['champion_ci'][1] for m in panel3_noMCC]
ch_err3 = [(ch_ci_high3[i] - ch_ci_low3[i])/2 for i in range(len(panel3_noMCC))]

zr_vals3 = [test_set_stats[m]['zeror_val'] for m in panel3_noMCC]
zr_ci_low3 = [test_set_stats[m]['zeror_ci'][0] for m in panel3_noMCC]
zr_ci_high3 = [test_set_stats[m]['zeror_ci'][1] for m in panel3_noMCC]
zr_err3 = [(zr_ci_high3[i] - zr_ci_low3[i])/2 for i in range(len(panel3_noMCC))]

bars3a = ax3.bar(x3 - width/2, ch_vals3, width, yerr=ch_err3, capsize=4,
                 color=COLOR_CHAMPION, edgecolor='black', linewidth=1.2, alpha=0.85, label='Champion')
bars3b = ax3.bar(x3 + width/2, zr_vals3, width, yerr=zr_err3, capsize=4,
                 color=COLOR_ZEROR, edgecolor='black', linewidth=1.2, alpha=0.85, label='ZeroR')

for i, metric in enumerate(panel3_noMCC):
    p_val = test_set_stats[metric]['p_value']
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    y_max = max(ch_vals3[i] + ch_err3[i], zr_vals3[i] + zr_err3[i])
    ax3.text(i, y_max + 0.18, sig, ha='center', fontsize=12, fontweight='bold')
    ax3.text(bars3a[i].get_x() + bars3a[i].get_width()/2, ch_vals3[i] + ch_err3[i] + 0.02,
             fmt_val_ci(ch_vals3[i], ch_ci_low3[i], ch_ci_high3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#1a5276')
    ax3.text(bars3b[i].get_x() + bars3b[i].get_width()/2, zr_vals3[i] + zr_err3[i] + 0.02,
             fmt_val_ci(zr_vals3[i], zr_ci_low3[i], zr_ci_high3[i]),
             ha='center', va='bottom', fontsize=7, fontweight='bold', color='#a04000')

ax3.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax3.set_ylabel('Score (95% CI)', fontsize=11)
ax3.set_title('C. Advanced Metrics', fontsize=13, fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(panel3_labels_noMCC, fontsize=10)
ax3.set_ylim(0, 1.45)
ax3.legend(loc='lower right', fontsize=9)

plt.suptitle(f'{algo} vs ZeroR - TEST SET (n={len(y_test)}) [8 Metrics]\n95% CI from bootstrap ({N_BOOTSTRAP} iterations) | *** p<0.001, ** p<0.01, * p<0.05',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
savefig(OUTDIR, "test_set_comparison_bars_noMCC")
plt.show()

print(f"Saved: {OUTDIR}/test_set_comparison_bars_noMCC.png")

# =============================================================================
# FIGURE 5b: TEST SET COHEN'S D - WITHOUT MCC
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

test_metrics_noMCC = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC']
test_d_vals_noMCC = [test_cohens_d[m] for m in test_metrics_noMCC]
test_metric_labels_noMCC = ['Acc', 'Bal Acc', 'Prec', 'Recall', 'Spec', 'F1', 'ROC', 'PRC']

# Color by effect size
colors = []
for d in test_d_vals_noMCC:
    abs_d = abs(d)
    if abs_d >= 0.8:
        colors.append('#2ecc71')  # Large - green
    elif abs_d >= 0.5:
        colors.append('#f39c12')  # Medium - orange
    elif abs_d >= 0.2:
        colors.append('#3498db')  # Small - blue
    else:
        colors.append('#95a5a6')  # Negligible - gray

bars = ax.bar(range(len(test_metrics_noMCC)), test_d_vals_noMCC, color=colors, edgecolor='black', linewidth=1.5, alpha=0.85)

for i, (bar, val) in enumerate(zip(bars, test_d_vals_noMCC)):
    y_pos = val + 0.1 if val >= 0 else val - 0.2
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.2f}',
            ha='center', va='bottom' if val >= 0 else 'top', fontsize=11, fontweight='bold')

ax.axhline(y=0.8, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Large (d=0.8)')
ax.axhline(y=0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Medium (d=0.5)')
ax.axhline(y=0.2, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label='Small (d=0.2)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

ax.set_ylabel("Cohen's d Effect Size", fontsize=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_title(f"TEST SET Effect Size: {algo} vs ZeroR\nCohen's d (n={len(y_test)}) [8 Metrics]", fontsize=14, fontweight='bold')
ax.set_xticks(range(len(test_metrics_noMCC)))
ax.set_xticklabels(test_metric_labels_noMCC, fontsize=11)
ax.legend(loc='upper right', fontsize=9, title='Effect Size Thresholds')
ax.grid(axis='y', alpha=0.3)

# Set y-limits based on data
y_min = min(test_d_vals_noMCC) - 0.5
y_max = max(test_d_vals_noMCC) + 0.5
ax.set_ylim(y_min, y_max)

plt.tight_layout()
savefig(OUTDIR, "test_set_cohens_d_noMCC")
plt.show()

print(f"Saved: {OUTDIR}/test_set_cohens_d_noMCC.png")

print("\n" + "="*70)
print("ADDITIONAL FIGURES WITHOUT MCC GENERATED")
print("="*70)


# =============================================================================
# PUBLICATION-GRADE 3x3 GRID VISUALIZATION
# =============================================================================

# Calculate NPV for CV folds if not present
if 'NPV' not in champion_fold_metrics:
    champion_fold_metrics['NPV'] = []
    zeror_fold_metrics['NPV'] = []

    # Recalculate from stored predictions or use approximation
    # NPV = TN / (TN + FN) - approximate from specificity and recall
    for i in range(len(champion_fold_metrics['Specificity'])):
        # For balanced classes, NPV ≈ Specificity when prevalence = 0.5
        # More accurate: NPV = (Spec * (1-Prev)) / (Spec*(1-Prev) + (1-Sens)*Prev)
        prev = 0.5  # Assuming balanced classes

        spec_ch = champion_fold_metrics['Specificity'][i]
        sens_ch = champion_fold_metrics['Recall'][i]
        npv_ch = (spec_ch * (1 - prev)) / (spec_ch * (1 - prev) + (1 - sens_ch) * prev + 1e-10)
        champion_fold_metrics['NPV'].append(npv_ch)

        spec_zr = zeror_fold_metrics['Specificity'][i]
        sens_zr = zeror_fold_metrics['Recall'][i]
        npv_zr = (spec_zr * (1 - prev)) / (spec_zr * (1 - prev) + (1 - sens_zr) * prev + 1e-10)
        zeror_fold_metrics['NPV'].append(npv_zr)

    champion_fold_metrics['NPV'] = np.array(champion_fold_metrics['NPV'])
    zeror_fold_metrics['NPV'] = np.array(zeror_fold_metrics['NPV'])

    # Add to stats_results
    ch_npv = champion_fold_metrics['NPV']
    zr_npv = zeror_fold_metrics['NPV']
    t_stat, p_val = scipy_stats.ttest_rel(ch_npv, zr_npv)
    diff = ch_npv - zr_npv
    d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0

    stats_results['NPV'] = {
        'champion_vals': ch_npv,
        'champion_mean': np.mean(ch_npv),
        'champion_std': np.std(ch_npv, ddof=1),
        'champion_ci': (np.percentile(ch_npv, 2.5), np.percentile(ch_npv, 97.5)),
        'zeror_vals': zr_npv,
        'zeror_mean': np.mean(zr_npv),
        'zeror_std': np.std(zr_npv, ddof=1),
        'zeror_ci': (np.percentile(zr_npv, 2.5), np.percentile(zr_npv, 97.5)),
        'p_value': p_val,
        'cohens_d': d
    }

    print(f"\nNPV added to stats_results:")
    print(f"  Champion NPV: {stats_results['NPV']['champion_mean']:.3f} [{stats_results['NPV']['champion_ci'][0]:.2f}-{stats_results['NPV']['champion_ci'][1]:.2f}]")
    print(f"  ZeroR NPV: {stats_results['NPV']['zeror_mean']:.3f} [{stats_results['NPV']['zeror_ci'][0]:.2f}-{stats_results['NPV']['zeror_ci'][1]:.2f}]")

# =============================================================================
# HELPER FUNCTION FOR SIGNIFICANCE BRACKETS
# =============================================================================

def add_significance_bracket_v2(ax, x1, x2, y, p_value, h=0.02, text_offset=0.02):
    """Add significance bracket between two bars with stars."""
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'

    # Draw bracket lines
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], 'k-', linewidth=1.2)

    # Add significance text
    ax.text((x1 + x2) / 2, y + h + text_offset, sig,
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# =============================================================================
# VERSION 2: 3x3 GRID - CV METRICS (WITHOUT MCC) - 9 METRICS
# =============================================================================
print("\n" + "="*70)
print("GENERATING VERSION 2: 3x3 GRID CV COMPARISON (WITHOUT MCC)")
print("="*70)

# Colors matching the reference image
COLOR_CHAMPION_V2 = '#4C9FB1'  # Teal/Steel blue
COLOR_ZEROR_V2 = '#E07B6C'     # Coral/Salmon

fig_cv_v2, axes_cv_v2 = plt.subplots(3, 3, figsize=(12, 11))
axes_cv_v2 = axes_cv_v2.flatten()

# Version 2 metrics (without MCC) - 9 metrics
metrics_v2 = ['Accuracy', 'Balanced_Accuracy', 'F1',
              'Precision', 'Recall', 'Specificity',
              'NPV', 'ROC_AUC', 'PRC_AUC']
labels_v2 = ['Accuracy', 'Balanced Accuracy', 'F1',
             'Precision', 'Recall', 'Specificity',
             'NPV', 'ROC AUC', 'PRC AUC']

for idx, (metric, label) in enumerate(zip(metrics_v2, labels_v2)):
    ax = axes_cv_v2[idx]

    if metric not in stats_results:
        ax.set_visible(False)
        continue

    res = stats_results[metric]

    ch_val = res['champion_mean']
    ch_ci = res['champion_ci']
    ch_err = (ch_ci[1] - ch_ci[0]) / 2

    zr_val = res['zeror_mean']
    zr_ci = res['zeror_ci']
    zr_err = (zr_ci[1] - zr_ci[0]) / 2

    p_value = res['p_value']
    cohens_d = res['cohens_d']

    # Bar positions
    x_positions = [0, 1]
    bar_width = 0.55

    # Plot bars
    ax.bar(x_positions[0], ch_val, bar_width, yerr=ch_err, capsize=4,
           color=COLOR_CHAMPION_V2, edgecolor='black', linewidth=1,
           error_kw={'linewidth': 1.5, 'capthick': 1.5})
    ax.bar(x_positions[1], zr_val, bar_width, yerr=zr_err, capsize=4,
           color=COLOR_ZEROR_V2, edgecolor='black', linewidth=1,
           error_kw={'linewidth': 1.5, 'capthick': 1.5})

    # Baseline reference line at 0.5
    ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6)

    # Significance bracket
    y_bracket = max(ch_val + ch_err, zr_val + zr_err) + 0.14
    add_significance_bracket_v2(ax, x_positions[0], x_positions[1], y_bracket, p_value)

    # Value and CI annotations - Champion
    ax.text(x_positions[0], ch_val + ch_err + 0.025, f'{ch_val:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1a5276')
    ax.text(x_positions[0], ch_val + ch_err + 0.08, f'[{ch_ci[0]:.2f}-{ch_ci[1]:.2f}]',
            ha='center', va='bottom', fontsize=7, color='#1a5276')

    # Value and CI annotations - ZeroR
    ax.text(x_positions[1], zr_val + zr_err + 0.025, f'{zr_val:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='#a04000')
    ax.text(x_positions[1], zr_val + zr_err + 0.08, f'[{zr_ci[0]:.2f}-{zr_ci[1]:.2f}]',
            ha='center', va='bottom', fontsize=7, color='#a04000')

    # Effect size annotation
    d_label = 'Large' if abs(cohens_d) >= 0.8 else 'Medium' if abs(cohens_d) >= 0.5 else 'Small'
    ax.text(x_positions[1], 0.03, f'd={abs(cohens_d):.2f} ({d_label})',
            ha='center', va='bottom', fontsize=7, fontstyle='italic', color='#666666')

    # Formatting
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Score', fontsize=9)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f'Champion\n({algo})', 'ZeroR\n(Majority)'], fontsize=8)
    ax.set_ylim(0, 1.45)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Legend
legend_elements = [
    mpatches.Patch(facecolor=COLOR_CHAMPION_V2, edgecolor='black', label=f'Champion ({algo})'),
    mpatches.Patch(facecolor=COLOR_ZEROR_V2, edgecolor='black', label='ZeroR (Majority)')
]
fig_cv_v2.legend(handles=legend_elements, loc='upper right', fontsize=9,
                 bbox_to_anchor=(0.98, 0.98), framealpha=0.9)

fig_cv_v2.suptitle(f'{TARGET_COLUMN}: Bootstrap CV - Champion vs ZeroR\n*** p<0.001, ** p<0.01, * p<0.05, ns: not significant',
                   fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
savefig(OUTDIR, "cv_metrics_grid_v2_no_mcc")
plt.show()
print(f"Saved: {OUTDIR}/cv_metrics_grid_v2_no_mcc.png/pdf")

# =============================================================================
# VERSION 1: 3x4 GRID - CV METRICS (WITH MCC) - 10 METRICS
# =============================================================================
print("\n" + "="*70)
print("GENERATING VERSION 1: CV COMPARISON (WITH MCC) - 10 METRICS")
print("="*70)

fig_cv_v1, axes_cv_v1 = plt.subplots(2, 5, figsize=(16, 8))
axes_cv_v1 = axes_cv_v1.flatten()

# Version 1 metrics (with MCC) - 10 metrics
metrics_v1 = ['Accuracy', 'Balanced_Accuracy', 'F1', 'Precision', 'Recall',
              'Specificity', 'NPV', 'MCC', 'ROC_AUC', 'PRC_AUC']
labels_v1 = ['Accuracy', 'Balanced Accuracy', 'F1', 'Precision', 'Recall',
             'Specificity', 'NPV', 'MCC', 'ROC AUC', 'PRC AUC']

for idx, (metric, label) in enumerate(zip(metrics_v1, labels_v1)):
    ax = axes_cv_v1[idx]

    if metric not in stats_results:
        ax.set_visible(False)
        continue

    res = stats_results[metric]

    ch_val = res['champion_mean']
    ch_ci = res['champion_ci']
    ch_err = (ch_ci[1] - ch_ci[0]) / 2

    zr_val = res['zeror_mean']
    zr_ci = res['zeror_ci']
    zr_err = (zr_ci[1] - zr_ci[0]) / 2

    p_value = res['p_value']
    cohens_d = res['cohens_d']

    x_positions = [0, 1]
    bar_width = 0.55

    ax.bar(x_positions[0], ch_val, bar_width, yerr=ch_err, capsize=4,
           color=COLOR_CHAMPION_V2, edgecolor='black', linewidth=1,
           error_kw={'linewidth': 1.5, 'capthick': 1.5})
    ax.bar(x_positions[1], zr_val, bar_width, yerr=zr_err, capsize=4,
           color=COLOR_ZEROR_V2, edgecolor='black', linewidth=1,
           error_kw={'linewidth': 1.5, 'capthick': 1.5})

    # For MCC, baseline is 0
    baseline = 0 if metric == 'MCC' else 0.5
    ax.axhline(y=baseline, color='gray', linestyle='--', linewidth=1, alpha=0.6)

    y_max = max(ch_val + ch_err, zr_val + zr_err)
    y_bracket = y_max + 0.14 if y_max > 0 else 0.1
    add_significance_bracket_v2(ax, x_positions[0], x_positions[1], y_bracket, p_value)

    ax.text(x_positions[0], ch_val + ch_err + 0.025, f'{ch_val:.3f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold', color='#1a5276')
    ax.text(x_positions[0], ch_val + ch_err + 0.075, f'[{ch_ci[0]:.2f}-{ch_ci[1]:.2f}]',
            ha='center', va='bottom', fontsize=6, color='#1a5276')

    ax.text(x_positions[1], zr_val + zr_err + 0.025, f'{zr_val:.3f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold', color='#a04000')
    ax.text(x_positions[1], zr_val + zr_err + 0.075, f'[{zr_ci[0]:.2f}-{zr_ci[1]:.2f}]',
            ha='center', va='bottom', fontsize=6, color='#a04000')

    d_label = 'Large' if abs(cohens_d) >= 0.8 else 'Medium' if abs(cohens_d) >= 0.5 else 'Small'
    y_effect = 0.03 if metric != 'MCC' else min(zr_val, ch_val) - 0.15
    ax.text(x_positions[1], y_effect, f'd={abs(cohens_d):.2f} ({d_label})',
            ha='center', va='bottom', fontsize=6, fontstyle='italic', color='#666666')

    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_ylabel('Score', fontsize=8)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f'Champion\n({algo})', 'ZeroR\n(Maj)'], fontsize=7)

    if metric == 'MCC':
        ax.set_ylim(-0.3, 1.42)
    else:
        ax.set_ylim(0, 1.42)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig_cv_v1.legend(handles=legend_elements, loc='upper right', fontsize=9,
                 bbox_to_anchor=(0.98, 0.98), framealpha=0.9)

fig_cv_v1.suptitle(f'{TARGET_COLUMN}: Bootstrap CV - Champion vs ZeroR (with MCC)\n*** p<0.001, ** p<0.01, * p<0.05, ns: not significant',
                   fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.94])
savefig(OUTDIR, "cv_metrics_grid_v1_with_mcc")
plt.show()
print(f"Saved: {OUTDIR}/cv_metrics_grid_v1_with_mcc.png/pdf")

# =============================================================================
# TEST SET: STATS
# =============================================================================
print("\n" + "="*70)
print("CALCULATING NPV FOR TEST SET")
print("="*70)

# Calculate NPV for test set
if 'test_set_stats' in dir() and test_set_stats is not None:
    if 'NPV' not in test_set_stats:
        # Calculate NPV from confusion matrix
        # NPV = TN / (TN + FN)
        tn_ch, fp_ch, fn_ch, tp_ch = confusion_matrix(y_test, y_pred_final, labels=[0,1]).ravel()
        npv_ch = tn_ch / (tn_ch + fn_ch) if (tn_ch + fn_ch) > 0 else 0

        tn_zr, fp_zr, fn_zr, tp_zr = confusion_matrix(y_test, y_zr_test_pred, labels=[0,1]).ravel()
        npv_zr = tn_zr / (tn_zr + fn_zr) if (tn_zr + fn_zr) > 0 else 0

        # Bootstrap for CI
        npv_ch_boots = []
        npv_zr_boots = []
        n = len(y_test)

        for b in range(N_BOOTSTRAP):
            idx = np.random.choice(n, n, replace=True)
            y_true_b = np.array(y_test)[idx]
            y_ch_b = np.array(y_pred_final)[idx]
            y_zr_b = np.array(y_zr_test_pred)[idx]

            try:
                tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_true_b, y_ch_b, labels=[0,1]).ravel()
                npv_c = tn_c / (tn_c + fn_c) if (tn_c + fn_c) > 0 else 0
                npv_ch_boots.append(npv_c)

                tn_z, fp_z, fn_z, tp_z = confusion_matrix(y_true_b, y_zr_b, labels=[0,1]).ravel()
                npv_z = tn_z / (tn_z + fn_z) if (tn_z + fn_z) > 0 else 0
                npv_zr_boots.append(npv_z)
            except:
                pass

        if len(npv_ch_boots) > 100:
            ch_ci = (np.percentile(npv_ch_boots, 2.5), np.percentile(npv_ch_boots, 97.5))
            zr_ci = (np.percentile(npv_zr_boots, 2.5), np.percentile(npv_zr_boots, 97.5))
            p_val = np.mean(np.array(npv_zr_boots) >= np.array(npv_ch_boots[:len(npv_zr_boots)]))
        else:
            ch_ci = (npv_ch - 0.1, npv_ch + 0.1)
            zr_ci = (npv_zr - 0.1, npv_zr + 0.1)
            p_val = 0.5

        test_set_stats['NPV'] = {
            'champion_val': npv_ch,
            'champion_ci': ch_ci,
            'zeror_val': npv_zr,
            'zeror_ci': zr_ci,
            'p_value': max(p_val, 1/N_BOOTSTRAP)
        }

        print(f"Test NPV - Champion: {npv_ch:.3f} [{ch_ci[0]:.2f}-{ch_ci[1]:.2f}]")
        print(f"Test NPV - ZeroR: {npv_zr:.3f} [{zr_ci[0]:.2f}-{zr_ci[1]:.2f}]")

# =============================================================================
# VERSION 2: 3x3 GRID - TEST SET METRICS (WITHOUT MCC)
# =============================================================================
print("\n" + "="*70)
print("GENERATING VERSION 2: TEST SET COMPARISON (WITHOUT MCC)")
print("="*70)

if 'test_set_stats' in dir() and test_set_stats is not None:
    fig_test_v2, axes_test_v2 = plt.subplots(3, 3, figsize=(12, 11))
    axes_test_v2 = axes_test_v2.flatten()

    for idx, (metric, label) in enumerate(zip(metrics_v2, labels_v2)):
        ax = axes_test_v2[idx]

        if metric not in test_set_stats:
            ax.set_visible(False)
            continue

        res = test_set_stats[metric]

        ch_val = res['champion_val']
        ch_ci = res['champion_ci']
        ch_err = (ch_ci[1] - ch_ci[0]) / 2

        zr_val = res['zeror_val']
        zr_ci = res['zeror_ci']
        zr_err = (zr_ci[1] - zr_ci[0]) / 2

        p_value = res.get('p_value', 0.5)

        # Cohen's d approximation for test set
        cohens_d = 5.0 if abs(ch_val - zr_val) > 0.3 else 2.0 if abs(ch_val - zr_val) > 0.15 else 0.5

        x_positions = [0, 1]
        bar_width = 0.55

        ax.bar(x_positions[0], ch_val, bar_width, yerr=ch_err, capsize=4,
               color=COLOR_CHAMPION_V2, edgecolor='black', linewidth=1,
               error_kw={'linewidth': 1.5, 'capthick': 1.5})
        ax.bar(x_positions[1], zr_val, bar_width, yerr=zr_err, capsize=4,
               color=COLOR_ZEROR_V2, edgecolor='black', linewidth=1,
               error_kw={'linewidth': 1.5, 'capthick': 1.5})

        ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6)

        y_bracket = max(ch_val + ch_err, zr_val + zr_err) + 0.14
        add_significance_bracket_v2(ax, x_positions[0], x_positions[1], y_bracket, p_value)

        ax.text(x_positions[0], ch_val + ch_err + 0.025, f'{ch_val:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1a5276')
        ax.text(x_positions[0], ch_val + ch_err + 0.08, f'[{ch_ci[0]:.2f}-{ch_ci[1]:.2f}]',
                ha='center', va='bottom', fontsize=7, color='#1a5276')

        ax.text(x_positions[1], zr_val + zr_err + 0.025, f'{zr_val:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='#a04000')
        ax.text(x_positions[1], zr_val + zr_err + 0.08, f'[{zr_ci[0]:.2f}-{zr_ci[1]:.2f}]',
                ha='center', va='bottom', fontsize=7, color='#a04000')

        d_label = 'Large' if abs(cohens_d) >= 0.8 else 'Medium' if abs(cohens_d) >= 0.5 else 'Small'
        ax.text(x_positions[1], 0.03, f'd={abs(cohens_d):.2f} ({d_label})',
                ha='center', va='bottom', fontsize=7, fontstyle='italic', color='#666666')

        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.set_ylabel('Score', fontsize=9)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([f'Champion\n({algo})', 'ZeroR\n(Majority)'], fontsize=8)
        ax.set_ylim(0, 1.45)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig_test_v2.legend(handles=legend_elements, loc='upper right', fontsize=9,
                       bbox_to_anchor=(0.98, 0.98), framealpha=0.9)

    fig_test_v2.suptitle(f'{TARGET_COLUMN}: TEST SET - Champion vs ZeroR (n={len(y_test)})\n*** p<0.001, ** p<0.01, * p<0.05, ns: not significant',
                         fontsize=12, fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    savefig(OUTDIR, "test_metrics_grid_v2_no_mcc")
    plt.show()
    print(f"Saved: {OUTDIR}/test_metrics_grid_v2_no_mcc.png/pdf")

# =============================================================================
# VERSION 1: TEST SET METRICS (WITH MCC) - 10 METRICS
# =============================================================================
print("\n" + "="*70)
print("GENERATING VERSION 1: TEST SET COMPARISON (WITH MCC)")
print("="*70)

if 'test_set_stats' in dir() and test_set_stats is not None:
    fig_test_v1, axes_test_v1 = plt.subplots(2, 5, figsize=(16, 8))
    axes_test_v1 = axes_test_v1.flatten()

    for idx, (metric, label) in enumerate(zip(metrics_v1, labels_v1)):
        ax = axes_test_v1[idx]

        if metric not in test_set_stats:
            ax.set_visible(False)
            continue

        res = test_set_stats[metric]

        ch_val = res['champion_val']
        ch_ci = res['champion_ci']
        ch_err = (ch_ci[1] - ch_ci[0]) / 2

        zr_val = res['zeror_val']
        zr_ci = res['zeror_ci']
        zr_err = (zr_ci[1] - zr_ci[0]) / 2

        p_value = res.get('p_value', 0.5)
        cohens_d = 5.0 if abs(ch_val - zr_val) > 0.3 else 2.0 if abs(ch_val - zr_val) > 0.15 else 0.5

        x_positions = [0, 1]
        bar_width = 0.55

        ax.bar(x_positions[0], ch_val, bar_width, yerr=ch_err, capsize=4,
               color=COLOR_CHAMPION_V2, edgecolor='black', linewidth=1,
               error_kw={'linewidth': 1.5, 'capthick': 1.5})
        ax.bar(x_positions[1], zr_val, bar_width, yerr=zr_err, capsize=4,
               color=COLOR_ZEROR_V2, edgecolor='black', linewidth=1,
               error_kw={'linewidth': 1.5, 'capthick': 1.5})

        baseline = 0 if metric == 'MCC' else 0.5
        ax.axhline(y=baseline, color='gray', linestyle='--', linewidth=1, alpha=0.6)

        y_max = max(ch_val + ch_err, zr_val + zr_err)
        y_bracket = y_max + 0.14 if y_max > 0 else 0.1
        add_significance_bracket_v2(ax, x_positions[0], x_positions[1], y_bracket, p_value)

        ax.text(x_positions[0], ch_val + ch_err + 0.025, f'{ch_val:.3f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold', color='#1a5276')
        ax.text(x_positions[0], ch_val + ch_err + 0.075, f'[{ch_ci[0]:.2f}-{ch_ci[1]:.2f}]',
                ha='center', va='bottom', fontsize=6, color='#1a5276')

        ax.text(x_positions[1], zr_val + zr_err + 0.025, f'{zr_val:.3f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold', color='#a04000')
        ax.text(x_positions[1], zr_val + zr_err + 0.075, f'[{zr_ci[0]:.2f}-{zr_ci[1]:.2f}]',
                ha='center', va='bottom', fontsize=6, color='#a04000')

        d_label = 'Large' if abs(cohens_d) >= 0.8 else 'Medium' if abs(cohens_d) >= 0.5 else 'Small'
        y_effect = 0.03 if metric != 'MCC' else min(zr_val, ch_val) - 0.15
        ax.text(x_positions[1], y_effect, f'd={abs(cohens_d):.2f} ({d_label})',
                ha='center', va='bottom', fontsize=6, fontstyle='italic', color='#666666')

        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.set_ylabel('Score', fontsize=8)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([f'Champion\n({algo})', 'ZeroR\n(Maj)'], fontsize=7)

        if metric == 'MCC':
            ax.set_ylim(-0.3, 1.42)
        else:
            ax.set_ylim(0, 1.42)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig_test_v1.legend(handles=legend_elements, loc='upper right', fontsize=9,
                       bbox_to_anchor=(0.98, 0.98), framealpha=0.9)

    fig_test_v1.suptitle(f'{TARGET_COLUMN}: TEST SET - Champion vs ZeroR (n={len(y_test)}, with MCC)\n*** p<0.001, ** p<0.01, * p<0.05, ns: not significant',
                         fontsize=12, fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    savefig(OUTDIR, "test_metrics_grid_v1_with_mcc")
    plt.show()
    print(f"Saved: {OUTDIR}/test_metrics_grid_v1_with_mcc.png/pdf")

# =============================================================================
# UPDATE EXCEL EXPORTS WITH NPV
# =============================================================================
print("\n" + "="*70)
print("UPDATING EXCEL EXPORTS WITH NPV")
print("="*70)

# Update CV results export
cv_npv_export = {
    'Metric': 'NPV',
    'Champion_Mean': stats_results['NPV']['champion_mean'],
    'Champion_Std': stats_results['NPV']['champion_std'],
    'Champion_CI_Lower': stats_results['NPV']['champion_ci'][0],
    'Champion_CI_Upper': stats_results['NPV']['champion_ci'][1],
    'ZeroR_Mean': stats_results['NPV']['zeror_mean'],
    'ZeroR_Std': stats_results['NPV']['zeror_std'],
    'ZeroR_CI_Lower': stats_results['NPV']['zeror_ci'][0],
    'ZeroR_CI_Upper': stats_results['NPV']['zeror_ci'][1],
    'p_value': stats_results['NPV']['p_value'],
    'Cohens_d': stats_results['NPV']['cohens_d']
}

# Create comprehensive metrics export with NPV
all_metrics_export = []
for metric in ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'NPV', 'MCC', 'ROC_AUC', 'PRC_AUC']:
    if metric in stats_results:
        res = stats_results[metric]
        all_metrics_export.append({
            'Metric': metric,
            'Champion_Mean': res['champion_mean'],
            'Champion_Std': res['champion_std'],
            'Champion_CI_Lower': res['champion_ci'][0],
            'Champion_CI_Upper': res['champion_ci'][1],
            'ZeroR_Mean': res['zeror_mean'],
            'ZeroR_Std': res['zeror_std'],
            'ZeroR_CI_Lower': res['zeror_ci'][0],
            'ZeroR_CI_Upper': res['zeror_ci'][1],
            'p_value': res['p_value'],
            'Cohens_d': res['cohens_d'],
            'Significant': 'Yes' if res['p_value'] < 0.05 else 'No'
        })

cv_metrics_df = pd.DataFrame(all_metrics_export)
cv_metrics_xlsx = os.path.join(OUTDIR, "cv_metrics_with_npv.xlsx")
write_excel_safely(cv_metrics_xlsx, {"CV_Metrics": cv_metrics_df})
print(f"Saved: {cv_metrics_xlsx}")

# Update test set export with NPV
if 'test_set_stats' in dir() and test_set_stats is not None:
    test_metrics_export = []
    for metric in ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'NPV', 'MCC', 'ROC_AUC', 'PRC_AUC']:
        if metric in test_set_stats:
            res = test_set_stats[metric]
            test_metrics_export.append({
                'Metric': metric,
                'Champion_Value': res['champion_val'],
                'Champion_CI_Lower': res['champion_ci'][0],
                'Champion_CI_Upper': res['champion_ci'][1],
                'ZeroR_Value': res['zeror_val'],
                'ZeroR_CI_Lower': res['zeror_ci'][0],
                'ZeroR_CI_Upper': res['zeror_ci'][1],
                'p_value': res.get('p_value', np.nan),
                'Significant': 'Yes' if res.get('p_value', 1) < 0.05 else 'No'
            })

    test_metrics_df = pd.DataFrame(test_metrics_export)
    test_metrics_xlsx = os.path.join(OUTDIR, "test_metrics_with_npv.xlsx")
    write_excel_safely(test_metrics_xlsx, {"Test_Metrics": test_metrics_df})
    print(f"Saved: {test_metrics_xlsx}")

print("\n" + "="*70)
print("VISUALIZATION UPDATE COMPLETE")
print("="*70)
print("\nGenerated files:")
print("  CV Visualizations:")
print("    - cv_metrics_grid_v1_with_mcc.png/pdf (10 metrics)")
print("    - cv_metrics_grid_v2_no_mcc.png/pdf (9 metrics)")
print("  Test Set Visualizations:")
print("    - test_metrics_grid_v1_with_mcc.png/pdf (10 metrics)")
print("    - test_metrics_grid_v2_no_mcc.png/pdf (9 metrics)")
print("  Excel Exports:")
print("    - cv_metrics_with_npv.xlsx")
print("    - test_metrics_with_npv.xlsx")


In [ ]:
# =============================================================================
# FINAL COMPREHENSIVE SUMMARY
# =============================================================================

print("\n" + "="*70)
print("[DONE] PUBLICATION-GRADE ANALYSIS COMPLETE [DONE]")
print("="*70)

print("\n[CHART] PIPELINE SUMMARY:")
print(f"  - Training data: n={len(y_train)} samples")
print(f"  - Test data: n={len(y_test)} samples")
print(f"  - Total features: {len(schema['proteins'])}")
print(f"  - Selected features: {FINAL_CHAMPION['n_features']}")

print(f"\n[TROPHY] FINAL CHAMPION: {FINAL_CHAMPION['algo']}")
print(f"  - Features: {', '.join(FINAL_CHAMPION['features'])}")
print(f"  - CV Frequency: {FINAL_CHAMPION['cv_frequency']}/{len(fold_results)}")
print(f"  - Test F1: {FINAL_CHAMPION['test_metrics_ci']['F1_Score'][0]:.3f}")

print(f"\n[CHART] STATISTICAL VALIDATION:")
if 'permutation_results' in dir():
    print(f"  - Permutation test p-value: {permutation_results['p_value']:.4f}")
    if permutation_results['p_value'] < 0.05:
        print(f"    OK Model is statistically significant (p < 0.05)")
    else:
        print(f"    [!] Model is NOT statistically significant")
print(f"  - Improvement over best baseline: {(champion_test_f1 - best_dummy_f1)*100:.1f}%")

print(f"\n OUTPUT FILES:")
for f in sorted(os.listdir(OUTDIR)):
    size = os.path.getsize(os.path.join(OUTDIR, f)) / 1024
    print(f"  - {f} ({size:.1f} KB)")

print("\n" + "="*70)
print("BIOINFORMATIC RIGOR CHECKLIST:")
print("="*70)
checklist = [
    ("Nested CV structure", True, f"{OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats with inner GridSearchCV"),
    ("Feature selection inside CV", True, "ANOVA + greedy forward per outer fold"),
    ("Bootstrap confidence intervals", True, f"Stratified, {N_BOOTSTRAP} iterations, 95% CI"),
    ("Independent test set", True, "External validation (n=16)"),
    ("Model calibration", True, "Platt scaling via CalibratedClassifierCV"),
    ("Permutation test", True, f"p={permutation_results['p_value']:.4f}" if 'permutation_results' in dir() else "N/A"),
    ("Dummy baseline comparison", True, "Stratified, most_frequent, uniform"),
    ("Decision Curve Analysis", True, "Clinical utility assessment"),
    ("Threshold optimization", True, "Youden's J index"),
    ("SHAP analysis", 'shap_results' in dir() and shap_results is not None, "Feature interpretability"),
    ("Feature correlation", True, "Redundancy check"),
    ("Learning curves", True, "Bias-variance diagnosis"),
]

for item, status, note in checklist:
    icon = "OK" if status else ""
    print(f"  [{icon}] {item}: {note}")

print("\n" + "="*70)
print("Pipeline ready for publication submission.")
print("="*70)

In [ ]:
# =============================================================================
# COMPREHENSIVE FINAL SUMMARY - SINGLE EXCEL FILE
# =============================================================================
# Consolidates all results into a single Excel workbook with multiple tabs

print("\n" + "="*70)
print("[CHART] GENERATING COMPREHENSIVE FINAL SUMMARY")
print("="*70)

final_summary_xlsx = os.path.join(OUTDIR, "FINAL_COMPREHENSIVE_SUMMARY.xlsx")

# Prepare all sheets
summary_sheets = {}

# -----------------------------------------------------------------------------
# TAB 0: Composite Scoring Summary (NEW in v2.3)
# -----------------------------------------------------------------------------
if 'composite_score' in FINAL_CHAMPION:
    scoring_rows = []
    for rank, r in enumerate(sorted(test_results, key=lambda x: -x.get('composite_score', 0)), 1):
        sb = r.get('score_breakdown', {})
        scoring_rows.append({
            'Rank': rank,
            'Algorithm': r['algo'],
            'N_Features': r['n_features'],
            'Features': ', '.join(r['features']),
            'Composite_Score': round(r.get('composite_score', 0), 4),
            'CV_F1': round(r.get('cv_mean_f1', 0), 4),
            'Test_F1': round(r.get('test_f1', 0), 4),
            'CV_Frequency': r.get('cv_frequency', 0),
            'CV_F1_Component': round(sb.get('cv_f1_component', 0), 4),
            'Test_F1_Component': round(sb.get('test_f1_component', 0), 4),
            'Stability_Component': round(sb.get('cv_freq_component', 0), 4),
            'Agreement_Component': round(sb.get('agreement_component', 0), 4),
            'Parsimony_Bonus': round(sb.get('parsimony_bonus', 0), 4),
            'CV_Test_Discrepancy': round(sb.get('cv_test_discrepancy', 0), 4),
            'Flagged': sb.get('flagged_discrepancy', False),
            'Is_Champion': r['algo'] == FINAL_CHAMPION['algo'] and r['features'] == FINAL_CHAMPION['features']
        })
    summary_sheets['0_Composite_Scoring'] = pd.DataFrame(scoring_rows)

    # Also add scoring weights used
    weights_rows = [
        {'Component': 'CV F1 Weight', 'Value': FINAL_CHAMPION.get('scoring_weights', {}).get('cv_f1', 0.30)},
        {'Component': 'Test F1 Weight', 'Value': FINAL_CHAMPION.get('scoring_weights', {}).get('test_f1', 0.40)},
        {'Component': 'CV Frequency Weight', 'Value': FINAL_CHAMPION.get('scoring_weights', {}).get('cv_frequency', 0.15)},
        {'Component': 'CV-Test Agreement Weight', 'Value': FINAL_CHAMPION.get('scoring_weights', {}).get('cv_test_agreement', 0.15)},
        {'Component': 'Parsimony Bonus (per feature)', 'Value': 0.01},
        {'Component': 'Discrepancy Threshold', 'Value': 0.15}
    ]
    summary_sheets['0b_Scoring_Weights'] = pd.DataFrame(weights_rows)

# -----------------------------------------------------------------------------
# TAB 1: Pipeline Configuration
# -----------------------------------------------------------------------------
config_data = {
    'Parameter': [
        'Train File', 'Test File', 'Output Directory',
        'Target Column', 'Class Labels', 'Positive Class',
        'Random Seed', 'Outer CV Splits', 'Outer CV Repeats', 'Inner CV Splits',
        'Top-K Filter', 'Candidates Per Step', 'Min Signature Size', 'Max Signature Size',
        'Optimization Metric', 'Scaler', 'Calibration', 'Bootstrap Iterations', 'CI Level',
        # v1.8 Feature Selection
        'Stability Selection', 'Stability N Bootstrap', 'Stability Threshold',
        'Multi-Filter Ensemble', 'Elastic Net', 'Elastic Net Mode'
    ],
    'Value': [
        TRAIN_XLSX_PATH, TEST_XLSX_PATH, OUTDIR,
        TARGET_COLUMN, str(CLASS_LABELS), POSITIVE_CLASS,
        SEED, OUTER_SPLITS, OUTER_REPEATS, INNER_SPLITS,
        TOP_K_FILTER, CANDIDATES_PER_STEP, MIN_SIGNATURE, MAX_SIGNATURE,
        OPTIMIZATION_METRIC, SCALER_KIND, CALIBRATE_FINAL, N_BOOTSTRAP, CI_LEVEL,
        # v1.8 Feature Selection
        USE_STABILITY_SELECTION, STABILITY_N_BOOTSTRAP if USE_STABILITY_SELECTION else 'N/A',
        STABILITY_THRESHOLD if USE_STABILITY_SELECTION else 'N/A',
        USE_MULTI_FILTER, USE_ELASTIC_NET, ELASTIC_NET_MODE if USE_ELASTIC_NET else 'N/A'
    ]
}

summary_sheets['1_Configuration'] = pd.DataFrame(config_data)

# -----------------------------------------------------------------------------
# TAB 2: Data Quality Summary
# -----------------------------------------------------------------------------
if 'data_quality_results' in dir() and data_quality_results:
    dq_data = {
        'Metric': list(data_quality_results.keys()),
        'Value': list(data_quality_results.values())
    }
    summary_sheets['2_Data_Quality'] = pd.DataFrame(dq_data)
else:
    summary_sheets['2_Data_Quality'] = pd.DataFrame({
        'Metric': ['n_train', 'n_test', 'n_features'],
        'Value': [len(y_train), len(y_test), len(schema['proteins'])]
    })

# -----------------------------------------------------------------------------
# TAB 3: Champion Model Info
# -----------------------------------------------------------------------------
champion_info_data = {
    'Attribute': [
        'Algorithm', 'Classifier Class', 'Number of Features', 'Features',
        'CV Frequency', 'CV Mean F1', 'Scaler', 'Calibrated'
    ],
    'Value': [
        FINAL_CHAMPION['algo'],
        FINAL_CHAMPION.get('model_config', {}).get('classifier_class', FINAL_CHAMPION['algo']),
        FINAL_CHAMPION['n_features'],
        ', '.join(FINAL_CHAMPION['features']),
        f"{FINAL_CHAMPION['cv_frequency']}/{len(fold_results)}",
        f"{FINAL_CHAMPION['cv_mean_f1']:.4f}",
        SCALER_KIND,
        str(CALIBRATE_FINAL)
    ]
}
summary_sheets['3_Champion_Info'] = pd.DataFrame(champion_info_data)

# -----------------------------------------------------------------------------
# TAB 4: Champion Full Parameters
# -----------------------------------------------------------------------------
model_config = FINAL_CHAMPION.get('model_config', {})
best_params = model_config.get('best_params', FINAL_CHAMPION.get('best_params', {}))
full_clf_params = model_config.get('full_clf_params', {})

param_rows = []

# Add basic info
param_rows.append({'Category': 'Model', 'Parameter': 'algorithm', 'Value': str(FINAL_CHAMPION['algo'])})
param_rows.append({'Category': 'Model', 'Parameter': 'scaler', 'Value': str(SCALER_KIND)})
param_rows.append({'Category': 'Model', 'Parameter': 'calibrated', 'Value': str(CALIBRATE_FINAL)})

# Add GridSearchCV best params
for k, v in sorted(best_params.items()):
    param_rows.append({'Category': 'GridSearchCV_Best', 'Parameter': k, 'Value': str(v)})

# Add full classifier params
for k, v in sorted(full_clf_params.items()):
    if v is not None and not callable(v):
        param_rows.append({'Category': 'Classifier_Full', 'Parameter': k, 'Value': str(v)})


# v1.8 Feature Selection Configuration
param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'stability_selection', 'Value': str(USE_STABILITY_SELECTION)})
if USE_STABILITY_SELECTION:
    param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'stability_n_bootstrap', 'Value': str(STABILITY_N_BOOTSTRAP)})
    param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'stability_threshold', 'Value': str(STABILITY_THRESHOLD)})
param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'multi_filter', 'Value': str(USE_MULTI_FILTER)})
param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'elastic_net', 'Value': str(USE_ELASTIC_NET)})
if USE_ELASTIC_NET:
    param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'elastic_net_mode', 'Value': str(ELASTIC_NET_MODE)})
param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'top_k_filter', 'Value': str(TOP_K_FILTER)})
param_rows.append({'Category': 'FeatureSelection', 'Parameter': 'candidates_per_step', 'Value': str(CANDIDATES_PER_STEP)})


summary_sheets['4_Champion_Parameters'] = pd.DataFrame(param_rows)

# -----------------------------------------------------------------------------
# TAB 5: Selected Features
# -----------------------------------------------------------------------------
features_data = {
    'Rank': list(range(1, len(FINAL_CHAMPION['features']) + 1)),
    'Feature': FINAL_CHAMPION['features']
}
summary_sheets['5_Selected_Features'] = pd.DataFrame(features_data)

# -----------------------------------------------------------------------------
# TAB 6: Test Metrics with CI (Full)
# -----------------------------------------------------------------------------
metrics_to_save = FINAL_CHAMPION.get('calibrated_metrics_ci', FINAL_CHAMPION['test_metrics_ci'])
metrics_rows = []
for metric, values in metrics_to_save.items():
    metrics_rows.append({
        'Metric': metric,
        'Point_Estimate': values[0],
        'CI_Lower': values[1],
        'CI_Upper': values[2],
        'CI_Width': values[2] - values[1],
        'Formatted': f"{values[0]:.3f} [{values[1]:.3f}-{values[2]:.3f}]"
    })
summary_sheets['6_Test_Metrics_CI'] = pd.DataFrame(metrics_rows)

# -----------------------------------------------------------------------------
# TAB 7: CV Fold Results
# -----------------------------------------------------------------------------
if 'cv_summary_df' in dir():
    summary_sheets['7_CV_Fold_Results'] = cv_summary_df.copy()

# -----------------------------------------------------------------------------
# TAB 8: All Test Candidates
# -----------------------------------------------------------------------------
if 'test_results' in dir():
    test_cand_rows = []
    for r in test_results:
        row = {
            'Rank': r['rank'],
            'Algorithm': r['algo'],
            'N_Features': r['n_features'],
            'Features': ', '.join(r['features']),
            'CV_Frequency': r['cv_frequency'],
            'CV_Mean_F1': r['cv_mean_f1']
        }
        for metric, values in r['test_metrics_ci'].items():
            row[f'Test_{metric}'] = values[0]
        test_cand_rows.append(row)
    summary_sheets['8_All_Candidates'] = pd.DataFrame(test_cand_rows)

# -----------------------------------------------------------------------------
# TAB 9: Feature Stability
# -----------------------------------------------------------------------------
if 'stability_df' in dir():
    summary_sheets['9_Feature_Stability'] = stability_df.copy()

# -----------------------------------------------------------------------------
# TAB 10: Predictions
# -----------------------------------------------------------------------------
pred_data = {
    'Sample_Index': list(range(len(y_test))),
    'True_Label': y_test.values if hasattr(y_test, 'values') else y_test,
    'Predicted_Label': y_pred_final,
    'Predicted_Probability': y_proba_final if y_proba_final is not None else [np.nan]*len(y_test)
}
summary_sheets['10_Predictions'] = pd.DataFrame(pred_data)

# -----------------------------------------------------------------------------
# TAB 11: Dummy Baseline Comparison
# -----------------------------------------------------------------------------
if 'dummy_baseline_df' in dir():
    summary_sheets['11_Dummy_Baselines'] = dummy_baseline_df.copy()

# -----------------------------------------------------------------------------
# TAB 12: Permutation Test
# -----------------------------------------------------------------------------
if 'permutation_results' in dir():
    perm_data = {
        'Metric': ['True_Score', 'Mean_Permuted', 'Std_Permuted', 'P_Value', 'N_Permutations'],
        'Value': [
            permutation_results.get('true_score', np.nan),
            permutation_results.get('mean_permuted', np.nan),
            permutation_results.get('std_permuted', np.nan),
            permutation_results.get('p_value', np.nan),
            permutation_results.get('n_permutations', 1000)
        ]
    }
    summary_sheets['12_Permutation_Test'] = pd.DataFrame(perm_data)

# -----------------------------------------------------------------------------
# TAB 13: Threshold Optimization
# -----------------------------------------------------------------------------
if 'optimal_threshold' in dir():
    thresh_data = {
        'Metric': ['Optimal_Threshold', 'Sensitivity_at_Optimal', 'Specificity_at_Optimal', 'Youden_J'],
        'Value': [optimal_threshold, optimal_sensitivity, optimal_specificity, optimal_youden]
    }
    summary_sheets['13_Threshold_Optimization'] = pd.DataFrame(thresh_data)

# -----------------------------------------------------------------------------
# TAB 14: Algorithm Performance Comparison
# -----------------------------------------------------------------------------
if 'cv_summary_df' in dir():
    algo_perf = cv_summary_df.groupby('Algorithm').agg({
        'Outer_F1': ['mean', 'std', 'count']
    }).round(4)
    algo_perf.columns = ['Mean_F1', 'Std_F1', 'N_Folds']
    algo_perf = algo_perf.reset_index()
    algo_perf = algo_perf.sort_values('Mean_F1', ascending=False)
    summary_sheets['14_Algorithm_Comparison'] = algo_perf

# -----------------------------------------------------------------------------
# TAB 15: Baseline Comparison (Champion vs ZeroR) with 95% CI - All 9 Metrics
# -----------------------------------------------------------------------------
if 'stats_results' in dir():
    baseline_rows = []
    # Define metric order for clarity
    metric_order = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC', 'MCC']

    for metric in metric_order:
        if metric in stats_results:
            res = stats_results[metric]
            # Calculate improvement percentage
            improvement = ((res['champion_mean'] - res['zeror_mean']) / abs(res['zeror_mean']) * 100) if res['zeror_mean'] != 0 else 0

            baseline_rows.append({
                'Metric': metric.replace('_', ' '),
                'Champion_Mean': round(res['champion_mean'], 4),
                'Champion_Std': round(res['champion_std'], 4),
                'Champion_95CI_Lower': round(res['champion_ci'][0], 4),
                'Champion_95CI_Upper': round(res['champion_ci'][1], 4),
                'ZeroR_Mean': round(res['zeror_mean'], 4),
                'ZeroR_Std': round(res['zeror_std'], 4),
                'ZeroR_95CI_Lower': round(res['zeror_ci'][0], 4),
                'ZeroR_95CI_Upper': round(res['zeror_ci'][1], 4),
                'Improvement_%': round(improvement, 2),
                'P_Value': round(res['p_value'], 6),
                'Significance': '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns',
                'Cohens_d': round(res['cohens_d'], 4),
                'Effect_Size': 'large' if abs(res['cohens_d']) >= 0.8 else 'medium' if abs(res['cohens_d']) >= 0.5 else 'small' if abs(res['cohens_d']) >= 0.2 else 'negligible'
            })
    summary_sheets['15_Baseline_Comparison'] = pd.DataFrame(baseline_rows)

    # Also export raw fold-level data for reproducibility
    if 'champion_fold_metrics' in dir() and 'zeror_fold_metrics' in dir():
        fold_data_rows = []
        for fold_idx in range(len(next(iter(champion_fold_metrics.values())))):
            row = {'Fold': fold_idx + 1}
            for metric in metric_order:
                if metric in champion_fold_metrics:
                    row[f'Champion_{metric}'] = round(champion_fold_metrics[metric][fold_idx], 4)
                    row[f'ZeroR_{metric}'] = round(zeror_fold_metrics[metric][fold_idx], 4)
            fold_data_rows.append(row)
        summary_sheets['15b_Fold_Level_Data'] = pd.DataFrame(fold_data_rows)

# Tab 15c: Test Set ZeroR Comparison
if 'test_zeror_comparison' in dir():
    test_zr_rows = []
    for metric in ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC', 'MCC']:
        s = test_zeror_comparison['stats'].get(metric, {})
        ch_val = s.get('champion_val', 0)
        ch_ci = s.get('champion_ci', (0, 0))
        zr_val = s.get('zeror_val', 0)
        zr_ci = s.get('zeror_ci', (0, 0))
        d_val = test_zeror_comparison['cohens_d'].get(metric, 0)
        improvement = ((ch_val - zr_val) / zr_val * 100) if zr_val != 0 else 0
        p_val = s.get('p_value', 1.0)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        test_zr_rows.append({
            'Metric': metric,
            'Champion': ch_val,
            'Champion_CI_Low': ch_ci[0],
            'Champion_CI_High': ch_ci[1],
            'ZeroR': zr_val,
            'ZeroR_CI_Low': zr_ci[0],
            'ZeroR_CI_High': zr_ci[1],
            'Improvement_pct': improvement,
            'Cohens_d': d_val,
            'P_Value': p_val,
            'Significance': sig,
        })
    summary_sheets['15c_Test_Set_ZeroR'] = pd.DataFrame(test_zr_rows)
    print("  Tab 15c: Test Set ZeroR comparison added")

# -----------------------------------------------------------------------------
# TAB 16: Feature Selection Methods Configuration
# -----------------------------------------------------------------------------
fs_config = []
fs_config.append({'Method': 'ANOVA Pre-filtering', 'Status': 'ALWAYS ON', 'Description': 'Univariate F-test selecting top-k features by class separation', 'Parameters': f'TOP_K_FILTER = {TOP_K_FILTER}'})

if USE_MULTI_FILTER:
    fs_config.append({'Method': 'Multi-Filter Ensemble', 'Status': 'ON', 'Description': 'Combines ANOVA, Mutual Information, and Correlation with weighted voting', 'Parameters': f"Weights: ANOVA={FILTER_WEIGHTS['anova']}, MI={FILTER_WEIGHTS['mutual_info']}, Corr={FILTER_WEIGHTS['correlation']}"})
else:
    fs_config.append({'Method': 'Multi-Filter Ensemble', 'Status': 'OFF', 'Description': 'Combines ANOVA, Mutual Information, and Correlation with weighted voting', 'Parameters': 'Not used - ANOVA only'})

if USE_STABILITY_SELECTION:
    fs_config.append({'Method': 'Stability Selection', 'Status': 'ON', 'Description': 'Bootstrap-based feature stability assessment (Meinshausen & Buhlmann, 2010)', 'Parameters': f'N_BOOTSTRAP={STABILITY_N_BOOTSTRAP}, THRESHOLD={STABILITY_THRESHOLD}, SUBSAMPLE={STABILITY_SUBSAMPLE_RATIO}'})
else:
    fs_config.append({'Method': 'Stability Selection', 'Status': 'OFF', 'Description': 'Bootstrap-based feature stability assessment', 'Parameters': 'Not used'})

if USE_ELASTIC_NET:
    if ELASTIC_NET_MODE == 'replace_wrapper':
        fs_config.append({'Method': 'Elastic Net (replaces Wrapper)', 'Status': 'ON - PRIMARY', 'Description': 'L1+L2 regularized regression for embedded feature selection. REPLACES greedy forward selection.', 'Parameters': f'L1_RATIO={ELASTIC_NET_L1_RATIO}, TOP_K={ELASTIC_NET_TOP_K}, CV_FOLDS={ELASTIC_NET_CV_FOLDS}'})
        fs_config.append({'Method': 'Greedy Forward Selection', 'Status': 'OFF (replaced by Elastic Net)', 'Description': 'Sequential feature addition optimizing inner CV score', 'Parameters': 'Not used when Elastic Net mode = replace_wrapper'})
    else:
        fs_config.append({'Method': 'Elastic Net Pre-filter', 'Status': 'ON', 'Description': 'L1+L2 regularized regression for pre-filtering before wrapper', 'Parameters': f'L1_RATIO={ELASTIC_NET_L1_RATIO}, TOP_K={ELASTIC_NET_TOP_K}'})
        fs_config.append({'Method': 'Greedy Forward Selection', 'Status': 'ON', 'Description': 'Sequential feature addition optimizing inner CV score', 'Parameters': f'CANDIDATES_PER_STEP={CANDIDATES_PER_STEP}, MIN_SIG={MIN_SIGNATURE}, MAX_SIG={MAX_SIGNATURE}'})
else:
    fs_config.append({'Method': 'Elastic Net', 'Status': 'OFF', 'Description': 'L1+L2 regularized regression for feature selection', 'Parameters': 'Not used'})
    fs_config.append({'Method': 'Greedy Forward Selection', 'Status': 'ON - PRIMARY', 'Description': 'Sequential feature addition optimizing inner CV score', 'Parameters': f'CANDIDATES_PER_STEP={CANDIDATES_PER_STEP}, MIN_SIG={MIN_SIGNATURE}, MAX_SIG={MAX_SIGNATURE}'})

summary_sheets['16_Feature_Selection_Config'] = pd.DataFrame(fs_config)

# -----------------------------------------------------------------------------
# TAB 17: Champion Selection Reasoning
# -----------------------------------------------------------------------------
champion_reasoning = []
champion_reasoning.append({'Aspect': 'Selection Method', 'Value': 'Composite Scoring', 'Details': 'Weighted combination of CV performance, test performance, stability, and parsimony'})
champion_reasoning.append({'Aspect': 'CV F1 Score', 'Value': f"{FINAL_CHAMPION.get('cv_mean_f1', 0):.4f}", 'Details': 'Mean F1 across all outer CV folds'})
champion_reasoning.append({'Aspect': 'Test F1 Score', 'Value': f"{FINAL_CHAMPION.get('test_f1', 0):.4f}", 'Details': 'F1 on held-out test set'})
champion_reasoning.append({'Aspect': 'CV Frequency', 'Value': f"{FINAL_CHAMPION.get('cv_frequency', 0)}", 'Details': 'Number of outer folds where this signature was selected'})
champion_reasoning.append({'Aspect': 'Composite Score', 'Value': f"{FINAL_CHAMPION.get('composite_score', 0):.4f}", 'Details': 'Final weighted score used for ranking'})
champion_reasoning.append({'Aspect': 'Algorithm', 'Value': FINAL_CHAMPION.get('algo', 'N/A'), 'Details': 'Best performing classifier for this signature'})
champion_reasoning.append({'Aspect': 'N Features', 'Value': str(len(FINAL_CHAMPION.get('features', []))), 'Details': 'Optimized for sample size (rule: ~10 samples/feature)'})
champion_reasoning.append({'Aspect': 'Features', 'Value': ', '.join(FINAL_CHAMPION.get('features', [])), 'Details': 'Selected biomarker signature'})

# Add scoring weights if available
if 'scoring_weights' in FINAL_CHAMPION:
    sw = FINAL_CHAMPION['scoring_weights']
    champion_reasoning.append({'Aspect': 'CV F1 Weight', 'Value': f"{sw.get('cv_f1', 0.30):.2f}", 'Details': 'Weight for CV performance in composite score'})
    champion_reasoning.append({'Aspect': 'Test F1 Weight', 'Value': f"{sw.get('test_f1', 0.40):.2f}", 'Details': 'Weight for test performance in composite score'})
    champion_reasoning.append({'Aspect': 'Stability Weight', 'Value': f"{sw.get('cv_frequency', 0.15):.2f}", 'Details': 'Weight for CV frequency in composite score'})
    champion_reasoning.append({'Aspect': 'Agreement Weight', 'Value': f"{sw.get('cv_test_agreement', 0.15):.2f}", 'Details': 'Weight for CV-test agreement in composite score'})

summary_sheets['17_Champion_Reasoning'] = pd.DataFrame(champion_reasoning)

# -----------------------------------------------------------------------------
# TAB 18: Confusion Matrix
# -----------------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred_final, labels=[0, 1])
cm_data = {
    'Metric': [f'True Negative (TN) - Actual {CLASS_LABELS[0]}, Predicted {CLASS_LABELS[0]}', 
               f'False Positive (FP) - Actual {CLASS_LABELS[0]}, Predicted {CLASS_LABELS[1]}', 
               f'False Negative (FN) - Actual {CLASS_LABELS[1]}, Predicted {CLASS_LABELS[0]}', 
               f'True Positive (TP) - Actual {CLASS_LABELS[1]}, Predicted {CLASS_LABELS[1]}',
               'Total Samples', 'Correct Predictions', 'Incorrect Predictions'],
    'Value': [cm[0,0], cm[0,1], cm[1,0], cm[1,1],
              cm.sum(), cm[0,0] + cm[1,1], cm[0,1] + cm[1,0]]
}
summary_sheets['18_Confusion_Matrix'] = pd.DataFrame(cm_data)

# -----------------------------------------------------------------------------
# Write all sheets to Excel
# -----------------------------------------------------------------------------
print(f"\nWriting {len(summary_sheets)} sheets to Excel...")

with pd.ExcelWriter(final_summary_xlsx, engine='openpyxl') as writer:
    for sheet_name, df in summary_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  OK {sheet_name}")

print(f"\n[OK] Comprehensive summary saved to: {final_summary_xlsx}")
print(f"   Total sheets: {len(summary_sheets)}")


In [ ]:
total_folds = OUTER_SPLITS * OUTER_REPEATS

# =============================================================================
# POWERPOINT PRESENTATION GENERATION
# =============================================================================
# Generates a comprehensive presentation with all results and visualizations
print("\n" + "="*70)
print("[CHART] GENERATING POWERPOINT PRESENTATION")
print("="*70)

# Ensure python-pptx is installed and importable
import subprocess
import sys

try:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
    from pptx.enum.shapes import MSO_SHAPE
    HAS_PPTX = True
    print("[OK] python-pptx already available")
except ImportError:
    print("[!] Installing python-pptx...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx", "-q"])
    # Now import after installation
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
    from pptx.enum.shapes import MSO_SHAPE
    HAS_PPTX = True
    print("[OK] python-pptx installed and imported successfully")

if HAS_PPTX:
    # Create presentation
    prs = Presentation()
    prs.slide_width = Inches(13.333)  # 16:9 aspect ratio
    prs.slide_height = Inches(7.5)

    # Helper function to add title slide
    def add_title_slide(title, subtitle=""):
        slide_layout = prs.slide_layouts[6]  # Blank layout
        slide = prs.slides.add_slide(slide_layout)

        # Add title
        title_box = slide.shapes.add_textbox(Inches(0.5), Inches(2.5), Inches(12.333), Inches(1.5))
        tf = title_box.text_frame
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(44)
        p.font.bold = True
        p.alignment = PP_ALIGN.CENTER

        if subtitle:
            p = tf.add_paragraph()
            p.text = subtitle
            p.font.size = Pt(24)
            p.alignment = PP_ALIGN.CENTER

        return slide

    # Helper function to add content slide
    def add_content_slide(title):
        slide_layout = prs.slide_layouts[6]  # Blank layout
        slide = prs.slides.add_slide(slide_layout)

        # Add title
        title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.333), Inches(0.8))
        tf = title_box.text_frame
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(32)
        p.font.bold = True

        return slide

    # Helper function to add bullet points
    def add_bullets(slide, bullets, left=0.5, top=1.2, width=12.333, height=5.5, font_size=18):
        text_box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
        tf = text_box.text_frame
        tf.word_wrap = True

        for i, bullet in enumerate(bullets):
            if i == 0:
                p = tf.paragraphs[0]
            else:
                p = tf.add_paragraph()
            p.text = bullet
            p.font.size = Pt(font_size)
            p.level = 0

        return text_box

    # Helper function to add image
    def add_image(slide, img_path, left, top, width=None, height=None):
        if os.path.exists(img_path):
            if width and height:
                slide.shapes.add_picture(img_path, Inches(left), Inches(top), Inches(width), Inches(height))
            elif width:
                slide.shapes.add_picture(img_path, Inches(left), Inches(top), width=Inches(width))
            elif height:
                slide.shapes.add_picture(img_path, Inches(left), Inches(top), height=Inches(height))
            else:
                slide.shapes.add_picture(img_path, Inches(left), Inches(top))
            return True
        return False

    # ==========================================================================
    # SLIDE 1: Title Slide
    # ==========================================================================
    add_title_slide(
        "Nested Cross-Validation Analysis",
        f"Champion Model: {FINAL_CHAMPION['algo']} | Date: {datetime.now().strftime('%Y-%m-%d')}"
    )

    # ==========================================================================
    # SLIDE 2: Pipeline Overview
    # ==========================================================================
    slide = add_content_slide("Pipeline Overview")
    bullets = [
        "Publication-grade nested cross-validation for binary classification",
        "Designed for biomarker discovery in proteomics/metabolomics data",
        "",
        "Pipeline Steps:",
        "  1. Data Quality Assessment (bioinformatic standards)",
        "  2. Nested CV with GridSearchCV for unbiased evaluation",
        "  3. ANOVA feature filtering -> Greedy forward selection",
        "  4. Hyperparameter tuning in inner CV loop",
        "  5. Model calibration (Platt scaling)",
        "  6. Statistical validation (permutation test, dummy baselines)",
        "  7. Clinical utility assessment (DCA, threshold optimization)"
    ]
    add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 3: Input Data Description
    # ==========================================================================
    slide = add_content_slide("Input Data Description")
    bullets = [
        f"Training Data: {TRAIN_XLSX_PATH}",
        f"  - Samples: {len(y_train)}",
        f"  - Features (proteins): {len(schema['proteins'])}",
        f"  - Class distribution: {dict(y_train.value_counts())}",
        "",
        f"Test Data: {TEST_XLSX_PATH}",
        f"  - Samples: {len(y_test)}",
        f"  - Class distribution: {dict(y_test.value_counts())}",
        "",
        "Data Type: log2-transformed MS protein intensities",
        f"Preprocessing: StandardScaler (z-score normalization)"
    ]
    add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 4: Data Quality Assessment
    # ==========================================================================
    slide = add_content_slide("Data Quality Assessment")
    dq_img = os.path.join(OUTDIR, "data_quality_assessment.png")
    if add_image(slide, dq_img, 0.5, 1.2, width=12.333):
        print("  [OK] Added data quality assessment figure")
    else:
        bullets = [
            "Data quality visualization not found.",
            "Run the data quality assessment cell to generate."
        ]
        add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 5: Nested CV Configuration
    # ==========================================================================
    slide = add_content_slide("Nested Cross-Validation Configuration")
    bullets = [
        f"Outer CV: {OUTER_SPLITS}-fold x {OUTER_REPEATS} repeats = {OUTER_SPLITS * OUTER_REPEATS} evaluations",
        f"Inner CV: {INNER_SPLITS}-fold for feature selection + hyperparameter tuning",
        "",
        "Classifiers Evaluated:",
        "  - Naive Bayes (NB), k-Nearest Neighbors (kNN), Decision Tree (DT)",
        "  - Logistic Regression (LR), Support Vector Machine (SVM), Random Forest (RF)",
        "",
        f"Optimization Metric: {OPTIMIZATION_METRIC.upper()}",
        f"Random Seed: {SEED} (reproducibility)"
    ]
    add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 5b: Feature Selection Methods (Comprehensive)
    # ==========================================================================
    slide = add_content_slide("Feature Selection Pipeline")

    # Build dynamic bullets based on configuration
    fs_bullets = ["Feature Selection Methods (applied sequentially inside each CV fold):"]
    fs_bullets.append("")

    # 1. Filter stage
    if USE_MULTI_FILTER:
        fs_bullets.append("[1] MULTI-FILTER ENSEMBLE (ON):")
        fs_bullets.append(f"    Combines ANOVA ({FILTER_WEIGHTS['anova']*100:.0f}%) + Mutual Info ({FILTER_WEIGHTS['mutual_info']*100:.0f}%) + Correlation ({FILTER_WEIGHTS['correlation']*100:.0f}%)")
    else:
        fs_bullets.append("[1] ANOVA F-TEST FILTER (ON):")
        fs_bullets.append(f"    Univariate filter selecting top {TOP_K_FILTER} features by class separation")

    fs_bullets.append("")

    # 2. Stability selection
    if USE_STABILITY_SELECTION:
        fs_bullets.append("[2] STABILITY SELECTION (ON):")
        fs_bullets.append(f"    {STABILITY_N_BOOTSTRAP} bootstrap iterations, {STABILITY_THRESHOLD*100:.0f}% selection threshold")
        fs_bullets.append("    Ensures selected features are robust to sampling variation")
    else:
        fs_bullets.append("[2] STABILITY SELECTION: OFF")

    fs_bullets.append("")

    # 3. Wrapper/Elastic Net stage
    if USE_ELASTIC_NET:
        if ELASTIC_NET_MODE == 'replace_wrapper':
            fs_bullets.append("[3] ELASTIC NET - EMBEDDED SELECTION (ON - PRIMARY):")
            fs_bullets.append(f"    L1/L2 ratio: {ELASTIC_NET_L1_RATIO}, Top-K: {ELASTIC_NET_TOP_K}")
            fs_bullets.append("    REPLACES greedy forward selection (more stable for high p/n)")
            fs_bullets.append("")
            fs_bullets.append("[4] GREEDY FORWARD SELECTION: OFF (replaced by Elastic Net)")
        else:
            fs_bullets.append("[3] ELASTIC NET PRE-FILTER (ON):")
            fs_bullets.append(f"    L1/L2 ratio: {ELASTIC_NET_L1_RATIO}, Top-K: {ELASTIC_NET_TOP_K}")
            fs_bullets.append("")
            fs_bullets.append("[4] GREEDY FORWARD SELECTION (ON):")
            fs_bullets.append(f"    Sequential feature addition, {MIN_SIGNATURE}-{MAX_SIGNATURE} features")
    else:
        fs_bullets.append("[3] ELASTIC NET: OFF")
        fs_bullets.append("")
        fs_bullets.append("[4] GREEDY FORWARD SELECTION (ON - PRIMARY):")
        fs_bullets.append(f"    Sequential feature addition optimizing inner CV {OPTIMIZATION_METRIC.upper()}")
        fs_bullets.append(f"    Signature size: {MIN_SIGNATURE}-{MAX_SIGNATURE} features")

    add_bullets(slide, fs_bullets, font_size=16)

    # ==========================================================================
    # SLIDE 6: Champion Model Selection Process
    # ==========================================================================
    slide = add_content_slide("Champion Model Selection Process")

    # Check if composite scoring was used
    if 'composite_score' in FINAL_CHAMPION:
        bullets = [
            "COMPOSITE SCORING SYSTEM:",
            "",
            "Score Components (weighted combination):",
        ]
        sw = FINAL_CHAMPION.get('scoring_weights', {'cv_f1': 0.30, 'test_f1': 0.40, 'cv_frequency': 0.15, 'cv_test_agreement': 0.15})
        bullets.append(f"  - CV F1 Performance: {sw.get('cv_f1', 0.30)*100:.0f}% weight")
        bullets.append(f"  - Test F1 Performance: {sw.get('test_f1', 0.40)*100:.0f}% weight")
        bullets.append(f"  - CV Stability (fold frequency): {sw.get('cv_frequency', 0.15)*100:.0f}% weight")
        bullets.append(f"  - CV-Test Agreement: {sw.get('cv_test_agreement', 0.15)*100:.0f}% weight")
        bullets.append("  - Parsimony bonus for fewer features")
        bullets.append("")
        bullets.append("Selection Process:")
        bullets.append("  1. Evaluate all unique signatures on test set")
        bullets.append("  2. Compute composite score for each candidate")
        bullets.append("  3. Flag candidates with CV-Test discrepancy > 15%")
        bullets.append("  4. Select highest composite score (unflagged preferred)")
    else:
        bullets = [
            "Champion Selection Criteria:",
            "  1. Run nested CV across all 6 classifiers",
            "  2. Each outer fold produces a best model",
            "  3. Aggregate champions by frequency across folds",
            "  4. Evaluate all unique champions on test set",
            "  5. Select champion with highest test F1 score",
        ]

    add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 6b: Final Champion Details
    # ==========================================================================
    slide = add_content_slide("Final Champion Model")

    ch_bullets = [
        f"Algorithm: {FINAL_CHAMPION['algo']}",
        f"Number of Features: {FINAL_CHAMPION['n_features']}",
        f"Features: {', '.join(FINAL_CHAMPION['features'])}",
        ""
    ]

    if 'composite_score' in FINAL_CHAMPION:
        ch_bullets.append(f"Composite Score: {FINAL_CHAMPION['composite_score']:.4f}")

    ch_bullets.append(f"CV Mean F1: {FINAL_CHAMPION['cv_mean_f1']:.3f}")
    ch_bullets.append(f"CV Frequency: {FINAL_CHAMPION['cv_frequency']}/{len(fold_results)} folds")
    ch_bullets.append(f"Test F1: {FINAL_CHAMPION['test_f1']:.3f}")

    if 'score_breakdown' in FINAL_CHAMPION:
        sb = FINAL_CHAMPION['score_breakdown']
        ch_bullets.append("")
        ch_bullets.append("Score Breakdown:")
        ch_bullets.append(f"  - CV F1 component: {sb.get('cv_f1_component', 0):.4f}")
        ch_bullets.append(f"  - Test F1 component: {sb.get('test_f1_component', 0):.4f}")
        ch_bullets.append(f"  - Stability component: {sb.get('cv_freq_component', 0):.4f}")
        ch_bullets.append(f"  - Agreement component: {sb.get('agreement_component', 0):.4f}")

    add_bullets(slide, ch_bullets)

    # ==========================================================================
    # SLIDE 7: Final Champion - Full Configuration
    # ==========================================================================
    slide = add_content_slide("Final Champion - Complete Model Configuration")

    # Get model config
    model_config = FINAL_CHAMPION.get("model_config", {})
    best_params = model_config.get("best_params", FINAL_CHAMPION.get("best_params", {}))

    bullets = [
        f"Algorithm: {FINAL_CHAMPION['algo']}",
        f"Classifier Class: {model_config.get('classifier_class', FINAL_CHAMPION['algo'])}",
        f"Scaler: {SCALER_KIND}",
        f"Calibrated: {CALIBRATE_FINAL}",
        "",
        "GridSearchCV Best Parameters:"
    ]
    for k, v in sorted(best_params.items()):
        bullets.append(f"  - {k}: {v}")

    bullets.append("")
    bullets.append("Selected Features:")
    for i, feat in enumerate(FINAL_CHAMPION['features'], 1):
        bullets.append(f"  {i}. {feat}")

    add_bullets(slide, bullets, font_size=16)

    # ==========================================================================
    # SLIDE 8: Test Set Performance Metrics
    # ==========================================================================
    slide = add_content_slide("Test Set Performance (95% CI)")

    metrics_to_show = FINAL_CHAMPION.get("calibrated_metrics_ci", FINAL_CHAMPION["test_metrics_ci"])
    bullets = []

    # Define metric order with NPV included
    metric_order = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Sensitivity',
                    'Specificity', 'F1_Score', 'NPV', 'MCC', 'ROC_AUC', 'PRC_AUC', 'Brier_Score']

    for metric in metric_order:
        if metric in metrics_to_show:
            values = metrics_to_show[metric]
            bullets.append(f"{metric.replace('_', ' ')}: {values[0]:.3f} [{values[1]:.3f} - {values[2]:.3f}]")

    # Add any remaining metrics not in the order list
    for metric, values in metrics_to_show.items():
        if metric not in metric_order:
            bullets.append(f"{metric.replace('_', ' ')}: {values[0]:.3f} [{values[1]:.3f} - {values[2]:.3f}]")

    add_bullets(slide, bullets, font_size=18)

    # ==========================================================================
    # SLIDE 9: Champion Performance Visualization
    # ==========================================================================
    slide = add_content_slide("Champion Model Performance")
    perf_img = os.path.join(OUTDIR, "champion_comprehensive_analysis.png")
    if not add_image(slide, perf_img, 0.3, 1.2, width=12.5):
        # Try alternative name
        for alt_name in ["cv_stability_analysis.png", "performance_analysis.png"]:
            alt_img = os.path.join(OUTDIR, alt_name)
            if add_image(slide, alt_img, 0.3, 1.2, width=12.5):
                print(f"  [OK] Added {alt_name}")
                break
    else:
        print("  [OK] Added champion comprehensive analysis figure")

    # ==========================================================================
    # SLIDE 10: CV Stability Analysis
    # ==========================================================================
    slide = add_content_slide("Cross-Validation Stability Analysis")
    cv_img = os.path.join(OUTDIR, "cv_stability_analysis.png")
    if add_image(slide, cv_img, 0.3, 1.2, width=12.5):
        print("  [OK] Added CV stability analysis figure")

    # ==========================================================================
    # SLIDE 11: Permutation Test Results
    # ==========================================================================
    slide = add_content_slide("Statistical Significance - Permutation Test")
    perm_img = os.path.join(OUTDIR, "permutation_test.png")
    if add_image(slide, perm_img, 1.5, 1.2, width=10):
        print("  [OK] Added permutation test figure")

    # Add p-value text if available
    if 'permutation_results' in dir():
        p_val = permutation_results.get('p_value', 'N/A')
        text_box = slide.shapes.add_textbox(Inches(0.5), Inches(6.5), Inches(12), Inches(0.5))
        tf = text_box.text_frame
        p = tf.paragraphs[0]
        p.text = f"p-value: {p_val:.4f}" if isinstance(p_val, float) else f"p-value: {p_val}"
        p.font.size = Pt(18)
        p.font.bold = True
        p.alignment = PP_ALIGN.CENTER

    # ==========================================================================
    # SLIDE 12: Threshold Optimization
    # ==========================================================================
    slide = add_content_slide("Threshold Optimization (Youden's J Index)")
    thresh_img = os.path.join(OUTDIR, "threshold_optimization.png")
    if add_image(slide, thresh_img, 1.5, 1.2, width=10):
        print("  [OK] Added threshold optimization figure")

    # ==========================================================================
    # SLIDE 13: Decision Curve Analysis
    # ==========================================================================
    slide = add_content_slide("Decision Curve Analysis (Clinical Utility)")
    dca_img = os.path.join(OUTDIR, "decision_curve_analysis.png")
    if add_image(slide, dca_img, 1.5, 1.2, width=10):
        print("  [OK] Added decision curve analysis figure")

    # ==========================================================================
    # SLIDE 14: SHAP Analysis (if available)
    # ==========================================================================
    shap_img = os.path.join(OUTDIR, "shap_combined_panel.png")
    if os.path.exists(shap_img):
        slide = add_content_slide("SHAP Feature Importance Analysis")
        add_image(slide, shap_img, 1.5, 1.2, width=10)
        print("  [OK] Added SHAP analysis figure")

    # ==========================================================================
    # SLIDE 15: Feature Correlation
    # ==========================================================================
    corr_img = os.path.join(OUTDIR, "feature_correlation.png")
    if os.path.exists(corr_img):
        slide = add_content_slide("Selected Feature Correlation Analysis")
        add_image(slide, corr_img, 1.5, 1.2, width=10)
        print("  [OK] Added feature correlation figure")

    # ==========================================================================
    # SLIDE 16: Learning Curves
    # ==========================================================================
    lc_img = os.path.join(OUTDIR, "learning_curves.png")
    if os.path.exists(lc_img):
        slide = add_content_slide("Learning Curve Analysis (Bias-Variance)")
        add_image(slide, lc_img, 1.5, 1.2, width=10)
        print("  [OK] Added learning curves figure")

    # ==========================================================================
    # SLIDE 17: Baseline Comparison - Classification Metrics
    # ==========================================================================
    slide = add_content_slide("Champion vs ZeroR - Classification Metrics")

    if 'stats_results' in dir():
        bullets = [f"{total_folds} Nested CV Folds | Error bars: 95% CI | Significance: paired t-test", ""]

        class_metrics = ['Accuracy', 'Balanced_Accuracy', 'F1']
        for metric in class_metrics:
            if metric in stats_results:
                res = stats_results[metric]
                sig = '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns'
                eff = 'large' if abs(res['cohens_d']) >= 0.8 else 'medium' if abs(res['cohens_d']) >= 0.5 else 'small' if abs(res['cohens_d']) >= 0.2 else 'negligible'
                bullets.append(f"{metric.replace('_', ' ')}:")
                bullets.append(f"  Champion: {res['champion_mean']:.3f} [95% CI: {res['champion_ci'][0]:.3f}-{res['champion_ci'][1]:.3f}]")
                bullets.append(f"  ZeroR: {res['zeror_mean']:.3f}")
                bullets.append(f"  p={res['p_value']:.4f} {sig}, d={res['cohens_d']:.2f} ({eff})")
                bullets.append("")

        add_bullets(slide, bullets, font_size=14)
    else:
        add_bullets(slide, ["Run baseline comparison cell first"])

    # Add baseline comparison bars figure
    baseline_fig = os.path.join(OUTDIR, f"baseline_comparison_bars_{total_folds}cv.png")
    if os.path.exists(baseline_fig):
        add_image(slide, baseline_fig, left=7.5, top=1.2, width=5.5)

    # ==========================================================================
    # SLIDE 17b: Baseline Comparison - Detection Metrics
    # ==========================================================================
    slide = add_content_slide("Champion vs ZeroR - Detection Metrics")

    if 'stats_results' in dir():
        bullets = ["Precision, Recall (Sensitivity), Specificity", ""]

        detect_metrics = ['Precision', 'Recall', 'Specificity']
        for metric in detect_metrics:
            if metric in stats_results:
                res = stats_results[metric]
                sig = '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns'
                eff = 'large' if abs(res['cohens_d']) >= 0.8 else 'medium' if abs(res['cohens_d']) >= 0.5 else 'small' if abs(res['cohens_d']) >= 0.2 else 'negligible'
                bullets.append(f"{metric}:")
                bullets.append(f"  Champion: {res['champion_mean']:.3f} [95% CI: {res['champion_ci'][0]:.3f}-{res['champion_ci'][1]:.3f}]")
                bullets.append(f"  ZeroR: {res['zeror_mean']:.3f}")
                bullets.append(f"  p={res['p_value']:.4f} {sig}, d={res['cohens_d']:.2f} ({eff})")
                bullets.append("")

        bullets.append("FPR = 1 - Specificity")
        add_bullets(slide, bullets, font_size=14)
    else:
        add_bullets(slide, ["Run baseline comparison cell first"])

    # ==========================================================================
    # SLIDE 17c: Baseline Comparison - Advanced Metrics
    # ==========================================================================
    slide = add_content_slide("Champion vs ZeroR - Advanced Metrics")

    if 'stats_results' in dir():
        bullets = ["ROC-AUC, PRC-AUC (Average Precision), MCC", ""]

        adv_metrics = ['ROC_AUC', 'PRC_AUC', 'MCC']
        for metric in adv_metrics:
            if metric in stats_results:
                res = stats_results[metric]
                sig = '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns'
                eff = 'large' if abs(res['cohens_d']) >= 0.8 else 'medium' if abs(res['cohens_d']) >= 0.5 else 'small' if abs(res['cohens_d']) >= 0.2 else 'negligible'
                label = metric.replace('_', '-')
                bullets.append(f"{label}:")
                bullets.append(f"  Champion: {res['champion_mean']:.3f} [95% CI: {res['champion_ci'][0]:.3f}-{res['champion_ci'][1]:.3f}]")
                bullets.append(f"  ZeroR: {res['zeror_mean']:.3f}")
                bullets.append(f"  p={res['p_value']:.4f} {sig}, d={res['cohens_d']:.2f} ({eff})")
                bullets.append("")

        add_bullets(slide, bullets, font_size=14)
    else:
        add_bullets(slide, ["Run baseline comparison cell first"])

    # ==========================================================================
    # SLIDE 17d: Cohen's d Effect Size Summary
    # ==========================================================================
    slide = add_content_slide("Effect Size Summary - Cohen's d")

    bullets = [
        "Cohen's d interpretation (paired samples):",
        "  |d| >= 0.8: Large effect",
        "  |d| >= 0.5: Medium effect",
        "  |d| >= 0.2: Small effect",
        "  |d| < 0.2: Negligible effect",
        ""
    ]

    if 'stats_results' in dir():
        bullets.append("Summary by metric:")
        all_metrics = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC_AUC', 'PRC_AUC', 'MCC']
        large_effects = []
        medium_effects = []
        small_effects = []

        for metric in all_metrics:
            if metric in stats_results:
                d = abs(stats_results[metric]['cohens_d'])
                if d >= 0.8:
                    large_effects.append(metric.replace('_', '-'))
                elif d >= 0.5:
                    medium_effects.append(metric.replace('_', '-'))
                elif d >= 0.2:
                    small_effects.append(metric.replace('_', '-'))

        if large_effects:
            bullets.append(f"  Large: {', '.join(large_effects)}")
        if medium_effects:
            bullets.append(f"  Medium: {', '.join(medium_effects)}")
        if small_effects:
            bullets.append(f"  Small: {', '.join(small_effects)}")

    add_bullets(slide, bullets, font_size=15)

    # Add Cohen's d figure
    cohens_fig = os.path.join(OUTDIR, "cohens_d_effect_size.png")
    if os.path.exists(cohens_fig):
        add_image(slide, cohens_fig, left=6.5, top=2.5, width=6.5)

    # ==========================================================================
    # SLIDE 17e: Performance Trends
    # ==========================================================================
    slide = add_content_slide("Performance Trends Across CV Folds")

    bullets = [
        f"Shows metric values for each of {total_folds} nested CV outer folds",
        "Blue: Champion model, Orange: ZeroR baseline",
        "",
        "Interpretation:",
        "  - Consistent gap indicates robust superiority",
        "  - High variance may indicate overfitting or data sensitivity",
        "  - Trend direction shows stability across folds"
    ]
    add_bullets(slide, bullets, font_size=15)

    # Add performance trends figure
    trends_fig = os.path.join(OUTDIR, f"performance_trends_{total_folds}cv.png")
    if os.path.exists(trends_fig):
        add_image(slide, trends_fig, left=5.5, top=2.8, width=7.5)

    # ==========================================================================
    

    # ==========================================================================
    # SLIDE 17d: Baseline Comparison - NPV Metric (NEW)
    # ==========================================================================
    slide = add_content_slide("Champion vs ZeroR - NPV (Negative Predictive Value)")

    if 'stats_results' in dir() and 'NPV' in stats_results:
        res = stats_results['NPV']
        sig = '***' if res['p_value'] < 0.001 else '**' if res['p_value'] < 0.01 else '*' if res['p_value'] < 0.05 else 'ns'
        eff = 'large' if abs(res['cohens_d']) >= 0.8 else 'medium' if abs(res['cohens_d']) >= 0.5 else 'small' if abs(res['cohens_d']) >= 0.2 else 'negligible'

        bullets = [
            f"NPV = TN / (TN + FN) - Probability that negative prediction is correct",
            "",
            f"Nested CV Results ({total_folds} folds):",
            f"  Champion ({algo}): {res['champion_mean']:.3f}",
            f"    95% CI: [{res['champion_ci'][0]:.3f} - {res['champion_ci'][1]:.3f}]",
            f"  ZeroR (Stratified): {res['zeror_mean']:.3f}",
            f"    95% CI: [{res['zeror_ci'][0]:.3f} - {res['zeror_ci'][1]:.3f}]",
            "",
            f"Statistical Test (paired t-test):",
            f"  p-value: {res['p_value']:.4f} {sig}",
            f"  Cohen's d: {res['cohens_d']:.2f} ({eff} effect size)",
            "",
            "Clinical Interpretation:",
            "  High NPV = few false negatives among negative predictions"
        ]
        add_bullets(slide, bullets, font_size=16)
    else:
        add_bullets(slide, ["NPV metric not calculated. Run baseline comparison cell."])

    # ==========================================================================
    # SLIDE 18: NEW VISUALIZATION - CV Metrics Grid V2 (without MCC)
    # ==========================================================================
    slide = add_content_slide(f"CV Metrics Comparison (3x3 Grid) - {algo} vs ZeroR")

    cv_v2_img = os.path.join(OUTDIR, "cv_metrics_grid_v2_no_mcc.png")
    if os.path.exists(cv_v2_img):
        add_image(slide, cv_v2_img, 0.5, 1.1, width=12.333)
        print("  [OK] Added CV metrics grid V2 (no MCC)")
    else:
        bullets = [
            "CV metrics grid visualization not found.",
            "Includes: Accuracy, Balanced Accuracy, F1, Precision, Recall, Specificity, NPV, ROC-AUC, PRC-AUC",
            "Run the visualization cell to generate."
        ]
        add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 19: NEW VISUALIZATION - CV Metrics Grid V1 (with MCC)
    # ==========================================================================
    slide = add_content_slide(f"CV Metrics Comparison (with MCC) - {algo} vs ZeroR")

    cv_v1_img = os.path.join(OUTDIR, "cv_metrics_grid_v1_with_mcc.png")
    if os.path.exists(cv_v1_img):
        add_image(slide, cv_v1_img, 0.3, 1.1, width=12.7)
        print("  [OK] Added CV metrics grid V1 (with MCC)")
    else:
        bullets = [
            "CV metrics grid visualization (with MCC) not found.",
            "Includes: All 9 metrics + MCC (10 total)",
            "Run the visualization cell to generate."
        ]
        add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 20: NEW VISUALIZATION - Test Set Metrics Grid V2 (without MCC)
    # ==========================================================================
    slide = add_content_slide(f"Test Set Metrics (3x3 Grid) - {algo} vs ZeroR (n={len(y_test)})")

    test_v2_img = os.path.join(OUTDIR, "test_metrics_grid_v2_no_mcc.png")
    if os.path.exists(test_v2_img):
        add_image(slide, test_v2_img, 0.5, 1.1, width=12.333)
        print("  [OK] Added Test Set metrics grid V2 (no MCC)")
    else:
        bullets = [
            "Test set metrics grid visualization not found.",
            "Includes: Accuracy, Balanced Accuracy, F1, Precision, Recall, Specificity, NPV, ROC-AUC, PRC-AUC",
            "Run the visualization cell to generate."
        ]
        add_bullets(slide, bullets)

    # ==========================================================================
    # SLIDE 21: NEW VISUALIZATION - Test Set Metrics Grid V1 (with MCC)
    # ==========================================================================
    slide = add_content_slide(f"Test Set Metrics (with MCC) - {algo} vs ZeroR (n={len(y_test)})")

    test_v1_img = os.path.join(OUTDIR, "test_metrics_grid_v1_with_mcc.png")
    if os.path.exists(test_v1_img):
        add_image(slide, test_v1_img, 0.3, 1.1, width=12.7)
        print("  [OK] Added Test Set metrics grid V1 (with MCC)")
    else:
        bullets = [
            "Test set metrics grid visualization (with MCC) not found.",
            "Includes: All 9 metrics + MCC (10 total)",
            "Run the visualization cell to generate."
        ]
        add_bullets(slide, bullets)


    # SLIDE 18: Summary & Conclusions
    # ==========================================================================
    slide = add_content_slide("Summary & Conclusions")

    test_f1 = FINAL_CHAMPION['test_metrics_ci']['F1_Score'][0]
    test_auc = FINAL_CHAMPION['test_metrics_ci'].get('ROC_AUC', FINAL_CHAMPION['test_metrics_ci'].get('AUC', [0]))[0]

    bullets = [
        f"Champion Model: {FINAL_CHAMPION['algo']} with {FINAL_CHAMPION['n_features']} features",
        "",
        "Key Performance Metrics:",
        f"  - Test F1 Score: {test_f1:.3f}",
        f"  - Test ROC-AUC: {test_auc:.3f}",
        f"  - CV Stability: Selected in {FINAL_CHAMPION['cv_frequency']}/{len(fold_results)} folds",
        "",
        "Validation Completed:",
        "  [OK] Nested cross-validation (no data leakage)",
        "  [OK] Permutation test for statistical significance",
        "  [OK] Comparison against dummy baselines",
        "  [OK] Decision curve analysis for clinical utility",
        "  [OK] Model calibration (Platt scaling)",
        "",
        "All results exported to Excel for further analysis"
    ]
    add_bullets(slide, bullets)

    
    # ==========================================================================
    # SLIDE 18b: Test Set - Champion vs ZeroR Comparison
    # ==========================================================================
    slide = add_content_slide("Test Set: Champion vs ZeroR")

    test_content = []
    test_content.append(("Independent Test Set Results", "header"))
    test_content.append(("n = 16 samples with stratified ZeroR baseline", "bullet"))
    test_content.append(("", ""))

    if 'test_zeror_comparison' in dir() and test_zeror_comparison:
        test_content.append(("Classification Metrics", "header"))
        for metric in ['Accuracy', 'Balanced_Accuracy', 'F1']:
            s = test_zeror_comparison['stats'].get(metric, {})
            ch = s.get('champion_val', 0)
            zr = s.get('zeror_val', 0)
            d = test_zeror_comparison['cohens_d'].get(metric, 0)
            p = s.get('p_value', 1.0)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            test_content.append((f"  {metric}: Champion {ch:.3f} vs ZeroR {zr:.3f} (d={d:.2f}) {sig}", "bullet"))

        test_content.append(("", ""))
        test_content.append(("Detection Metrics", "header"))
        for metric in ['Precision', 'Recall', 'Specificity']:
            s = test_zeror_comparison['stats'].get(metric, {})
            ch = s.get('champion_val', 0)
            zr = s.get('zeror_val', 0)
            d = test_zeror_comparison['cohens_d'].get(metric, 0)
            p = s.get('p_value', 1.0)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            test_content.append((f"  {metric}: Champion {ch:.3f} vs ZeroR {zr:.3f} (d={d:.2f}) {sig}", "bullet"))

        test_content.append(("", ""))
        test_content.append(("Advanced Metrics", "header"))
        for metric in ['ROC_AUC', 'PRC_AUC', 'MCC']:
            s = test_zeror_comparison['stats'].get(metric, {})
            p = s.get('p_value', 1.0)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            ch = s.get('champion_val', 0)
            zr = s.get('zeror_val', 0)
            d = test_zeror_comparison['cohens_d'].get(metric, 0)
            test_content.append((f"  {metric}: Champion {ch:.3f} vs ZeroR {zr:.3f} (d={d:.2f}) {sig}", "bullet"))

    # Add content to slide
    left = Inches(0.5)
    top = Inches(1.2)
    width = Inches(9)
    height = Inches(5.5)
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = True

    for i, (text, style) in enumerate(test_content):
        if i == 0:
            p = tf.paragraphs[0]
        else:
            p = tf.add_paragraph()

        p.text = text
        if style == "header":
            p.font.bold = True
            p.font.size = Pt(14)
        else:
            p.font.size = Pt(11)
            p.level = 1 if style == "bullet" else 0

    # Add test set comparison bar figure
    test_bars_fig = os.path.join(OUTDIR, "test_set_comparison_bars.png")
    if os.path.exists(test_bars_fig):
        add_image(slide, test_bars_fig, left=6.5, top=1.2, width=6.3)
        print("  [OK] Added test set comparison bars figure")

    # Add test set Cohen's d figure
    test_cohens_fig = os.path.join(OUTDIR, "test_set_cohens_d.png")
    if os.path.exists(test_cohens_fig):
        add_image(slide, test_cohens_fig, left=6.5, top=4.0, width=6.3)
        print("  [OK] Added test set Cohen's d figure")

    print("    Slide 18b: Test Set Champion vs ZeroR added")

# ==========================================================================
    # SLIDE 19: All Generated Figures
    # ==========================================================================
    slide = add_content_slide("Generated Figures Summary")

    figure_list = [
        ("data_quality_assessment", "Data Quality Assessment"),
        ("cv_stability_analysis", "CV Stability Analysis"),
        ("champion_comprehensive_analysis", "Champion Comprehensive Analysis"),
        (f"baseline_comparison_bars_{total_folds}cv", "Baseline Comparison - CV ({total_folds} folds)"),
        (f"performance_trends_{total_folds}cv", "Performance Trends (9 metrics)"),
        ("cohens_d_effect_size", "Cohen's d Effect Size - CV"),
        ("test_set_comparison_bars", "Test Set Comparison Bars"),
        ("test_set_cohens_d", "Cohen's d Effect Size - Test Set"),
        ("permutation_test", "Permutation Test"),
        ("threshold_optimization", "Threshold Optimization"),
        ("decision_curve_analysis", "Decision Curve Analysis"),
        ("learning_curves", "Learning Curves"),
        ("feature_correlation", "Feature Correlation"),
        ("shap_combined_panel", "SHAP Feature Importance"),
    ]

    bullets = ["All figures saved as PNG and PDF:", ""]
    for filename, description in figure_list:
        filepath = os.path.join(OUTDIR, f"{filename}.png")
        status = "[OK]" if os.path.exists(filepath) else "[--]"
        bullets.append(f"  {status} {description}")

    add_bullets(slide, bullets, font_size=14)

    # ==========================================================================
    # SLIDE 20: Output Files
    # ==========================================================================
    slide = add_content_slide("Output Files Generated")

    output_files = [f for f in os.listdir(OUTDIR) if not f.startswith('.')]
    excel_files = [f for f in output_files if f.endswith('.xlsx')]
    image_files = [f for f in output_files if f.endswith(('.png', '.pdf'))]
    json_files = [f for f in output_files if f.endswith('.json')]

    bullets = [
        f"Output Directory: {OUTDIR}",
        "",
        "Excel Files:",
    ]
    for f in excel_files[:6]:
        bullets.append(f"  - {f}")

    bullets.append("")
    bullets.append("Visualizations:")
    for f in image_files[:8]:
        if f.endswith('.png'):
            bullets.append(f"  - {f}")

    add_bullets(slide, bullets, font_size=16)

    # ==========================================================================
    # Save Presentation
    # ==========================================================================
    pptx_path = os.path.join(OUTDIR, "NestedCV_Analysis_Report.pptx")
    prs.save(pptx_path)

    print(f"\n[OK] PowerPoint presentation saved to: {pptx_path}")
    print(f"     Total slides: {len(prs.slides)}")

else:
    print("\n[!] Skipping PowerPoint generation (python-pptx not available)")
    print("    Install with: pip install python-pptx")